# Notebook Overview — Run Representation-Based VideoQA

## Purpose

This notebook performs representation-based Video Question Answering (VideoQA) using precomputed text and video representations. Rather than performing direct multimodal inference on the original videos, the notebook combines shared `clip_text` question–answer representations with one of three supported video representation sources:

* `clip_video`
* `autoencoder_video`
* `hybrid_clip_autoencoder`

The notebook prepares independent training and validation datasets, loads and validates the required representation artifacts, constructs one candidate-level record for each answer choice, groups the candidate records into five-choice QA samples, executes the selected prediction method, generates validation predictions, validates the resulting prediction dataset, and saves experiment artifacts for downstream evaluation.

The notebook supports five prediction methods:

* Cosine Similarity
* Fusion MLP
* Interaction Fusion
* Gated Fusion
* Bilinear Fusion

Cosine Similarity performs validation-only scoring using compatible CLIP text and video embedding dimensions. The learned fusion methods train on the NExT-QA training split and generate predictions for the configured validation split. When the hybrid representation source is selected, normalized CLIP and autoencoder video representations are concatenated before prediction, allowing direct evaluation of combined semantic and reconstruction-based representations.

## Inputs

* Shared `clip_text` question–answer representation artifacts
* Selected video representation artifacts (`clip_video`, `autoencoder_video`, or `hybrid_clip_autoencoder`)
* NExT-QA training and validation annotations
* Shared project configuration
* Prediction-method and classifier configuration

## Outputs

* Multiple-choice validation prediction dataset
* Prediction validation report
* Experiment summary report
* Training-history information for learned methods
* Saved prediction artifacts for Notebook 08

## Processing Workflow

1. Initialize the notebook environment.
2. Configure the representation-based VideoQA experiment.
3. Verify runtime readiness.
4. Prepare independent training and validation datasets.
5. Load and validate the shared CLIP text representations.
6. Load and validate the selected video representations.
7. Construct candidate-level representation datasets.
8. Build grouped five-choice QA samples.
9. Generate hybrid video representations when selected.
10. Execute the selected prediction method.
11. Build the standardized prediction dataset.
12. Validate prediction artifacts.
13. Save experiment artifacts.
14. Generate the prediction summary report.
15. Display representative predictions.
16. Summarize the completed experiment.

## Downstream Consumer

**Notebook 08 — Evaluate Development Results**

### 🔷 Step 0 — Configure Experiment Execution

* Configure the experiment settings used throughout the notebook before execution begins.
* Specify EXPERIMENT_NAME, which identifies the experiment and determines the corresponding output directory.
* Select the prediction method for representation-based VideoQA experiments (Notebook 07 only) and ensure it matches the value specified in EXPERIMENT_NAME.
* Choose whether to evaluate the full validation split or a development subset and specify the development subset size when applicable.
* Configure notebook runtime options, including GPU requirements and verbose progress reporting.
* Keep these settings consistent across all notebooks that participate in the same experiment.

In [ ]:
# ============================================================
# Step 0: Configure Experiment Execution
# ============================================================
# Review these settings before running the notebook.
# Keep these values consistent across all notebooks in the experiment.
# ------------------------------------------------------------
#
# EXPERIMENT_NAME identifies the experiment and output directory.
#
# Format:
#   <representation>_<prediction_method>_<dataset>
#
# Representation:
#   qwen2vl
#   clip
#   ae_seg6s_stride4
#   hybrid_clip_ae
#
# Prediction method:
#   baseline
#   similarity
#   mlp
#   interaction
#   gated
#   bilinear
#
# Dataset:
#   dev100
#   dev500
#   full
#
# Examples:
#   qwen2vl_baseline_dev100
#   clip_bilinear_full
#   ae_seg6s_stride4_mlp_dev100
#   hybrid_clip_ae_bilinear_dev500
#
EXPERIMENT_NAME = "ae_seg6s_stride4_mlp_dev100"

# Prediction method (used only by Notebook 07).
# Must match the prediction method specified in EXPERIMENT_NAME.
# Supported values:
#   cosine_similarity
#   fusion_mlp_classifier
#   interaction_fusion_classifier
#   gated_fusion_classifier
#   bilinear_fusion_classifier
#
REPRESENTATION_VIDEOQA_METHOD = "fusion_mlp_classifier"

# Evaluate the full validation split if True.
RUN_FULL_EVALUATION_SPLIT = False

# Development subset size (typically 100 or 500).
DEVELOPMENT_SUBSET_SIZE = 100

# Require an NVIDIA L4 GPU.
REQUIRE_L4_GPU = True

# Display detailed notebook progress.
VERBOSE = True

# Save generated artifacts to Google Drive.
# Disabled by default for the public tutorial.
ENABLE_GOOGLE_DRIVE_WRITES = False



### 🔷 Step 1 — Initialize Environment for Representation-Based VideoQA

* Mount Google Drive and prepare the Colab execution environment.
* Clone the public GitHub repository using sparse checkout without requiring authentication.
* Load shared project configuration constants and reusable utility modules.
* Verify required project directories and initialize experiment output locations.
* Load the NExT-QA annotation metadata used to prepare the training and validation datasets.
* Verify that the required representation artifacts are available for the selected experiment configuration.
* Display environment and dataset summary information when verbose output is enabled.
* Prepare the notebook for representation-based VideoQA experimentation.


In [ ]:
# ============================================================
# Step 1: Initialize Environment for Representation-Based VideoQA
# ============================================================

# Enable detailed notebook output.
VERBOSE = True

# Standard library modules for file and path management.
import os
from pathlib import Path

# ====================================
# Required for the Fusion Steps 10-13
# ====================================

# Libraries used throughout the representation-based VideoQA pipeline.
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# Colab utilities for Drive access and secure credentials.
from google.colab import drive

print("Initializing notebook environment...")
print("-" * 60)

# ------------------------------------------------------------
# DRIVE MOUNT
# ------------------------------------------------------------

# Google Drive stores the archived representation artifacts.
GOOGLE_DRIVE_MOUNT = "/content/drive"

# Mount Drive only if it has not already been mounted.
if not os.path.exists(GOOGLE_DRIVE_MOUNT):
    print("Mounting Google Drive...")
    drive.mount(GOOGLE_DRIVE_MOUNT)
else:
    print("Google Drive already mounted.")

# ------------------------------------------------------------
# REPO SETUP
# ------------------------------------------------------------

# Repository location within the Colab runtime.
REPO_NAME = "videoqa-representation-comparison"
REPO_OWNER = "pgailinas"
REPO_BASE_DIR = "/content"
REPO_DIR = os.path.join(REPO_BASE_DIR, REPO_NAME)

# Build the public repository URL.
repo_url = (
    f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
)

os.chdir(REPO_BASE_DIR)

# Clone the repository only if it is not already available.
if not os.path.exists(REPO_DIR):
    print("Cloning project repository...")
    !git clone --quiet --filter=blob:none --no-checkout {repo_url}
    os.chdir(REPO_DIR)
    !git sparse-checkout init --cone
    !git sparse-checkout set src datasets outputs
    !git checkout --quiet main
else:
    print("Project repository already available.")
    os.chdir(REPO_DIR)

print(f"Repository ready: {REPO_DIR}")

# ------------------------------------------------------------
# LOAD CONFIG
# ------------------------------------------------------------

# Load the shared experiment configuration and dataset utilities.
from src.videoqa_representation_config import *
from src.videoqa_project_restore import restore_project_artifacts
from src.nextqa_metadata import *

# ------------------------------------------------------------
# EXPERIMENT SETUP
# ------------------------------------------------------------

# Configure paths and settings for the selected experiment.
configure_experiment(EXPERIMENT_NAME)

# Determine the overall experiment type.
EXPERIMENT_TYPE = infer_experiment_type(EXPERIMENT_NAME)

print(f"Experiment name : {EXPERIMENT_NAME}")
print(f"Experiment type : {EXPERIMENT_TYPE}")

# ------------------------------------------------------------
# REQUIRED PATHS
# ------------------------------------------------------------

# Create the output directory if needed.
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# Verify that all required project directories exist.
required_paths = [
    Path("src"),
    Path("datasets"),
    QUESTIONS_DIR,
    METADATA_DIR,
    OUTPUTS_DIR,
    GOOGLE_DRIVE_SOURCE_ROOT,
]

missing_paths = [p for p in required_paths if not p.exists()]

if missing_paths:
    raise FileNotFoundError(f"Missing paths: {missing_paths}")

# ------------------------------------------------------------
# RESTORE PROJECT ARTIFACTS
# ------------------------------------------------------------

print("\nRestoring representation artifacts...")

# Every experiment uses CLIP text representations.
required_artifact_paths = [
    CLIP_TEXT_REPRESENTATIONS_LOCAL_CSV,
]

# Select the required video representation artifacts.
if EXPERIMENT_NAME.startswith("clip"):

    required_artifact_paths.append(
        CLIP_VIDEO_REPRESENTATIONS_LOCAL_CSV
    )

elif EXPERIMENT_NAME.startswith("hybrid_"):

    required_artifact_paths.extend([
        CLIP_VIDEO_REPRESENTATIONS_LOCAL_CSV,
        AUTOENCODER_LOCAL_VIDEO_REPRESENTATIONS_CSV,
    ])

elif EXPERIMENT_NAME.startswith("ae_"):

    required_artifact_paths.append(
        AUTOENCODER_LOCAL_VIDEO_REPRESENTATIONS_CSV
    )

else:
    raise ValueError(
        f"Unknown experiment type for video selection: {EXPERIMENT_NAME}"
    )

# Restore and verify the required representation artifacts.
restore_project_artifacts(
    drive_archive_path=PROJECT_ARTIFACTS_DRIVE_ARCHIVE,
    local_archive_path=PROJECT_ARTIFACTS_LOCAL_ARCHIVE,
    project_dir=VIDEOQA_PROJECT_DIR,
    required_paths=required_artifact_paths,
    verbose=VERBOSE,
)

print("\nArtifact restoration complete.")

# ------------------------------------------------------------
# LOAD NExT-QA
# ------------------------------------------------------------

print("\nLoading NExT-QA annotations...")

# Load the annotation files for each dataset split.
split_annotations = load_nextqa_split_annotations(
    annotations_dir=QUESTIONS_DIR,
    verbose=VERBOSE,
)

# Combine all splits into a single annotation table.
annotations_df = combine_nextqa_annotations(split_annotations, verbose=VERBOSE)

# Summarize the dataset by split.
split_summary_df = summarize_nextqa_splits(annotations_df)

if annotations_df.empty:
    raise RuntimeError("No NExT-QA annotation records loaded.")

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

# Display a summary of the loaded dataset.
print("\nDataset loaded.")
print(f"Annotation records: {len(annotations_df):,}")

if VERBOSE:
    print("\nSplit Summary")
    print("-" * 60)
    display(split_summary_df)

print("\nEnvironment ready for VideoQA.")



### 🔷 Step 2 — Define Representation-Based VideoQA Configuration

* Configure development-subset or full-split execution for representation-based VideoQA.
* Select the active video representation source (`clip_video`, `autoencoder_video`, or `hybrid_clip_autoencoder`).
* Specify the shared text representation source (`clip_text`).
* Select the representation-based prediction method:

  * Cosine Similarity;
  * Fusion MLP;
  * Interaction Fusion;
  * Gated Fusion; or
  * Bilinear Fusion.
* Validate compatibility between the selected video representation source and prediction method.
* Define the training split, validation split, development subset size, random seed, and answer mode.
* Configure the shared fusion dimensions, hidden layers, dropout, optimizer settings, batch size, learning rate, weight decay, and training epochs used by learned classifiers.
* Configure the input representation artifacts, experiment output locations, and hybrid representation settings when applicable.
* Display the active experiment configuration.



In [ ]:
# ============================================================
# Step 2: Define Representation-Based VideoQA Configuration
# ============================================================

# Standard path and table utilities.
from pathlib import Path

import pandas as pd

print("Defining representation-based VideoQA configuration...")

# Existing autoencoder experiment supplying the 256-D representations
# for hybrid experiments.
SOURCE_AUTOENCODER_EXPERIMENT_NAME = (
    "ae_seg6s_stride4_mlp_dev500"
)

# ------------------------------------------------------------
# Validate shared method configuration
# ------------------------------------------------------------

# Confirm that shared method definitions were loaded in Step 1.
required_method_configuration = [
    "SUPPORTED_REPRESENTATION_VIDEOQA_METHODS",
    "REPRESENTATION_VIDEOQA_METHOD_LABELS",
    "REPRESENTATION_VIDEOQA_METHOD_EXPERIMENT_TOKENS",
]

missing_method_configuration = [
    name
    for name in required_method_configuration
    if name not in globals()
]

if missing_method_configuration:
    raise NameError(
        "Missing shared representation-method configuration values: "
        + ", ".join(missing_method_configuration)
    )

# Reject unsupported prediction methods before configuring the pipeline.
if (
    REPRESENTATION_VIDEOQA_METHOD
    not in SUPPORTED_REPRESENTATION_VIDEOQA_METHODS
):
    raise ValueError(
        f"Unsupported REPRESENTATION_VIDEOQA_METHOD: "
        f"{REPRESENTATION_VIDEOQA_METHOD}. "
        f"Expected one of: "
        f"{sorted(SUPPORTED_REPRESENTATION_VIDEOQA_METHODS)}"
    )

# Resolve the display label and experiment-name token for the method.
representation_videoqa_method_label = (
    REPRESENTATION_VIDEOQA_METHOD_LABELS[
        REPRESENTATION_VIDEOQA_METHOD
    ]
)

representation_videoqa_method_experiment_token = (
    REPRESENTATION_VIDEOQA_METHOD_EXPERIMENT_TOKENS[
        REPRESENTATION_VIDEOQA_METHOD
    ]
)

# ------------------------------------------------------------
# Representation source configuration
# ------------------------------------------------------------

# All experiments use the shared CLIP text representation source.
TEXT_REPRESENTATION_SOURCE = (
    DEFAULT_TEXT_REPRESENTATION_SOURCE
)

# Select the video representation source from the experiment type.
if EXPERIMENT_TYPE == "autoencoder":

    VIDEO_REPRESENTATION_SOURCE = (
        AUTOENCODER_VIDEO_REPRESENTATION_SOURCE
    )

elif EXPERIMENT_TYPE == "clip_video":

    VIDEO_REPRESENTATION_SOURCE = (
        CLIP_VIDEO_REPRESENTATION_SOURCE
    )

elif EXPERIMENT_TYPE == "hybrid_video":

    VIDEO_REPRESENTATION_SOURCE = (
        HYBRID_VIDEO_REPRESENTATION_SOURCE
    )

else:
    raise ValueError(
        f"Unsupported EXPERIMENT_TYPE: "
        f"{EXPERIMENT_TYPE}"
    )

# ------------------------------------------------------------
# Method/source compatibility validation
# ------------------------------------------------------------

# Cosine similarity requires directly comparable CLIP embeddings.
if (
    REPRESENTATION_VIDEOQA_METHOD
    == "cosine_similarity"
):

    if (
        VIDEO_REPRESENTATION_SOURCE
        != CLIP_VIDEO_REPRESENTATION_SOURCE
    ):
        raise ValueError(
            "cosine_similarity is only supported for "
            "CLIP video representations. "
            f"Current video source: "
            f"{VIDEO_REPRESENTATION_SOURCE}"
        )

    if (
        TEXT_REPRESENTATION_SOURCE
        != DEFAULT_TEXT_REPRESENTATION_SOURCE
    ):
        raise ValueError(
            "cosine_similarity currently expects "
            "CLIP text representations. "
            f"Current text source: "
            f"{TEXT_REPRESENTATION_SOURCE}"
        )

# Learned methods support CLIP, autoencoder, and hybrid video inputs.
learned_prediction_methods = {
    "fusion_mlp_classifier",
    "interaction_fusion_classifier",
    "gated_fusion_classifier",
    "bilinear_fusion_classifier",
}

if (
    REPRESENTATION_VIDEOQA_METHOD
    in learned_prediction_methods
):

    if VIDEO_REPRESENTATION_SOURCE not in {
        CLIP_VIDEO_REPRESENTATION_SOURCE,
        AUTOENCODER_VIDEO_REPRESENTATION_SOURCE,
        HYBRID_VIDEO_REPRESENTATION_SOURCE,
    }:
        raise ValueError(
            f"{REPRESENTATION_VIDEOQA_METHOD} does not "
            "support the configured video representation "
            f"source: {VIDEO_REPRESENTATION_SOURCE}"
        )

    if (
        TEXT_REPRESENTATION_SOURCE
        != DEFAULT_TEXT_REPRESENTATION_SOURCE
    ):
        raise ValueError(
            f"{REPRESENTATION_VIDEOQA_METHOD} currently "
            "expects CLIP text representations. "
            f"Current text source: "
            f"{TEXT_REPRESENTATION_SOURCE}"
        )

# ------------------------------------------------------------
# Development / evaluation configuration
# ------------------------------------------------------------

# Copy shared dataset settings into notebook-level variables.
evaluation_split = EVALUATION_SPLIT
development_subset_size = DEVELOPMENT_SUBSET_SIZE
random_seed = RANDOM_SEED

answer_mode = ANSWER_MODE
choice_columns = CHOICE_COLUMNS
question_column = QUESTION_COLUMN
video_id_column = VIDEO_ID_COLUMN

ground_truth_answer_column = (
    GROUND_TRUTH_ANSWER_COLUMN
)

# ------------------------------------------------------------
# Input artifact configuration
# ------------------------------------------------------------

# CLIP text representations are common to all experiment types.
TEXT_REPRESENTATIONS_CSV = (
    CLIP_TEXT_REPRESENTATIONS_LOCAL_CSV
)

# Select the local video artifact required by the active source.
if (
    VIDEO_REPRESENTATION_SOURCE
    == CLIP_VIDEO_REPRESENTATION_SOURCE
):

    VIDEO_REPRESENTATIONS_CSV = (
        CLIP_VIDEO_REPRESENTATIONS_LOCAL_CSV
    )

elif (
    VIDEO_REPRESENTATION_SOURCE
    == HYBRID_VIDEO_REPRESENTATION_SOURCE
):

    # Step 8-1/2 will combine these CLIP representations
    # with the source autoencoder representations.
    VIDEO_REPRESENTATIONS_CSV = (
        CLIP_VIDEO_REPRESENTATIONS_LOCAL_CSV
    )

elif (
    VIDEO_REPRESENTATION_SOURCE
    == AUTOENCODER_VIDEO_REPRESENTATION_SOURCE
):

    VIDEO_REPRESENTATIONS_CSV = (
        AUTOENCODER_LOCAL_VIDEO_REPRESENTATIONS_CSV
    )

else:
    raise ValueError(
        f"Unsupported VIDEO_REPRESENTATION_SOURCE: "
        f"{VIDEO_REPRESENTATION_SOURCE}"
    )

# ------------------------------------------------------------
# Output artifact configuration
# ------------------------------------------------------------

# Create the experiment-specific output directory.
REPRESENTATION_VIDEOQA_LOCAL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ------------------------------------------------------------
# Strict configuration validation
# ------------------------------------------------------------

# Verify that all values required by later steps are available.
required_config_values = [
    "EXPERIMENT_NAME",
    "EXPERIMENT_TYPE",
    "TEXT_REPRESENTATION_SOURCE",
    "VIDEO_REPRESENTATION_SOURCE",
    "REPRESENTATION_VIDEOQA_METHOD",
    "representation_videoqa_method_label",
    "representation_videoqa_method_experiment_token",
    "TEXT_REPRESENTATIONS_CSV",
    "VIDEO_REPRESENTATIONS_CSV",
    "REPRESENTATION_VIDEOQA_LOCAL_DIR",
    "REPRESENTATION_VIDEOQA_PREDICTIONS_CSV",
    "REPRESENTATION_VIDEOQA_VALIDATION_CSV",
    "REPRESENTATION_VIDEOQA_SUMMARY_CSV",
    "evaluation_split",
    "development_subset_size",
    "random_seed",
    "answer_mode",
    "choice_columns",
    "question_column",
    "video_id_column",
    "ground_truth_answer_column",
]

missing_config_values = [
    name
    for name in required_config_values
    if name not in globals()
]

if missing_config_values:
    raise NameError(
        "Missing required representation-based "
        "VideoQA configuration values: "
        + ", ".join(missing_config_values)
    )

# Hybrid experiments must identify the source autoencoder run.
if (
    EXPERIMENT_TYPE == "hybrid_video"
    and not SOURCE_AUTOENCODER_EXPERIMENT_NAME
):
    raise ValueError(
        "SOURCE_AUTOENCODER_EXPERIMENT_NAME is required "
        "for hybrid video experiments."
    )

# Notebook 07 currently supports multiple-choice evaluation only.
if answer_mode != "multiple_choice":
    raise ValueError(
        f"Unsupported answer mode: {answer_mode}. "
        "Notebook 07 currently supports "
        "multiple_choice only."
    )

# Confirm that the configured answer-choice count matches NExT-QA.
if len(choice_columns) != NUM_MULTIPLE_CHOICE_ANSWERS:
    raise ValueError(
        f"Expected {NUM_MULTIPLE_CHOICE_ANSWERS} "
        f"choice columns, found {len(choice_columns)}."
    )

# Confirm that all configured input artifacts exist locally.
if not Path(
    TEXT_REPRESENTATIONS_CSV
).exists():
    raise FileNotFoundError(
        "Missing text representation artifact: "
        f"{TEXT_REPRESENTATIONS_CSV}"
    )

if not Path(
    VIDEO_REPRESENTATIONS_CSV
).exists():
    raise FileNotFoundError(
        "Missing video representation artifact: "
        f"{VIDEO_REPRESENTATIONS_CSV}"
    )

if not REPRESENTATION_VIDEOQA_LOCAL_DIR.exists():
    raise FileNotFoundError(
        "Representation VideoQA local output "
        "directory was not created: "
        f"{REPRESENTATION_VIDEOQA_LOCAL_DIR}"
    )

# ------------------------------------------------------------
# Temporary experiment-name consistency validation
# ------------------------------------------------------------

# Confirm that the experiment name identifies the selected method.
expected_method_token = (
    representation_videoqa_method_experiment_token
)

if expected_method_token not in EXPERIMENT_NAME:
    raise ValueError(
        "EXPERIMENT_NAME does not appear to match the "
        "selected prediction method. "
        f"Selected method: {REPRESENTATION_VIDEOQA_METHOD}; "
        f"expected experiment token: {expected_method_token}; "
        f"experiment name: {EXPERIMENT_NAME}"
    )

# ------------------------------------------------------------
# Display active configuration
# ------------------------------------------------------------

# Collect the active settings into a readable configuration summary.
representation_videoqa_config_summary = {
    "experiment_name": EXPERIMENT_NAME,
    "experiment_type": EXPERIMENT_TYPE,
    "dataset_mode": (
        "full_split"
        if RUN_FULL_EVALUATION_SPLIT
        else "development"
    ),
    "evaluation_split": evaluation_split,
    "development_subset_size": (
        "not applicable"
        if RUN_FULL_EVALUATION_SPLIT
        else development_subset_size
    ),
    "random_seed": random_seed,
    "answer_mode": answer_mode,
    "video_representation_source":
        VIDEO_REPRESENTATION_SOURCE,
    "text_representation_source":
        TEXT_REPRESENTATION_SOURCE,
    "prediction_method":
        REPRESENTATION_VIDEOQA_METHOD,
    "prediction_method_label":
        representation_videoqa_method_label,
    "prediction_method_experiment_token":
        representation_videoqa_method_experiment_token,
    "text_representation_file":
        str(TEXT_REPRESENTATIONS_CSV),
    "video_representation_file":
        str(VIDEO_REPRESENTATIONS_CSV),
    "local_output_directory":
        str(REPRESENTATION_VIDEOQA_LOCAL_DIR),
    "predictions_output_file":
        str(REPRESENTATION_VIDEOQA_PREDICTIONS_CSV),
    "validation_output_file":
        str(REPRESENTATION_VIDEOQA_VALIDATION_CSV),
    "summary_output_file":
        str(REPRESENTATION_VIDEOQA_SUMMARY_CSV),
}

representation_videoqa_config_df = pd.DataFrame(
    representation_videoqa_config_summary.items(),
    columns=[
        "Configuration Item",
        "Value",
    ],
)

print(
    "Representation-based VideoQA "
    "configuration defined."
)

display(
    representation_videoqa_config_df
)



### 🔷 Step 3 — Verify Runtime Environment and Dependencies

* Verify that required runtime objects were initialized by previous steps.
* Confirm that the required representation artifacts are available for the selected experiment configuration.
* Validate the required NExT-QA annotation schema for representation-based VideoQA.
* Create and verify the local experiment output directory.
* Display runtime environment information, dependency versions, and hardware configuration.
* Confirm that the notebook is ready to load representation artifacts and execute the selected representation-based VideoQA workflow.

In [ ]:
# ============================================================
# Step 3: Verify Runtime Environment and Dependencies
# ============================================================

# Runtime, path, numerical, and table utilities.
import platform
from pathlib import Path

import numpy as np
import pandas as pd

print("Verifying runtime environment and dependencies...")

# ------------------------------------------------------------
# Verify required in-memory objects
# ------------------------------------------------------------

# Confirm that Steps 1 and 2 created all required objects.
required_runtime_objects = [
    "annotations_df",
    "split_summary_df",
    "TEXT_REPRESENTATIONS_CSV",
    "VIDEO_REPRESENTATIONS_CSV",
    "REPRESENTATION_VIDEOQA_LOCAL_DIR",
    "REPRESENTATION_VIDEOQA_PREDICTIONS_CSV",
    "REPRESENTATION_VIDEOQA_VALIDATION_CSV",
    "REPRESENTATION_VIDEOQA_SUMMARY_CSV",
    "EXPERIMENT_NAME",
    "EXPERIMENT_TYPE",
    "TEXT_REPRESENTATION_SOURCE",
    "VIDEO_REPRESENTATION_SOURCE",
    "REPRESENTATION_VIDEOQA_METHOD",
    "SUPPORTED_REPRESENTATION_VIDEOQA_METHODS",
    "REPRESENTATION_VIDEOQA_METHOD_LABELS",
    "REPRESENTATION_VIDEOQA_METHOD_EXPERIMENT_TOKENS",
    "DEFAULT_TEXT_REPRESENTATION_SOURCE",
    "CLIP_VIDEO_REPRESENTATION_SOURCE",
    "AUTOENCODER_VIDEO_REPRESENTATION_SOURCE",
    "HYBRID_VIDEO_REPRESENTATION_SOURCE",
    "evaluation_split",
    "development_subset_size",
    "answer_mode",
    "choice_columns",
    "question_column",
    "video_id_column",
    "ground_truth_answer_column",
]

missing_runtime_objects = [
    name
    for name in required_runtime_objects
    if name not in globals()
]

if missing_runtime_objects:
    raise NameError(
        "Missing required runtime objects: "
        + ", ".join(missing_runtime_objects)
    )

# ------------------------------------------------------------
# Verify shared prediction-method configuration
# ------------------------------------------------------------

# Confirm that the selected prediction method is supported.
if (
    REPRESENTATION_VIDEOQA_METHOD
    not in SUPPORTED_REPRESENTATION_VIDEOQA_METHODS
):
    raise ValueError(
        "Unsupported REPRESENTATION_VIDEOQA_METHOD. "
        f"Expected one of "
        f"{sorted(SUPPORTED_REPRESENTATION_VIDEOQA_METHODS)}, "
        f"found: {REPRESENTATION_VIDEOQA_METHOD}"
    )

# Verify that every supported method has a display label.
if (
    set(REPRESENTATION_VIDEOQA_METHOD_LABELS)
    != SUPPORTED_REPRESENTATION_VIDEOQA_METHODS
):
    raise ValueError(
        "REPRESENTATION_VIDEOQA_METHOD_LABELS does not match "
        "SUPPORTED_REPRESENTATION_VIDEOQA_METHODS."
    )

# Verify that every supported method has an experiment-name token.
if (
    set(REPRESENTATION_VIDEOQA_METHOD_EXPERIMENT_TOKENS)
    != SUPPORTED_REPRESENTATION_VIDEOQA_METHODS
):
    raise ValueError(
        "REPRESENTATION_VIDEOQA_METHOD_EXPERIMENT_TOKENS does not match "
        "SUPPORTED_REPRESENTATION_VIDEOQA_METHODS."
    )

# Resolve the configured label and experiment token.
prediction_method_label = (
    REPRESENTATION_VIDEOQA_METHOD_LABELS[
        REPRESENTATION_VIDEOQA_METHOD
    ]
)

prediction_method_experiment_token = (
    REPRESENTATION_VIDEOQA_METHOD_EXPERIMENT_TOKENS[
        REPRESENTATION_VIDEOQA_METHOD
    ]
)

# ------------------------------------------------------------
# Verify representation-source configuration
# ------------------------------------------------------------

# All experiments currently require CLIP text representations.
if (
    TEXT_REPRESENTATION_SOURCE
    != DEFAULT_TEXT_REPRESENTATION_SOURCE
):
    raise ValueError(
        "Text representation source does not match configuration. "
        f"Found: {TEXT_REPRESENTATION_SOURCE}; "
        f"expected: {DEFAULT_TEXT_REPRESENTATION_SOURCE}"
    )

# Define the supported video representation sources.
supported_video_representation_sources = {
    CLIP_VIDEO_REPRESENTATION_SOURCE,
    AUTOENCODER_VIDEO_REPRESENTATION_SOURCE,
    HYBRID_VIDEO_REPRESENTATION_SOURCE,
}

if (
    VIDEO_REPRESENTATION_SOURCE
    not in supported_video_representation_sources
):
    raise ValueError(
        "Unsupported video representation source. "
        f"Expected one of "
        f"{sorted(supported_video_representation_sources)}, "
        f"found: {VIDEO_REPRESENTATION_SOURCE}"
    )

# ------------------------------------------------------------
# Verify method/source compatibility
# ------------------------------------------------------------

# Cosine similarity is valid only for CLIP text and video embeddings.
if (
    REPRESENTATION_VIDEOQA_METHOD
    == "cosine_similarity"
):

    if (
        VIDEO_REPRESENTATION_SOURCE
        != CLIP_VIDEO_REPRESENTATION_SOURCE
    ):
        raise ValueError(
            "cosine_similarity requires CLIP video representations. "
            f"Found: {VIDEO_REPRESENTATION_SOURCE}"
        )

    if (
        TEXT_REPRESENTATION_SOURCE
        != DEFAULT_TEXT_REPRESENTATION_SOURCE
    ):
        raise ValueError(
            "cosine_similarity requires the configured CLIP text "
            "representation source."
        )

# All remaining methods use learned prediction models.
learned_prediction_methods = (
    SUPPORTED_REPRESENTATION_VIDEOQA_METHODS
    - {"cosine_similarity"}
)

if (
    REPRESENTATION_VIDEOQA_METHOD
    in learned_prediction_methods
):

    if (
        VIDEO_REPRESENTATION_SOURCE
        not in {
            CLIP_VIDEO_REPRESENTATION_SOURCE,
            AUTOENCODER_VIDEO_REPRESENTATION_SOURCE,
            HYBRID_VIDEO_REPRESENTATION_SOURCE,
        }
    ):
        raise ValueError(
            f"{REPRESENTATION_VIDEOQA_METHOD} does not support "
            f"video representation source "
            f"{VIDEO_REPRESENTATION_SOURCE}."
        )

# ------------------------------------------------------------
# Verify answer configuration
# ------------------------------------------------------------

# Notebook 07 currently supports multiple-choice questions only.
if answer_mode != "multiple_choice":
    raise ValueError(
        f"Unsupported answer mode: {answer_mode}. "
        "Notebook 07 supports multiple_choice only."
    )

# Confirm that the configured number of choices matches NExT-QA.
if len(choice_columns) != NUM_MULTIPLE_CHOICE_ANSWERS:
    raise ValueError(
        f"Expected {NUM_MULTIPLE_CHOICE_ANSWERS} answer-choice "
        f"columns, found {len(choice_columns)}."
    )

# ------------------------------------------------------------
# Verify artifact availability
# ------------------------------------------------------------

# Collect the required local representation files.
required_input_files = [
    Path(TEXT_REPRESENTATIONS_CSV),
    Path(VIDEO_REPRESENTATIONS_CSV),
]

missing_input_files = [
    path
    for path in required_input_files
    if not path.exists()
]

if missing_input_files:

    for path in missing_input_files:
        print(f"Missing input artifact: {path}")

    raise FileNotFoundError(
        "One or more required representation artifacts are missing."
    )

# ------------------------------------------------------------
# Verify required annotation columns
# ------------------------------------------------------------

# Confirm that annotations contain all fields needed downstream.
required_annotation_columns = [
    "split",
    video_id_column,
    question_column,
    ground_truth_answer_column,
    *choice_columns,
]

missing_annotation_columns = [
    column
    for column in required_annotation_columns
    if column not in annotations_df.columns
]

if missing_annotation_columns:
    raise ValueError(
        "annotations_df is missing required columns: "
        + ", ".join(missing_annotation_columns)
    )

# ------------------------------------------------------------
# Verify annotation split availability
# ------------------------------------------------------------

# Identify the annotation splits available in memory.
available_annotation_splits = set(
    annotations_df["split"]
    .astype(str)
    .unique()
)

# Training and evaluation splits are required for learned methods.
required_annotation_splits = {
    "train",
    evaluation_split,
}

missing_annotation_splits = (
    required_annotation_splits
    - available_annotation_splits
)

if missing_annotation_splits:
    raise ValueError(
        "annotations_df is missing required splits: "
        + ", ".join(sorted(missing_annotation_splits))
    )

# ------------------------------------------------------------
# Verify output directory
# ------------------------------------------------------------

# Create the experiment output directory if needed.
REPRESENTATION_VIDEOQA_LOCAL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

if not REPRESENTATION_VIDEOQA_LOCAL_DIR.exists():
    raise FileNotFoundError(
        "Output directory was not created: "
        f"{REPRESENTATION_VIDEOQA_LOCAL_DIR}"
    )

# ------------------------------------------------------------
# Verify configured output paths
# ------------------------------------------------------------

# Confirm that every output artifact uses the configured directory.
configured_output_paths = [
    Path(REPRESENTATION_VIDEOQA_PREDICTIONS_CSV),
    Path(REPRESENTATION_VIDEOQA_VALIDATION_CSV),
    Path(REPRESENTATION_VIDEOQA_SUMMARY_CSV),
]

for output_path in configured_output_paths:

    if output_path.parent != Path(
        REPRESENTATION_VIDEOQA_LOCAL_DIR
    ):
        raise ValueError(
            "Configured output artifact is not located inside "
            "REPRESENTATION_VIDEOQA_LOCAL_DIR: "
            f"{output_path}"
        )

# ------------------------------------------------------------
# Display runtime summary
# ------------------------------------------------------------

# Collect environment and configuration details for verification.
runtime_summary = {
    "python_version":
        platform.python_version(),

    "pandas_version":
        pd.__version__,

    "numpy_version":
        np.__version__,

    "annotation_records":
        len(annotations_df),

    "available_annotation_splits":
        ", ".join(
            sorted(available_annotation_splits)
        ),

    "text_representation_file":
        str(TEXT_REPRESENTATIONS_CSV),

    "video_representation_file":
        str(VIDEO_REPRESENTATIONS_CSV),

    "text_representation_file_exists":
        Path(TEXT_REPRESENTATIONS_CSV).exists(),

    "video_representation_file_exists":
        Path(VIDEO_REPRESENTATIONS_CSV).exists(),

    "output_directory":
        str(REPRESENTATION_VIDEOQA_LOCAL_DIR),

    "experiment_name":
        EXPERIMENT_NAME,

    "experiment_type":
        EXPERIMENT_TYPE,

    "prediction_method":
        REPRESENTATION_VIDEOQA_METHOD,

    "prediction_method_label":
        prediction_method_label,

    "prediction_method_experiment_token":
        prediction_method_experiment_token,

    "text_representation_source":
        TEXT_REPRESENTATION_SOURCE,

    "video_representation_source":
        VIDEO_REPRESENTATION_SOURCE,
}

# Convert the summary to a readable two-column table.
runtime_summary_df = pd.DataFrame(
    runtime_summary.items(),
    columns=[
        "Runtime Item",
        "Value",
    ],
)

print("Runtime environment verification complete.")

display(runtime_summary_df)



### 🔷 Step 4 — Prepare NExT-QA Training and Validation Datasets

* Prepare independent NExT-QA question-answer datasets for the training split and configured validation split.
* Apply deterministic development-subset selection by unique video when full-split execution is disabled.
* Include all question-answer records associated with each selected video.
* Reconstruct stable annotation identifiers matching the shared CLIP text representation dataset.
* Normalize video identifiers, questions, answer choices, and ground-truth labels.
* Create stable representation-based question-answer record identifiers.
* Validate required fields, duplicate identifiers, dataset splits, and multiple-choice answer labels.
* Preserve the validation dataset as the prediction target for downstream evaluation.
* Display training and validation dataset statistics for verification.


In [ ]:
# ============================================================
# Step 4: Prepare NExT-QA Training and Validation Datasets
# ============================================================

# Table and path utilities.
import pandas as pd
from pathlib import Path

print("Preparing NExT-QA training and validation datasets...")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

# Confirm that prior steps created all required objects.
required_step4_objects = [
    "annotations_df",
    "VIDEO_REPRESENTATIONS_CSV",
    "VIDEO_REPRESENTATION_SOURCE",
    "CLIP_VIDEO_REPRESENTATION_SOURCE",
    "AUTOENCODER_VIDEO_REPRESENTATION_SOURCE",
    "HYBRID_VIDEO_REPRESENTATION_SOURCE",
    "video_id_column",
    "question_column",
    "ground_truth_answer_column",
    "choice_columns",
    "evaluation_split",
    "development_subset_size",
    "random_seed",
    "RUN_FULL_EVALUATION_SPLIT",
    "NUM_MULTIPLE_CHOICE_ANSWERS",
]

missing_step4_objects = [
    name
    for name in required_step4_objects
    if name not in globals()
]

if missing_step4_objects:
    raise NameError(
        "Missing required Step 4 objects: "
        + ", ".join(missing_step4_objects)
    )

# ------------------------------------------------------------
# Validate annotation schema
# ------------------------------------------------------------

# Verify that the annotation table contains all required fields.
required_annotation_columns = [
    "split",
    "question_id",
    video_id_column,
    question_column,
    ground_truth_answer_column,
    *choice_columns,
]

missing_annotation_columns = [
    column
    for column in required_annotation_columns
    if column not in annotations_df.columns
]

if missing_annotation_columns:
    raise ValueError(
        "annotations_df is missing required columns: "
        + ", ".join(missing_annotation_columns)
    )

# ------------------------------------------------------------
# Reconstruct Notebook 05 annotation_id values
# ------------------------------------------------------------

# Work from a copy to preserve the original annotation table.
annotations_with_ids_df = annotations_df.copy()

# Preserve the original row index for stable identifier construction.
annotations_with_ids_df["annotation_source_index"] = (
    annotations_with_ids_df.index
)

# Recreate the annotation identifiers used by Notebook 05.
annotations_with_ids_df["annotation_id"] = (
    annotations_with_ids_df["split"].astype(str)
    + "_"
    + annotations_with_ids_df[video_id_column].astype(str)
    + "_"
    + annotations_with_ids_df["question_id"].astype(str)
    + "_"
    + annotations_with_ids_df[
        "annotation_source_index"
    ].astype(str)
)

# Normalize annotation identifiers before subset selection.
annotations_with_ids_df["split"] = (
    annotations_with_ids_df["split"]
    .astype(str)
    .str.strip()
)

annotations_with_ids_df[video_id_column] = (
    annotations_with_ids_df[video_id_column]
    .astype(str)
    .str.strip()
)

# ------------------------------------------------------------
# Load authoritative autoencoder development video IDs
# ------------------------------------------------------------

# Store split-specific video IDs defined by autoencoder artifacts.
autoencoder_subset_video_ids = {}

# Autoencoder development runs use their generated representation subset.
using_autoencoder_subset = (
    not RUN_FULL_EVALUATION_SPLIT
    and VIDEO_REPRESENTATION_SOURCE
    == AUTOENCODER_VIDEO_REPRESENTATION_SOURCE
)

# Hybrid development runs must use the same autoencoder video subset.
using_hybrid_subset = (
    not RUN_FULL_EVALUATION_SPLIT
    and VIDEO_REPRESENTATION_SOURCE
    == HYBRID_VIDEO_REPRESENTATION_SOURCE
)

using_autoencoder_defined_subset = (
    using_autoencoder_subset
    or using_hybrid_subset
)

# Default to the configured video representation artifact.
subset_representation_csv = Path(
    VIDEO_REPRESENTATIONS_CSV
)

if using_hybrid_subset:

    # Confirm that hybrid source-path utilities are available.
    required_hybrid_subset_objects = [
        "SOURCE_AUTOENCODER_EXPERIMENT_NAME",
        "get_hybrid_video_representation_paths",
    ]

    missing_hybrid_subset_objects = [
        name
        for name in required_hybrid_subset_objects
        if name not in globals()
    ]

    if missing_hybrid_subset_objects:
        raise NameError(
            "Missing required hybrid subset objects: "
            + ", ".join(missing_hybrid_subset_objects)
        )

    # Resolve the source autoencoder representation artifact.
    hybrid_representation_paths = (
        get_hybrid_video_representation_paths(
            SOURCE_AUTOENCODER_EXPERIMENT_NAME
        )
    )

    subset_representation_csv = Path(
        hybrid_representation_paths[
            "hybrid_autoencoder_video_representations_csv"
        ]
    )

if using_autoencoder_defined_subset:

    # Require the representation artifact that defines the subset.
    if not subset_representation_csv.exists():
        raise FileNotFoundError(
            "The autoencoder representation artifact is required "
            "to define the development video subset, but it was "
            f"not found: {subset_representation_csv}"
        )

    # Load only the columns needed to recover split video IDs.
    subset_representation_df = pd.read_csv(
        subset_representation_csv,
        usecols=[
            video_id_column,
            "split",
        ],
    )

    if subset_representation_df.empty:
        raise RuntimeError(
            "The autoencoder representation artifact was loaded "
            "but contains no records."
        )

    # Normalize representation identifiers before comparison.
    subset_representation_df[video_id_column] = (
        subset_representation_df[video_id_column]
        .astype(str)
        .str.strip()
    )

    subset_representation_df["split"] = (
        subset_representation_df["split"]
        .astype(str)
        .str.strip()
    )

    # Each video must appear once per split.
    duplicate_subset_video_ids = (
        subset_representation_df[
            [
                video_id_column,
                "split",
            ]
        ]
        .duplicated()
        .sum()
    )

    if duplicate_subset_video_ids > 0:
        raise ValueError(
            f"Found {duplicate_subset_video_ids} duplicate video/split "
            "records in the autoencoder representation artifact."
        )

    # Training and evaluation subsets must both be available.
    required_subset_splits = [
        "train",
        evaluation_split,
    ]

    for split_name in required_subset_splits:

        # Recover the authoritative video IDs for this split.
        split_video_ids = sorted(
            subset_representation_df.loc[
                subset_representation_df["split"] == split_name,
                video_id_column,
            ]
            .astype(str)
            .unique()
            .tolist()
        )

        if not split_video_ids:
            raise ValueError(
                "No autoencoder video representations were found "
                f"for split: {split_name}"
            )

        if len(split_video_ids) != development_subset_size:
            raise ValueError(
                f"Expected {development_subset_size} autoencoder videos "
                f"for split {split_name}, but found "
                f"{len(split_video_ids)}."
            )

        autoencoder_subset_video_ids[split_name] = set(
            split_video_ids
        )

# ------------------------------------------------------------
# Define reusable QA input builder
# ------------------------------------------------------------

# Build a validated QA dataset for one annotation split.
def build_qa_input_dataset(split_name):

    # Isolate the requested split.
    split_annotations_df = (
        annotations_with_ids_df[
            annotations_with_ids_df["split"] == split_name
        ]
        .copy()
        .reset_index(drop=True)
    )

    if split_annotations_df.empty:
        raise ValueError(
            f"No annotation records found for split: {split_name}"
        )

    # --------------------------------------------------------
    # Select full split or development subset
    # --------------------------------------------------------

    if RUN_FULL_EVALUATION_SPLIT:

        # Use every video available in the split.
        selected_video_ids = set(
            split_annotations_df[video_id_column]
            .astype(str)
            .unique()
        )

        subset_selection_method = "full_split"

    elif using_autoencoder_defined_subset:

        # Reuse the video subset defined by the source artifact.
        if split_name not in autoencoder_subset_video_ids:
            raise KeyError(
                "No authoritative autoencoder subset was loaded "
                f"for split: {split_name}"
            )

        selected_video_ids = (
            autoencoder_subset_video_ids[split_name]
        )

        subset_selection_method = (
            "hybrid_source_autoencoder_representation_artifact"
            if using_hybrid_subset
            else "autoencoder_representation_artifact"
        )

    else:

        # Select a reproducible development subset from annotations.
        available_development_video_ids = sorted(
            split_annotations_df[video_id_column]
            .astype(str)
            .unique()
            .tolist()
        )

        selected_video_count = min(
            development_subset_size,
            len(available_development_video_ids),
        )

        selected_video_ids = set(
            pd.Series(available_development_video_ids)
            .sample(
                n=selected_video_count,
                random_state=random_seed,
            )
            .astype(str)
            .tolist()
        )

        subset_selection_method = (
            "seeded_random_annotation_sample"
        )

    if not selected_video_ids:
        raise RuntimeError(
            f"No videos were selected from split: {split_name}"
        )

    # --------------------------------------------------------
    # Filter annotations to selected videos
    # --------------------------------------------------------

    # Retain all QA records associated with the selected videos.
    split_qa_input_df = (
        split_annotations_df[
            split_annotations_df[video_id_column]
            .astype(str)
            .isin(selected_video_ids)
        ]
        .copy()
        .sort_values(
            [
                video_id_column,
                question_column,
            ]
        )
        .reset_index(drop=True)
    )

    if split_qa_input_df.empty:
        raise RuntimeError(
            "No representation-based VideoQA input records were "
            f"generated for split: {split_name}"
        )

    # --------------------------------------------------------
    # Confirm selected video coverage
    # --------------------------------------------------------

    # Compare selected IDs with IDs represented in the QA table.
    qa_video_ids = set(
        split_qa_input_df[video_id_column]
        .astype(str)
        .unique()
    )

    selected_videos_without_annotations = sorted(
        selected_video_ids - qa_video_ids
    )

    unexpected_qa_video_ids = sorted(
        qa_video_ids - selected_video_ids
    )

    if selected_videos_without_annotations:
        raise ValueError(
            "Selected videos were missing from the annotation "
            f"records for split {split_name}: "
            + ", ".join(
                selected_videos_without_annotations[:20]
            )
        )

    if unexpected_qa_video_ids:
        raise ValueError(
            "QA records contain unexpected video IDs for split "
            f"{split_name}: "
            + ", ".join(
                unexpected_qa_video_ids[:20]
            )
        )

    # --------------------------------------------------------
    # Normalize key fields
    # --------------------------------------------------------

    # Standardize key data types before joining representations.
    split_qa_input_df[video_id_column] = (
        split_qa_input_df[video_id_column]
        .astype(str)
        .str.strip()
    )

    split_qa_input_df[question_column] = (
        split_qa_input_df[question_column]
        .astype(str)
    )

    split_qa_input_df["annotation_id"] = (
        split_qa_input_df["annotation_id"]
        .astype(str)
    )

    for choice_column in choice_columns:
        split_qa_input_df[choice_column] = (
            split_qa_input_df[choice_column]
            .astype(str)
        )

    split_qa_input_df[ground_truth_answer_column] = (
        split_qa_input_df[ground_truth_answer_column]
        .astype(int)
    )

    # --------------------------------------------------------
    # Add stable QA record ID
    # --------------------------------------------------------

    # Create a stable identifier for downstream prediction records.
    split_qa_input_df = (
        split_qa_input_df.reset_index(drop=True)
    )

    split_qa_input_df["qa_record_id"] = (
        split_qa_input_df["annotation_id"].astype(str)
        + "_representation_videoqa"
    )

    # --------------------------------------------------------
    # Strict validation
    # --------------------------------------------------------

    # QA record IDs must uniquely identify each row.
    duplicate_qa_ids = (
        split_qa_input_df["qa_record_id"]
        .duplicated()
        .sum()
    )

    if duplicate_qa_ids > 0:
        raise ValueError(
            f"Found {duplicate_qa_ids} duplicate qa_record_id "
            f"values for split: {split_name}."
        )

    # Annotation IDs must also remain unique.
    duplicate_annotation_ids = (
        split_qa_input_df["annotation_id"]
        .duplicated()
        .sum()
    )

    if duplicate_annotation_ids > 0:
        raise ValueError(
            f"Found {duplicate_annotation_ids} duplicate "
            f"annotation_id values for split: {split_name}."
        )

    if (
        len(choice_columns)
        != NUM_MULTIPLE_CHOICE_ANSWERS
    ):
        raise ValueError(
            f"Expected {NUM_MULTIPLE_CHOICE_ANSWERS} answer "
            f"choices, but found {len(choice_columns)}."
        )

    # Ground-truth labels must map to valid choice indices.
    invalid_ground_truth_values = sorted(
        set(
            split_qa_input_df[
                ground_truth_answer_column
            ].unique()
        )
        - set(
            range(NUM_MULTIPLE_CHOICE_ANSWERS)
        )
    )

    if invalid_ground_truth_values:
        raise ValueError(
            "Found invalid ground-truth answer values: "
            + str(invalid_ground_truth_values)
        )

    # Required QA fields must be complete.
    missing_required_values = (
        split_qa_input_df[
            [
                "qa_record_id",
                "annotation_id",
                "split",
                video_id_column,
                question_column,
                ground_truth_answer_column,
                *choice_columns,
            ]
        ]
        .isna()
        .sum()
        .sum()
    )

    if missing_required_values > 0:
        raise ValueError(
            f"Found {missing_required_values} missing required "
            f"QA values for split: {split_name}."
        )

    return (
        split_annotations_df,
        split_qa_input_df,
        subset_selection_method,
    )


# ------------------------------------------------------------
# Build training and validation inputs
# ------------------------------------------------------------

# Build the training QA dataset.
(
    train_annotations_df,
    train_qa_input_df,
    training_subset_selection_method,
) = build_qa_input_dataset("train")

# Build the evaluation QA dataset.
(
    evaluation_annotations_df,
    validation_qa_input_df,
    validation_subset_selection_method,
) = build_qa_input_dataset(evaluation_split)

# Preserve existing downstream evaluation dataset name.
qa_input_df = validation_qa_input_df.copy()

# Record whether this run uses full or development data.
dataset_mode_label = (
    "full_split"
    if RUN_FULL_EVALUATION_SPLIT
    else "development"
)

# ------------------------------------------------------------
# Cross-split strict validation
# ------------------------------------------------------------

# Training and validation videos must remain disjoint.
train_video_ids = set(
    train_qa_input_df[video_id_column]
    .astype(str)
    .unique()
)

validation_video_ids = set(
    validation_qa_input_df[video_id_column]
    .astype(str)
    .unique()
)

overlapping_train_validation_videos = sorted(
    train_video_ids & validation_video_ids
)

if overlapping_train_validation_videos:
    raise ValueError(
        "Training and validation QA datasets contain overlapping "
        "video IDs: "
        + ", ".join(
            overlapping_train_validation_videos[:20]
        )
    )

# ------------------------------------------------------------
# Display summary
# ------------------------------------------------------------

# Report the prepared dataset sizes and subset sources.
print(
    "NExT-QA training and validation datasets "
    "prepared successfully."
)

print(f"Dataset mode             : {dataset_mode_label}")
print("Training split           : train")
print(f"Validation split         : {evaluation_split}")

print(
    f"Training subset source   : "
    f"{training_subset_selection_method}"
)

print(
    f"Validation subset source : "
    f"{validation_subset_selection_method}"
)

print(
    f"Training source records  : "
    f"{len(train_annotations_df):,}"
)

print(
    f"Validation source records: "
    f"{len(evaluation_annotations_df):,}"
)

print(
    f"Training QA records      : "
    f"{len(train_qa_input_df):,}"
)

print(
    f"Validation QA records    : "
    f"{len(validation_qa_input_df):,}"
)

print(
    f"Training unique videos   : "
    f"{train_qa_input_df[video_id_column].nunique():,}"
)

print(
    f"Validation unique videos : "
    f"{validation_qa_input_df[video_id_column].nunique():,}"
)

print(f"Answer mode              : {answer_mode}")
print(f"Answer choices           : {len(choice_columns)}")

print("\nValidation QA Input Preview:")

display(
    validation_qa_input_df[
        [
            "qa_record_id",
            "annotation_id",
            "split",
            video_id_column,
            question_column,
            ground_truth_answer_column,
            *choice_columns,
        ]
    ].head(10)
)



### 🔷 Step 5 — Load CLIP Text Representations

* Load the shared CLIP question–answer representation artifact generated by Notebook 05.
* Validate the required metadata and CLIP text embedding columns.
* Identify the CLIP text embedding dimensionality.
* Verify that all embedding values are numeric, complete, and internally consistent.
* Confirm that representation identifiers are unique and that the representation type is `question_answer`.
* Filter the shared representation dataset independently for the training and validation question-answer datasets.
* Verify that every selected question-answer record contains exactly five ordered candidate-answer representations.
* Confirm complete CLIP text representation coverage for both dataset splits.
* Display previews of the filtered training and validation CLIP text representation datasets.


In [ ]:
# ============================================================
# Step 5: Load CLIP Text Representations
# ============================================================

# Table and path utilities.
import pandas as pd
from pathlib import Path

print("Loading CLIP text representations...")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

# Confirm that prior steps created all required objects.
required_step5_objects = [
    "train_qa_input_df",
    "validation_qa_input_df",
    "TEXT_REPRESENTATIONS_CSV",
    "TEXT_REPRESENTATION_SOURCE",
    "TEXT_INPUT_TYPES",
    "CLIP_TEXT_EMBEDDING_DIM",
    "NUM_MULTIPLE_CHOICE_ANSWERS",
    "choice_columns",
]

missing_step5_objects = [
    name for name in required_step5_objects
    if name not in globals()
]

if missing_step5_objects:
    raise NameError(
        "Missing required Step 5 objects: "
        + ", ".join(missing_step5_objects)
    )

# Require the local text representation artifact.
if not Path(TEXT_REPRESENTATIONS_CSV).exists():
    raise FileNotFoundError(
        f"Missing CLIP text representation file: {TEXT_REPRESENTATIONS_CSV}"
    )

# ------------------------------------------------------------
# Load text representation artifact
# ------------------------------------------------------------

# Load the full CLIP text representation dataset.
clip_text_representation_df = pd.read_csv(TEXT_REPRESENTATIONS_CSV)

if clip_text_representation_df.empty:
    raise RuntimeError(
        "CLIP text representation artifact was loaded but is empty."
    )

# ------------------------------------------------------------
# Validate required columns
# ------------------------------------------------------------

# Verify that metadata and representation fields are available.
required_text_columns = [
    "record_id",
    video_id_column,
    "question_id",
    "annotation_id",
    "text_type",
    "choice_index",
    "text",
    "answer",
    "ground_truth_text",
    "split",
    "representation_source",
]

missing_text_columns = [
    col for col in required_text_columns
    if col not in clip_text_representation_df.columns
]

if missing_text_columns:
    raise ValueError(
        "CLIP text representation dataset is missing required columns: "
        + ", ".join(missing_text_columns)
    )

# ------------------------------------------------------------
# Identify embedding columns
# ------------------------------------------------------------

# Locate and numerically order the CLIP embedding dimensions.
clip_text_columns = sorted(
    [
        col
        for col in clip_text_representation_df.columns
        if col.startswith("clip_text_")
        and col.replace("clip_text_", "").isdigit()
    ],
    key=lambda col: int(col.replace("clip_text_", ""))
)

text_embedding_dimension = len(clip_text_columns)

if text_embedding_dimension == 0:
    raise ValueError("No CLIP text embedding columns were found.")

# Confirm that the artifact uses the configured CLIP dimension.
if text_embedding_dimension != CLIP_TEXT_EMBEDDING_DIM:
    raise ValueError(
        f"Expected {CLIP_TEXT_EMBEDDING_DIM} CLIP text dimensions, "
        f"found {text_embedding_dimension}."
    )

# ------------------------------------------------------------
# Strict validation
# ------------------------------------------------------------

# Embedding vectors must be complete.
missing_embedding_values = (
    clip_text_representation_df[clip_text_columns]
    .isna()
    .sum()
    .sum()
)

if missing_embedding_values > 0:
    raise ValueError(
        f"Found {missing_embedding_values} missing CLIP text embedding values."
    )

# Every embedding dimension must contain numeric values.
non_numeric_embedding_columns = [
    col for col in clip_text_columns
    if not pd.api.types.is_numeric_dtype(clip_text_representation_df[col])
]

if non_numeric_embedding_columns:
    raise TypeError(
        "Found non-numeric CLIP text embedding columns: "
        + ", ".join(non_numeric_embedding_columns[:20])
    )

# Record IDs must uniquely identify representation rows.
duplicate_text_record_ids = (
    clip_text_representation_df["record_id"]
    .duplicated()
    .sum()
)

if duplicate_text_record_ids > 0:
    raise ValueError(
        f"Found {duplicate_text_record_ids} duplicate CLIP text record_id values."
    )

# Verify that the artifact contains only configured text types.
expected_text_types = set(TEXT_INPUT_TYPES)

actual_text_types = set(
    clip_text_representation_df["text_type"]
    .astype(str)
    .unique()
)

unexpected_text_types = actual_text_types - expected_text_types

if unexpected_text_types:
    raise ValueError(
        "Unexpected CLIP text types found: "
        + ", ".join(sorted(unexpected_text_types))
    )

# Notebook 07 uses one question-answer embedding per answer choice.
if actual_text_types != {"question_answer"}:
    raise ValueError(
        "Notebook 07 expects CLIP text representations with only "
        "text_type='question_answer'. "
        f"Found: {sorted(actual_text_types)}"
    )

# Confirm that all rows use the configured representation source.
unexpected_sources = set(
    clip_text_representation_df["representation_source"]
    .astype(str)
    .unique()
) - {TEXT_REPRESENTATION_SOURCE}

if unexpected_sources:
    raise ValueError(
        "Unexpected text representation sources found: "
        + ", ".join(sorted(unexpected_sources))
    )

# ------------------------------------------------------------
# Filter to training and validation QA records
# ------------------------------------------------------------

# Normalize join keys and categorical values.
clip_text_representation_df["annotation_id"] = (
    clip_text_representation_df["annotation_id"]
    .astype(str)
)

clip_text_representation_df["split"] = (
    clip_text_representation_df["split"]
    .astype(str)
)

clip_text_representation_df["choice_index"] = (
    clip_text_representation_df["choice_index"]
    .astype(int)
)

clip_text_representation_df["answer"] = (
    clip_text_representation_df["answer"]
    .astype(int)
)

train_qa_input_df["annotation_id"] = (
    train_qa_input_df["annotation_id"]
    .astype(str)
)

validation_qa_input_df["annotation_id"] = (
    validation_qa_input_df["annotation_id"]
    .astype(str)
)

# Collect annotation IDs selected in Step 4.
train_annotation_ids = set(train_qa_input_df["annotation_id"])
validation_annotation_ids = set(validation_qa_input_df["annotation_id"])

# Retain text representations for the selected training records.
train_clip_text_df = (
    clip_text_representation_df[
        clip_text_representation_df["annotation_id"].isin(train_annotation_ids)
        &
        (clip_text_representation_df["split"] == "train")
    ]
    .copy()
    .reset_index(drop=True)
)

# Retain text representations for the selected evaluation records.
validation_clip_text_df = (
    clip_text_representation_df[
        clip_text_representation_df["annotation_id"].isin(validation_annotation_ids)
        &
        (clip_text_representation_df["split"] == evaluation_split)
    ]
    .copy()
    .reset_index(drop=True)
)

# Preserve existing downstream evaluation text representation name
filtered_clip_text_df = validation_clip_text_df.copy()

# Expect one representation for every answer choice.
expected_train_text_records = (
    len(train_qa_input_df)
    * NUM_MULTIPLE_CHOICE_ANSWERS
)

expected_validation_text_records = (
    len(validation_qa_input_df)
    * NUM_MULTIPLE_CHOICE_ANSWERS
)

if len(train_clip_text_df) != expected_train_text_records:
    raise ValueError(
        f"Expected {expected_train_text_records:,} training CLIP text "
        f"representation records, found {len(train_clip_text_df):,}."
    )

if len(validation_clip_text_df) != expected_validation_text_records:
    raise ValueError(
        f"Expected {expected_validation_text_records:,} validation CLIP text "
        f"representation records, found {len(validation_clip_text_df):,}."
    )

# Validate answer-choice coverage for each training annotation.
train_records_per_annotation = (
    train_clip_text_df
    .groupby("annotation_id")
    .size()
)

invalid_train_record_counts = train_records_per_annotation[
    train_records_per_annotation != NUM_MULTIPLE_CHOICE_ANSWERS
]

if len(invalid_train_record_counts) > 0:
    raise ValueError(
        "Each training annotation must have exactly one question-answer "
        "text representation per answer choice."
    )

# Validate answer-choice coverage for each evaluation annotation.
validation_records_per_annotation = (
    validation_clip_text_df
    .groupby("annotation_id")
    .size()
)

invalid_validation_record_counts = validation_records_per_annotation[
    validation_records_per_annotation != NUM_MULTIPLE_CHOICE_ANSWERS
]

if len(invalid_validation_record_counts) > 0:
    raise ValueError(
        "Each validation annotation must have exactly one question-answer "
        "text representation per answer choice."
    )

# ------------------------------------------------------------
# Display summary
# ------------------------------------------------------------

# Report artifact dimensions and selected record counts.
print("CLIP question-answer text representations loaded successfully.")
print(f"Source file              : {TEXT_REPRESENTATIONS_CSV}")
print(f"Total text records       : {len(clip_text_representation_df):,}")
print(f"Training text records    : {len(train_clip_text_df):,}")
print(f"Validation text records  : {len(validation_clip_text_df):,}")
print(f"Text types               : {', '.join(sorted(actual_text_types))}")
print(f"Representation source    : {TEXT_REPRESENTATION_SOURCE}")
print(f"Embedding dimensions     : {text_embedding_dimension:,}")
print(f"Answer choices/question  : {NUM_MULTIPLE_CHOICE_ANSWERS}")

print("\nValidation CLIP Text Representation Preview:")

display(
    validation_clip_text_df[
        [
            "record_id",
            video_id_column,
            "question_id",
            "annotation_id",
            "split",
            "text_type",
            "choice_index",
            "text",
            "answer",
            "ground_truth_text",
            "representation_source",
        ]
    ].head(10)
)



### 🔷 Step 6 — Load Video Representations

* Load the configured `clip_video`, `autoencoder_video`, or `hybrid_clip_autoencoder` video representation artifact.
* Accept standardized `embedding_###` columns and convert legacy CLIP `clip_video_###` columns when necessary.
* Validate the required representation metadata, dataset split values, and embedding columns.
* Identify the video embedding dimensionality and confirm that it matches the selected representation source.
* Verify that all video embeddings are numeric, complete, and uniquely keyed by video and dataset split.
* Filter the video representation dataset independently for the training and validation videos.
* Confirm that every selected training and validation video has a matching representation.
* Display previews of the filtered training and validation video representation datasets.


In [ ]:
# ============================================================
# Step 6: Load Video Representations
# ============================================================

# Table, path, and regular-expression utilities.
import pandas as pd
from pathlib import Path
import re

print("Loading video representations...")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

# Confirm that prior steps created all required objects.
required_step6_objects = [
    "train_qa_input_df",
    "validation_qa_input_df",
    "VIDEO_REPRESENTATIONS_CSV",
    "VIDEO_REPRESENTATION_SOURCE",
    "video_id_column",
    "CLIP_VIDEO_EMBEDDING_DIM",
    "AUTOENCODER_VIDEO_EMBEDDING_DIM",
    "CLIP_VIDEO_REPRESENTATION_SOURCE",
    "HYBRID_VIDEO_REPRESENTATION_SOURCE",
]

missing_step6_objects = [
    name for name in required_step6_objects
    if name not in globals()
]

if missing_step6_objects:
    raise NameError(
        "Missing required Step 6 objects: "
        + ", ".join(missing_step6_objects)
    )

# Require the configured local video representation artifact.
if not Path(VIDEO_REPRESENTATIONS_CSV).exists():
    raise FileNotFoundError(
        f"Missing video representation file: {VIDEO_REPRESENTATIONS_CSV}"
    )

# ------------------------------------------------------------
# Load video representation artifact selected in Step 2
# ------------------------------------------------------------

# Load the complete configured video representation dataset.
video_representation_df = pd.read_csv(VIDEO_REPRESENTATIONS_CSV)

if video_representation_df.empty:
    raise RuntimeError(
        "Video representation artifact was loaded but is empty."
    )

# ------------------------------------------------------------
# Validate required columns
# ------------------------------------------------------------

# Verify that identifiers, split metadata, and source metadata exist.
required_video_representation_columns = [
    "record_id",
    video_id_column,
    "split",
    "representation_source",
]

missing_video_representation_columns = [
    col for col in required_video_representation_columns
    if col not in video_representation_df.columns
]

if missing_video_representation_columns:
    raise ValueError(
        "Video representation dataset is missing required columns: "
        + ", ".join(missing_video_representation_columns)
    )

# ------------------------------------------------------------
# Identify embedding columns
# ------------------------------------------------------------

# Locate and numerically order standardized embedding columns.
video_embedding_columns = sorted(
    [
        col for col in video_representation_df.columns
        if re.fullmatch(r"embedding_\d{3}", col)
    ],
    key=lambda col: int(col.replace("embedding_", ""))
)

# Accept and standardize legacy CLIP column names when necessary.
if (
    not video_embedding_columns
    and VIDEO_REPRESENTATION_SOURCE
    in {
        CLIP_VIDEO_REPRESENTATION_SOURCE,
        HYBRID_VIDEO_REPRESENTATION_SOURCE,
    }
):

    legacy_clip_columns = sorted(
        [
            col for col in video_representation_df.columns
            if re.fullmatch(r"clip_video_\d{3}", col)
        ],
        key=lambda col: int(col.replace("clip_video_", ""))
    )

    if legacy_clip_columns:
        rename_map = {
            old_col: f"embedding_{idx:03d}"
            for idx, old_col in enumerate(legacy_clip_columns)
        }

        video_representation_df = video_representation_df.rename(
            columns=rename_map
        )

        video_embedding_columns = list(rename_map.values())

        print(
            "CLIP video embedding columns standardized from "
            "clip_video_### to embedding_###."
        )

if not video_embedding_columns:
    raise ValueError(
        "No video embedding columns found. Expected standardized "
        "embedding_### columns. For CLIP video artifacts, clip_video_### "
        "columns are also accepted and standardized."
    )

# Determine the loaded embedding dimension.
video_embedding_dimension = len(video_embedding_columns)

# Hybrid processing initially loads the CLIP portion of the representation.
expected_video_embedding_dimension = (
    CLIP_VIDEO_EMBEDDING_DIM
    if VIDEO_REPRESENTATION_SOURCE
    in {
        CLIP_VIDEO_REPRESENTATION_SOURCE,
        HYBRID_VIDEO_REPRESENTATION_SOURCE,
    }
    else AUTOENCODER_VIDEO_EMBEDDING_DIM
)

if video_embedding_dimension != expected_video_embedding_dimension:
    raise ValueError(
        f"Expected {expected_video_embedding_dimension} video embedding "
        f"dimensions for {VIDEO_REPRESENTATION_SOURCE}, "
        f"found {video_embedding_dimension}."
    )

# ------------------------------------------------------------
# Strict validation
# ------------------------------------------------------------

# Normalize identifiers before filtering and comparison.
video_representation_df[video_id_column] = (
    video_representation_df[video_id_column]
    .astype(str)
)

video_representation_df["split"] = (
    video_representation_df["split"]
    .astype(str)
)

# Embedding vectors must be complete.
missing_embedding_values = (
    video_representation_df[video_embedding_columns]
    .isna()
    .sum()
    .sum()
)

if missing_embedding_values > 0:
    raise ValueError(
        f"Found {missing_embedding_values} missing video embedding values."
    )

# Every embedding dimension must contain numeric values.
non_numeric_embedding_columns = [
    col for col in video_embedding_columns
    if not pd.api.types.is_numeric_dtype(video_representation_df[col])
]

if non_numeric_embedding_columns:
    raise TypeError(
        "Found non-numeric video embedding columns: "
        + ", ".join(non_numeric_embedding_columns[:20])
    )

# Record IDs must uniquely identify representation rows.
duplicate_video_records = (
    video_representation_df["record_id"]
    .duplicated()
    .sum()
)

if duplicate_video_records > 0:
    raise ValueError(
        f"Found {duplicate_video_records} duplicate video representation "
        "record_id values."
    )

# Each video may have only one representation per split.
duplicate_video_split_ids = (
    video_representation_df[
        [
            video_id_column,
            "split",
        ]
    ]
    .duplicated()
    .sum()
)

if duplicate_video_split_ids > 0:
    raise ValueError(
        f"Found {duplicate_video_split_ids} duplicate video/split "
        "representation records."
    )

# Verify the representation source stored in the artifact.
actual_video_sources = set(
    video_representation_df["representation_source"]
    .astype(str)
    .unique()
)

# Hybrid runs load CLIP representations before adding autoencoder features.
expected_loaded_video_source = (
    CLIP_VIDEO_REPRESENTATION_SOURCE
    if VIDEO_REPRESENTATION_SOURCE
    == HYBRID_VIDEO_REPRESENTATION_SOURCE
    else VIDEO_REPRESENTATION_SOURCE
)

if actual_video_sources != {expected_loaded_video_source}:
    raise ValueError(
        f"Expected only video representation source "
        f"{expected_loaded_video_source}, found: "
        + ", ".join(sorted(actual_video_sources))
    )

# ------------------------------------------------------------
# Filter to training and validation QA videos
# ------------------------------------------------------------

# Normalize QA video identifiers before matching representations.
train_qa_input_df[video_id_column] = (
    train_qa_input_df[video_id_column]
    .astype(str)
)

validation_qa_input_df[video_id_column] = (
    validation_qa_input_df[video_id_column]
    .astype(str)
)

# Collect the video IDs selected during dataset preparation.
train_qa_video_ids = set(
    train_qa_input_df[video_id_column]
    .astype(str)
    .unique()
)

validation_qa_video_ids = set(
    validation_qa_input_df[video_id_column]
    .astype(str)
    .unique()
)

# Retain representations for selected training videos.
train_video_representation_df = (
    video_representation_df[
        video_representation_df[video_id_column].isin(train_qa_video_ids)
        &
        (video_representation_df["split"] == "train")
    ]
    .copy()
    .reset_index(drop=True)
)

# Retain representations for selected evaluation videos.
validation_video_representation_df = (
    video_representation_df[
        video_representation_df[video_id_column].isin(validation_qa_video_ids)
        &
        (video_representation_df["split"] == evaluation_split)
    ]
    .copy()
    .reset_index(drop=True)
)

# Preserve existing downstream evaluation video representation name
filtered_video_representation_df = validation_video_representation_df.copy()

if train_video_representation_df.empty:
    raise RuntimeError(
        "No video representations matched the selected training QA records."
    )

if validation_video_representation_df.empty:
    raise RuntimeError(
        "No video representations matched the selected validation QA records."
    )

# Confirm complete representation coverage for selected training videos.
missing_train_video_ids = sorted(
    train_qa_video_ids
    - set(
        train_video_representation_df[video_id_column]
        .astype(str)
        .unique()
    )
)

# Confirm complete representation coverage for selected validation videos.
missing_validation_video_ids = sorted(
    validation_qa_video_ids
    - set(
        validation_video_representation_df[video_id_column]
        .astype(str)
        .unique()
    )
)

if missing_train_video_ids:
    raise ValueError(
        "Missing video representations for training QA videos: "
        + ", ".join(missing_train_video_ids[:20])
    )

if missing_validation_video_ids:
    raise ValueError(
        "Missing video representations for validation QA videos: "
        + ", ".join(missing_validation_video_ids[:20])
    )

# ------------------------------------------------------------
# Display summary
# ------------------------------------------------------------

# Report source, coverage, and embedding dimensions.
print("Video representations loaded successfully.")
print(f"Source file                  : {VIDEO_REPRESENTATIONS_CSV}")
print(f"Total video records          : {len(video_representation_df):,}")
print(f"Training video records       : {len(train_video_representation_df):,}")
print(f"Validation video records     : {len(validation_video_representation_df):,}")
print(f"Training QA input videos     : {len(train_qa_video_ids):,}")
print(f"Validation QA input videos   : {len(validation_qa_video_ids):,}")
print(f"Representation source        : {VIDEO_REPRESENTATION_SOURCE}")
print(f"Embedding dimensions         : {video_embedding_dimension:,}")

print("\nValidation Video Representation Preview:")

display(
    validation_video_representation_df[
        [
            "record_id",
            video_id_column,
            "split",
            "representation_source",
        ]
    ].head(10)
)



### 🔷 Step 7 — Prepare Representation VideoQA Candidate Datasets

* Build independent candidate-level representation datasets for the training and validation splits.
* Associate each question-answer record with its corresponding video representation.
* Select the five candidate question–answer representations generated by Notebook 05.
* Construct one candidate-level record for each answer choice.
* Preserve candidate ordering, ground-truth labels, representation identifiers, questions, answers, and dataset metadata.
* Verify that each question-answer record contains exactly five candidate-answer records.
* Confirm complete text and video representation coverage.
* Validate dataset split consistency, candidate indices, duplicate records, and required identifiers.
* Prepare the standardized candidate datasets used by all supported prediction methods.



In [ ]:
# ============================================================
# Step 7: Prepare Representation VideoQA Candidate Datasets
# ============================================================

# Table utilities.
import pandas as pd

print("Preparing representation-based VideoQA candidate datasets...")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

# Confirm that previous steps produced all required datasets.
required_objects = [
    "train_qa_input_df",
    "validation_qa_input_df",
    "train_clip_text_df",
    "validation_clip_text_df",
    "train_video_representation_df",
    "validation_video_representation_df",
    "clip_text_columns",
    "video_embedding_columns",
    "video_id_column",
    "question_column",
    "ground_truth_answer_column",
    "choice_columns",
    "NUM_MULTIPLE_CHOICE_ANSWERS",
    "VIDEO_REPRESENTATION_SOURCE",
    "CLIP_VIDEO_REPRESENTATION_SOURCE",
    "HYBRID_VIDEO_REPRESENTATION_SOURCE",
    "REPRESENTATION_VIDEOQA_METHOD",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects for representation QA preparation: "
        + ", ".join(missing_objects)
    )

# Notebook 07 assumes a fixed number of answer choices.
if len(choice_columns) != NUM_MULTIPLE_CHOICE_ANSWERS:
    raise ValueError(
        f"Expected {NUM_MULTIPLE_CHOICE_ANSWERS} choices, "
        f"found {len(choice_columns)}."
    )

# ------------------------------------------------------------
# Define reusable candidate dataset builder
# ------------------------------------------------------------

# Build a candidate-answer dataset for one annotation split.
def build_representation_qa_dataset(
    qa_input_df,
    filtered_clip_text_df,
    filtered_video_representation_df,
    expected_split_name,
):

    # Work on local copies to preserve source datasets.
    qa_input_df = qa_input_df.copy()
    filtered_clip_text_df = filtered_clip_text_df.copy()
    filtered_video_representation_df = filtered_video_representation_df.copy()

    # Normalize key fields before merging.
    qa_input_df["annotation_id"] = qa_input_df["annotation_id"].astype(str)
    qa_input_df[video_id_column] = qa_input_df[video_id_column].astype(str)

    filtered_clip_text_df["annotation_id"] = (
        filtered_clip_text_df["annotation_id"].astype(str)
    )

    filtered_clip_text_df["choice_index"] = (
        filtered_clip_text_df["choice_index"].astype(int)
    )

    filtered_video_representation_df[video_id_column] = (
        filtered_video_representation_df[video_id_column].astype(str)
    )

    # Verify that each input contains only the expected split.
    qa_splits = set(qa_input_df["split"].astype(str).unique())

    if qa_splits != {expected_split_name}:
        raise ValueError(
            f"Expected QA split '{expected_split_name}', "
            f"found: {sorted(qa_splits)}"
        )

    text_splits = set(filtered_clip_text_df["split"].astype(str).unique())

    if text_splits != {expected_split_name}:
        raise ValueError(
            f"Expected text representation split '{expected_split_name}', "
            f"found: {sorted(text_splits)}"
        )

    video_splits = set(
        filtered_video_representation_df["split"].astype(str).unique()
    )

    if video_splits != {expected_split_name}:
        raise ValueError(
            f"Expected video representation split '{expected_split_name}', "
            f"found: {sorted(video_splits)}"
        )

    # Notebook 07 currently uses question-answer text representations only.
    question_answer_text_df = filtered_clip_text_df[
        filtered_clip_text_df["text_type"] == "question_answer"
    ].copy()

    if question_answer_text_df.empty:
        raise RuntimeError(
            "No question-answer text representations were found."
        )

    # Every annotation must provide one representation per answer choice.
    question_answer_count_by_annotation = (
        question_answer_text_df
        .groupby("annotation_id")
        .size()
    )

    invalid_question_answer_counts = question_answer_count_by_annotation[
        question_answer_count_by_annotation != NUM_MULTIPLE_CHOICE_ANSWERS
    ]

    if len(invalid_question_answer_counts) > 0:
        raise ValueError(
            "Each QA record must have exactly "
            f"{NUM_MULTIPLE_CHOICE_ANSWERS} question-answer representations. "
            f"Invalid records found: {len(invalid_question_answer_counts)}"
        )

    # Candidate indices must match the supported answer range.
    invalid_choice_values = sorted(
        set(question_answer_text_df["choice_index"].unique())
        - set(range(NUM_MULTIPLE_CHOICE_ANSWERS))
    )

    if invalid_choice_values:
        raise ValueError(
            "Found invalid question-answer choice_index values: "
            + str(invalid_choice_values)
        )

    # Each annotation/choice combination must be unique.
    duplicate_question_answer_rows = (
        question_answer_text_df[
            [
                "annotation_id",
                "choice_index",
            ]
        ]
        .duplicated()
        .sum()
    )

    if duplicate_question_answer_rows > 0:
        raise ValueError(
            f"Found {duplicate_question_answer_rows} duplicate "
            "annotation_id/choice_index question-answer representations."
        )

    # Build a lookup table containing one video representation per video.
    video_lookup_df = (
        filtered_video_representation_df[
            [
                video_id_column,
                "record_id",
                "representation_source",
            ]
        ]
        .rename(
            columns={
                "record_id": "video_representation_id",
                "representation_source": "video_representation_source",
            }
        )
    )

    # Hybrid processing still references CLIP video representations here.
    expected_candidate_video_source = (
        CLIP_VIDEO_REPRESENTATION_SOURCE
        if VIDEO_REPRESENTATION_SOURCE
        == HYBRID_VIDEO_REPRESENTATION_SOURCE
        else VIDEO_REPRESENTATION_SOURCE
    )

    unexpected_video_sources = set(
        video_lookup_df["video_representation_source"]
        .astype(str)
        .unique()
    ) - {expected_candidate_video_source}

    if unexpected_video_sources:
        raise ValueError(
            "Unexpected video representation sources found: "
            + ", ".join(sorted(unexpected_video_sources))
        )

    # Retain only the question-answer representation metadata needed downstream.
    question_answer_input_df = (
        question_answer_text_df[
            [
                "annotation_id",
                "record_id",
                "choice_index",
                "text",
            ]
        ]
        .rename(
            columns={
                "record_id": "question_answer_representation_id",
                "text": "question_answer_text_from_representation",
            }
        )
        .copy()
    )

    # Combine QA records with their video and text representations.
    representation_qa_df = (
        qa_input_df
        .merge(video_lookup_df, on=video_id_column, how="left")
        .merge(question_answer_input_df, on="annotation_id", how="left")
    )

    # Expect one candidate row per answer choice.
    expected_candidate_rows = (
        len(qa_input_df)
        * NUM_MULTIPLE_CHOICE_ANSWERS
    )

    if len(representation_qa_df) != expected_candidate_rows:
        raise ValueError(
            f"Expected {expected_candidate_rows:,} representation QA rows, "
            f"found {len(representation_qa_df):,}."
        )

    # Verify that all downstream fields are present.
    required_representation_qa_columns = [
        "qa_record_id",
        "annotation_id",
        "split",
        video_id_column,
        question_column,
        ground_truth_answer_column,
        "video_representation_id",
        "video_representation_source",
        "question_answer_representation_id",
        "choice_index",
        "question_answer_text_from_representation",
    ]

    missing_representation_qa_columns = [
        col for col in required_representation_qa_columns
        if col not in representation_qa_df.columns
    ]

    if missing_representation_qa_columns:
        raise ValueError(
            "representation_qa_df is missing required columns: "
            + ", ".join(missing_representation_qa_columns)
        )

    # Candidate rows must be complete.
    missing_required_values = (
        representation_qa_df[required_representation_qa_columns]
        .isna()
        .sum()
        .sum()
    )

    if missing_required_values > 0:
        raise ValueError(
            f"Found {missing_required_values} missing required values "
            "in representation_qa_df."
        )

    # Every QA record should expand into exactly five candidates.
    candidate_count_by_qa = representation_qa_df.groupby("qa_record_id").size()

    invalid_candidate_counts = candidate_count_by_qa[
        candidate_count_by_qa != NUM_MULTIPLE_CHOICE_ANSWERS
    ]

    if len(invalid_candidate_counts) > 0:
        raise ValueError(
            "Each QA record must have exactly "
            f"{NUM_MULTIPLE_CHOICE_ANSWERS} candidate answers. "
            f"Invalid QA records found: {len(invalid_candidate_counts)}"
        )

    # Each QA/candidate combination must be unique.
    duplicate_candidate_rows = (
        representation_qa_df[
            [
                "qa_record_id",
                "choice_index",
            ]
        ]
        .duplicated()
        .sum()
    )

    if duplicate_candidate_rows > 0:
        raise ValueError(
            f"Found {duplicate_candidate_rows} duplicate QA/choice rows."
        )

    return representation_qa_df


# ------------------------------------------------------------
# Build train and validation candidate datasets
# ------------------------------------------------------------

# Build candidate rows for the training split.
train_representation_qa_df = build_representation_qa_dataset(
    qa_input_df=train_qa_input_df,
    filtered_clip_text_df=train_clip_text_df,
    filtered_video_representation_df=train_video_representation_df,
    expected_split_name="train",
)

# Build candidate rows for the evaluation split.
validation_representation_qa_df = build_representation_qa_dataset(
    qa_input_df=validation_qa_input_df,
    filtered_clip_text_df=validation_clip_text_df,
    filtered_video_representation_df=validation_video_representation_df,
    expected_split_name=evaluation_split,
)

# Preserve existing downstream validation dataset name.
representation_qa_df = validation_representation_qa_df.copy()

# ------------------------------------------------------------
# Display summary
# ------------------------------------------------------------

# Report the size and structure of the candidate datasets.
print("Representation-based VideoQA candidate datasets prepared successfully.")
print(f"Prediction method          : {REPRESENTATION_VIDEOQA_METHOD}")
print(f"Training QA records        : {len(train_qa_input_df):,}")
print(f"Training candidate rows    : {len(train_representation_qa_df):,}")
print(f"Validation QA records      : {len(validation_qa_input_df):,}")
print(f"Validation candidate rows  : {len(validation_representation_qa_df):,}")
print(f"Choices per question       : {NUM_MULTIPLE_CHOICE_ANSWERS}")
print(
    "Training unique videos     : "
    f"{train_representation_qa_df[video_id_column].nunique():,}"
)
print(
    "Validation unique videos   : "
    f"{validation_representation_qa_df[video_id_column].nunique():,}"
)
print(f"Video source               : {VIDEO_REPRESENTATION_SOURCE}")
print(f"Text embedding dim         : {len(clip_text_columns):,}")
print(f"Video embedding dim        : {len(video_embedding_columns):,}")

print("\nValidation Representation QA Preview:")

display(
    validation_representation_qa_df[
        [
            "qa_record_id",
            "annotation_id",
            "split",
            video_id_column,
            question_column,
            ground_truth_answer_column,
            "choice_index",
            "question_answer_text_from_representation",
            "video_representation_id",
            "video_representation_source",
            "question_answer_representation_id",
        ]
    ].head(10)
)



### 🔷 Step 8 — Build Grouped Representation QA Samples

* Convert the candidate-level datasets from Step 7 into one grouped sample for each question-answer record.
* Preserve the five ordered candidate question-answer embeddings associated with each question.
* Attach the corresponding video representation, ground-truth answer index, and representation metadata.
* Build independent grouped sample collections for the training and validation splits.
* Validate embedding dimensions, candidate ordering, answer labels, and required representation identifiers.
* Establish the standardized input structure used by all supported representation-based prediction methods.


In [ ]:
# ============================================================
# Step 8: Build Grouped Representation QA Samples
# ============================================================

# Numerical utilities for embedding construction.
import numpy as np

print("Building grouped representation-based QA samples...")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

# Confirm that all required datasets and configuration are available.
required_objects = [
    "train_representation_qa_df",
    "validation_representation_qa_df",
    "train_clip_text_df",
    "validation_clip_text_df",
    "train_video_representation_df",
    "validation_video_representation_df",
    "clip_text_columns",
    "video_embedding_columns",
    "ground_truth_answer_column",
    "NUM_MULTIPLE_CHOICE_ANSWERS",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects for grouped QA sample construction: "
        + ", ".join(missing_objects)
    )

# ------------------------------------------------------------
# Validate embedding column configuration
# ------------------------------------------------------------

# Both text and video embedding definitions must be available.
if not clip_text_columns:
    raise ValueError("No CLIP text embedding columns were defined.")

if not video_embedding_columns:
    raise ValueError("No video embedding columns were defined.")

text_embedding_dimension = len(clip_text_columns)
video_embedding_dimension = len(video_embedding_columns)

if text_embedding_dimension <= 0:
    raise ValueError(
        f"Invalid text embedding dimension: {text_embedding_dimension}"
    )

if video_embedding_dimension <= 0:
    raise ValueError(
        f"Invalid video embedding dimension: {video_embedding_dimension}"
    )

# ------------------------------------------------------------
# Define reusable grouped-sample builder
# ------------------------------------------------------------

# Convert candidate-level rows into one grouped sample per QA record.
def build_representation_qa_samples(
    representation_qa_df,
    clip_text_df,
    video_representation_df,
    expected_split_name,
):
    """
    Convert candidate-level representation rows into grouped
    five-choice VideoQA samples.

    Each returned sample contains:
      - one video embedding,
      - five ordered question-answer embeddings,
      - one ground-truth choice index,
      - QA-level metadata,
      - candidate-level metadata.
    """

    # Work on local copies to preserve the original datasets.
    representation_qa_df = representation_qa_df.copy()
    clip_text_df = clip_text_df.copy()
    video_representation_df = video_representation_df.copy()

    # --------------------------------------------------------
    # Validate source datasets
    # --------------------------------------------------------

    # Every source dataset must contain data.
    if representation_qa_df.empty:
        raise RuntimeError(
            f"No candidate rows were provided for split "
            f"'{expected_split_name}'."
        )

    if clip_text_df.empty:
        raise RuntimeError(
            f"No CLIP text representations were provided for split "
            f"'{expected_split_name}'."
        )

    if video_representation_df.empty:
        raise RuntimeError(
            f"No video representations were provided for split "
            f"'{expected_split_name}'."
        )

    # Verify that required metadata fields are available.
    required_candidate_columns = [
        "qa_record_id",
        "split",
        "choice_index",
        "question_answer_representation_id",
        "video_representation_id",
        ground_truth_answer_column,
    ]

    missing_candidate_columns = [
        column
        for column in required_candidate_columns
        if column not in representation_qa_df.columns
    ]

    if missing_candidate_columns:
        raise ValueError(
            "Representation QA dataset is missing required columns: "
            + ", ".join(missing_candidate_columns)
        )

    if "record_id" not in clip_text_df.columns:
        raise ValueError(
            "CLIP text representation dataset is missing 'record_id'."
        )

    if "record_id" not in video_representation_df.columns:
        raise ValueError(
            "Video representation dataset is missing 'record_id'."
        )

    # Confirm that every configured embedding column exists.
    missing_text_embedding_columns = [
        column
        for column in clip_text_columns
        if column not in clip_text_df.columns
    ]

    if missing_text_embedding_columns:
        raise ValueError(
            "CLIP text representation dataset is missing embedding columns: "
            + ", ".join(missing_text_embedding_columns[:10])
        )

    missing_video_embedding_columns = [
        column
        for column in video_embedding_columns
        if column not in video_representation_df.columns
    ]

    if missing_video_embedding_columns:
        raise ValueError(
            "Video representation dataset is missing embedding columns: "
            + ", ".join(missing_video_embedding_columns[:10])
        )

    # --------------------------------------------------------
    # Validate split membership
    # --------------------------------------------------------

    # All candidate rows should belong to one split.
    candidate_splits = set(
        representation_qa_df["split"].astype(str).unique()
    )

    if candidate_splits != {expected_split_name}:
        raise ValueError(
            f"Expected candidate split '{expected_split_name}', "
            f"found: {sorted(candidate_splits)}"
        )

    # --------------------------------------------------------
    # Validate representation identifiers
    # --------------------------------------------------------

    # Representation identifiers must uniquely identify embeddings.
    if clip_text_df["record_id"].duplicated().any():
        duplicate_ids = (
            clip_text_df.loc[
                clip_text_df["record_id"].duplicated(keep=False),
                "record_id",
            ]
            .astype(str)
            .unique()
            .tolist()
        )

        raise ValueError(
            "Duplicate CLIP text representation IDs found: "
            + str(duplicate_ids[:10])
        )

    if video_representation_df["record_id"].duplicated().any():
        duplicate_ids = (
            video_representation_df.loc[
                video_representation_df["record_id"].duplicated(keep=False),
                "record_id",
            ]
            .astype(str)
            .unique()
            .tolist()
        )

        raise ValueError(
            "Duplicate video representation IDs found: "
            + str(duplicate_ids[:10])
        )

    # --------------------------------------------------------
    # Build representation lookup dictionaries
    # --------------------------------------------------------

    # Convert CLIP text embeddings into fast lookup dictionaries.
    text_embedding_lookup = {}

    for row in clip_text_df.to_dict("records"):
        embedding = np.asarray(
            [row[column] for column in clip_text_columns],
            dtype=np.float32,
        )

        if embedding.shape != (text_embedding_dimension,):
            raise ValueError(
                f"Invalid text embedding shape for {row['record_id']}: "
                f"{embedding.shape}"
            )

        if not np.isfinite(embedding).all():
            raise ValueError(
                f"Non-finite text embedding values found for "
                f"{row['record_id']}."
            )

        text_embedding_lookup[str(row["record_id"])] = embedding

    # Convert video embeddings into fast lookup dictionaries.
    video_embedding_lookup = {}

    for row in video_representation_df.to_dict("records"):
        embedding = np.asarray(
            [row[column] for column in video_embedding_columns],
            dtype=np.float32,
        )

        if embedding.shape != (video_embedding_dimension,):
            raise ValueError(
                f"Invalid video embedding shape for {row['record_id']}: "
                f"{embedding.shape}"
            )

        if not np.isfinite(embedding).all():
            raise ValueError(
                f"Non-finite video embedding values found for "
                f"{row['record_id']}."
            )

        video_embedding_lookup[str(row["record_id"])] = embedding

    # --------------------------------------------------------
    # Build one grouped sample per QA record
    # --------------------------------------------------------

    # Combine one video embedding with five ordered text embeddings.
    qa_samples = []

    grouped_candidate_rows = representation_qa_df.groupby(
        "qa_record_id",
        sort=False,
    )

    for qa_record_id, group_df in grouped_candidate_rows:

        # Preserve the expected answer-choice ordering.
        group_df = (
            group_df
            .sort_values("choice_index")
            .reset_index(drop=True)
        )

        if len(group_df) != NUM_MULTIPLE_CHOICE_ANSWERS:
            raise ValueError(
                f"QA record {qa_record_id} has {len(group_df)} candidates; "
                f"expected {NUM_MULTIPLE_CHOICE_ANSWERS}."
            )

        actual_choice_indices = group_df["choice_index"].astype(int).tolist()
        expected_choice_indices = list(
            range(NUM_MULTIPLE_CHOICE_ANSWERS)
        )

        if actual_choice_indices != expected_choice_indices:
            raise ValueError(
                f"QA record {qa_record_id} has invalid choice ordering: "
                f"{actual_choice_indices}; "
                f"expected {expected_choice_indices}."
            )

        first_row = group_df.iloc[0]

        # Every candidate row should reference the same ground-truth label.
        ground_truth_choice = int(
            first_row[ground_truth_answer_column]
        )

        if ground_truth_choice not in expected_choice_indices:
            raise ValueError(
                f"Invalid ground-truth choice for {qa_record_id}: "
                f"{ground_truth_choice}"
            )

        ground_truth_values = set(
            group_df[ground_truth_answer_column]
            .astype(int)
            .tolist()
        )

        if ground_truth_values != {ground_truth_choice}:
            raise ValueError(
                f"Inconsistent ground-truth choices for QA record "
                f"{qa_record_id}: {sorted(ground_truth_values)}"
            )

        # Each QA record should reference exactly one video embedding.
        video_representation_ids = (
            group_df["video_representation_id"]
            .astype(str)
            .unique()
            .tolist()
        )

        if len(video_representation_ids) != 1:
            raise ValueError(
                f"QA record {qa_record_id} references multiple video "
                f"representations: {video_representation_ids}"
            )

        video_representation_id = video_representation_ids[0]

        if video_representation_id not in video_embedding_lookup:
            raise KeyError(
                f"Missing video embedding: {video_representation_id}"
            )

        # Assemble the ordered text embeddings for the five candidates.
        question_answer_embeddings = []

        for _, candidate_row in group_df.iterrows():

            question_answer_representation_id = str(
                candidate_row[
                    "question_answer_representation_id"
                ]
            )

            if (
                question_answer_representation_id
                not in text_embedding_lookup
            ):
                raise KeyError(
                    "Missing question-answer embedding: "
                    f"{question_answer_representation_id}"
                )

            question_answer_embeddings.append(
                text_embedding_lookup[
                    question_answer_representation_id
                ]
            )

        question_answer_embedding_matrix = np.stack(
            question_answer_embeddings,
            axis=0,
        )

        expected_text_shape = (
            NUM_MULTIPLE_CHOICE_ANSWERS,
            text_embedding_dimension,
        )

        if (
            question_answer_embedding_matrix.shape
            != expected_text_shape
        ):
            raise ValueError(
                f"Invalid question-answer embedding shape for "
                f"{qa_record_id}: "
                f"{question_answer_embedding_matrix.shape}; "
                f"expected {expected_text_shape}."
            )

        # Store the grouped sample used by downstream prediction models.
        qa_samples.append(
            {
                "qa_record_id": str(qa_record_id),
                "metadata": first_row.to_dict(),
                "candidate_metadata": group_df.to_dict("records"),
                "video_embedding": video_embedding_lookup[
                    video_representation_id
                ],
                "question_answer_embeddings":
                    question_answer_embedding_matrix,
                "label": ground_truth_choice,
            }
        )

    if not qa_samples:
        raise RuntimeError(
            f"No grouped QA samples were created for split "
            f"'{expected_split_name}'."
        )

    return qa_samples


# ------------------------------------------------------------
# Build training and validation samples
# ------------------------------------------------------------

# Build grouped samples for model training.
train_qa_samples = build_representation_qa_samples(
    representation_qa_df=train_representation_qa_df,
    clip_text_df=train_clip_text_df,
    video_representation_df=train_video_representation_df,
    expected_split_name="train",
)

# Build grouped samples for evaluation.
validation_qa_samples = build_representation_qa_samples(
    representation_qa_df=validation_representation_qa_df,
    clip_text_df=validation_clip_text_df,
    video_representation_df=validation_video_representation_df,
    expected_split_name=evaluation_split,
)

# ------------------------------------------------------------
# Strict record-count validation
# ------------------------------------------------------------

# Confirm that every QA record produced one grouped sample.
expected_training_qa_records = (
    train_representation_qa_df["qa_record_id"].nunique()
)

expected_validation_qa_records = (
    validation_representation_qa_df["qa_record_id"].nunique()
)

if len(train_qa_samples) != expected_training_qa_records:
    raise ValueError(
        f"Expected {expected_training_qa_records:,} grouped training "
        f"QA samples, found {len(train_qa_samples):,}."
    )

if len(validation_qa_samples) != expected_validation_qa_records:
    raise ValueError(
        f"Expected {expected_validation_qa_records:,} grouped validation "
        f"QA samples, found {len(validation_qa_samples):,}."
    )

# ------------------------------------------------------------
# Initialize shared method-output contract
# ------------------------------------------------------------

# Initialize shared outputs used by all prediction methods.
all_scores = None
fusion_training_history_df = None
training_epochs = None
final_training_loss = None
fusion_embedding_dimension = None

# ------------------------------------------------------------
# Display summary
# ------------------------------------------------------------

# Report the grouped dataset sizes and embedding dimensions.
print("Grouped representation-based QA samples created successfully.")
print(f"Training QA samples      : {len(train_qa_samples):,}")
print(f"Validation QA samples    : {len(validation_qa_samples):,}")
print(f"Choices per QA sample    : {NUM_MULTIPLE_CHOICE_ANSWERS}")
print(f"Text embedding dimension : {text_embedding_dimension}")
print(f"Video embedding dimension: {video_embedding_dimension}")
print("Shared method outputs    : initialized")



### 🔷 Step 9 — Load and Validate Autoencoder Video Representations

* Load the autoencoder video representations from the selected experiment.
* Validate the representation schema, embedding dimensions, and metadata.
* Verify that the selected CLIP and autoencoder video subsets match exactly for the training and validation splits.
* Build an autoencoder embedding lookup for hybrid representation construction.



In [ ]:
# ============================================================
# Step 9: Load and Validate Autoencoder Video Representations
# ============================================================

# Numerical, table, path, and column-pattern utilities.
import numpy as np
import pandas as pd
from pathlib import Path
import re

print("Loading and validating autoencoder video representations...")

# ------------------------------------------------------------
# Preserve existing non-hybrid execution paths
# ------------------------------------------------------------

# Leave standard CLIP and autoencoder experiments unchanged.
if EXPERIMENT_TYPE != "hybrid_video":

    print(
        "Autoencoder representation loading skipped. "
        f"Experiment type: {EXPERIMENT_TYPE}"
    )

else:

    # --------------------------------------------------------
    # Verify required Step 9 inputs
    # --------------------------------------------------------

    # Confirm that the hybrid source configuration and selected
    # CLIP video datasets were created by earlier notebook steps.
    required_step9_objects = [
        "SOURCE_AUTOENCODER_EXPERIMENT_NAME",
        "get_hybrid_video_representation_paths",
        "train_video_representation_df",
        "validation_video_representation_df",
        "video_id_column",
        "evaluation_split",
        "AUTOENCODER_VIDEO_EMBEDDING_DIM",
        "AUTOENCODER_VIDEO_REPRESENTATION_SOURCE",
    ]

    missing_step9_objects = [
        name
        for name in required_step9_objects
        if name not in globals()
    ]

    if missing_step9_objects:
        raise NameError(
            "Missing required Step 9 objects: "
            + ", ".join(missing_step9_objects)
        )

    # --------------------------------------------------------
    # Resolve source representation paths
    # --------------------------------------------------------

    # Resolve the autoencoder artifact associated with this hybrid run.
    hybrid_representation_paths = (
        get_hybrid_video_representation_paths(
            SOURCE_AUTOENCODER_EXPERIMENT_NAME
        )
    )

    autoencoder_video_representations_csv = Path(
        hybrid_representation_paths[
            "hybrid_autoencoder_video_representations_csv"
        ]
    )

    if not autoencoder_video_representations_csv.exists():
        raise FileNotFoundError(
            "Missing source autoencoder video representation file: "
            f"{autoencoder_video_representations_csv}"
        )

    # --------------------------------------------------------
    # Load source autoencoder video representations
    # --------------------------------------------------------

    # Load the autoencoder vectors that will complement CLIP.
    autoencoder_video_representation_df = pd.read_csv(
        autoencoder_video_representations_csv
    )

    if autoencoder_video_representation_df.empty:
        raise RuntimeError(
            "The source autoencoder video representation "
            "artifact is empty."
        )

    # Verify required identifiers and representation metadata.
    required_autoencoder_columns = [
        "record_id",
        video_id_column,
        "split",
        "representation_source",
    ]

    missing_autoencoder_columns = [
        column
        for column in required_autoencoder_columns
        if column not in autoencoder_video_representation_df.columns
    ]

    if missing_autoencoder_columns:
        raise ValueError(
            "Autoencoder video representation dataset is missing "
            "required columns: "
            + ", ".join(missing_autoencoder_columns)
        )

    # --------------------------------------------------------
    # Identify autoencoder embedding columns
    # --------------------------------------------------------

    # Locate and numerically order standardized embedding columns.
    autoencoder_embedding_columns = sorted(
        [
            column
            for column in autoencoder_video_representation_df.columns
            if re.fullmatch(r"embedding_\d{3}", column)
        ],
        key=lambda column: int(
            column.replace("embedding_", "")
        ),
    )

    if not autoencoder_embedding_columns:
        raise ValueError(
            "No standardized autoencoder embedding columns were "
            "found. Expected embedding_### columns."
        )

    # Confirm that the artifact matches the configured latent dimension.
    if (
        len(autoencoder_embedding_columns)
        != AUTOENCODER_VIDEO_EMBEDDING_DIM
    ):
        raise ValueError(
            f"Expected {AUTOENCODER_VIDEO_EMBEDDING_DIM} "
            "autoencoder embedding dimensions, found "
            f"{len(autoencoder_embedding_columns)}."
        )

    # --------------------------------------------------------
    # Strict autoencoder artifact validation
    # --------------------------------------------------------

    # Normalize identifiers before split and coverage comparisons.
    autoencoder_video_representation_df[video_id_column] = (
        autoencoder_video_representation_df[video_id_column]
        .astype(str)
    )

    autoencoder_video_representation_df["split"] = (
        autoencoder_video_representation_df["split"]
        .astype(str)
    )

    # The source artifact must contain only autoencoder vectors.
    actual_autoencoder_sources = set(
        autoencoder_video_representation_df[
            "representation_source"
        ]
        .astype(str)
        .unique()
    )

    if actual_autoencoder_sources != {
        AUTOENCODER_VIDEO_REPRESENTATION_SOURCE
    }:
        raise ValueError(
            "Expected only autoencoder video representation source "
            f"'{AUTOENCODER_VIDEO_REPRESENTATION_SOURCE}', found: "
            + ", ".join(sorted(actual_autoencoder_sources))
        )

    # Each video must have at most one representation per split.
    duplicate_autoencoder_video_split_ids = (
        autoencoder_video_representation_df[
            [
                video_id_column,
                "split",
            ]
        ]
        .duplicated()
        .sum()
    )

    if duplicate_autoencoder_video_split_ids > 0:
        raise ValueError(
            f"Found {duplicate_autoencoder_video_split_ids} duplicate "
            "autoencoder video/split representation records."
        )

    # Autoencoder vectors must be complete and numeric.
    missing_autoencoder_embedding_values = (
        autoencoder_video_representation_df[
            autoencoder_embedding_columns
        ]
        .isna()
        .sum()
        .sum()
    )

    if missing_autoencoder_embedding_values > 0:
        raise ValueError(
            f"Found {missing_autoencoder_embedding_values} missing "
            "autoencoder embedding values."
        )

    non_numeric_autoencoder_columns = [
        column
        for column in autoencoder_embedding_columns
        if not pd.api.types.is_numeric_dtype(
            autoencoder_video_representation_df[column]
        )
    ]

    if non_numeric_autoencoder_columns:
        raise TypeError(
            "Found non-numeric autoencoder embedding columns: "
            + ", ".join(
                non_numeric_autoencoder_columns[:20]
            )
        )

    # --------------------------------------------------------
    # Compare selected CLIP and autoencoder video subsets
    # --------------------------------------------------------

    # Recover the exact CLIP video subsets selected in earlier steps.
    selected_train_clip_video_ids = set(
        train_video_representation_df[video_id_column]
        .astype(str)
        .unique()
    )

    selected_validation_clip_video_ids = set(
        validation_video_representation_df[video_id_column]
        .astype(str)
        .unique()
    )

    # Isolate matching autoencoder source splits.
    source_train_autoencoder_df = (
        autoencoder_video_representation_df[
            autoencoder_video_representation_df["split"]
            == "train"
        ]
        .copy()
        .reset_index(drop=True)
    )

    source_validation_autoencoder_df = (
        autoencoder_video_representation_df[
            autoencoder_video_representation_df["split"]
            == evaluation_split
        ]
        .copy()
        .reset_index(drop=True)
    )

    source_train_autoencoder_video_ids = set(
        source_train_autoencoder_df[video_id_column]
        .astype(str)
        .unique()
    )

    source_validation_autoencoder_video_ids = set(
        source_validation_autoencoder_df[video_id_column]
        .astype(str)
        .unique()
    )

    # Compare both directions to require exact subset equality.
    missing_train_from_autoencoder = sorted(
        selected_train_clip_video_ids
        - source_train_autoencoder_video_ids
    )

    extra_train_in_autoencoder = sorted(
        source_train_autoencoder_video_ids
        - selected_train_clip_video_ids
    )

    missing_validation_from_autoencoder = sorted(
        selected_validation_clip_video_ids
        - source_validation_autoencoder_video_ids
    )

    extra_validation_in_autoencoder = sorted(
        source_validation_autoencoder_video_ids
        - selected_validation_clip_video_ids
    )

    print("\nHybrid Video Coverage Validation")
    print("-" * 60)
    print(
        "Selected training CLIP videos      : "
        f"{len(selected_train_clip_video_ids):,}"
    )
    print(
        "Source training AE videos          : "
        f"{len(source_train_autoencoder_video_ids):,}"
    )
    print(
        "Training videos missing from AE    : "
        f"{len(missing_train_from_autoencoder):,}"
    )
    print(
        "Training AE videos not selected    : "
        f"{len(extra_train_in_autoencoder):,}"
    )
    print(
        "Selected validation CLIP videos    : "
        f"{len(selected_validation_clip_video_ids):,}"
    )
    print(
        "Source validation AE videos        : "
        f"{len(source_validation_autoencoder_video_ids):,}"
    )
    print(
        "Validation videos missing from AE  : "
        f"{len(missing_validation_from_autoencoder):,}"
    )
    print(
        "Validation AE videos not selected  : "
        f"{len(extra_validation_in_autoencoder):,}"
    )

    # Fail before fusion if either representation uses a different subset.
    if (
        missing_train_from_autoencoder
        or extra_train_in_autoencoder
        or missing_validation_from_autoencoder
        or extra_validation_in_autoencoder
    ):
        diagnostic_messages = []

        if missing_train_from_autoencoder:
            diagnostic_messages.append(
                "Training videos missing from AE: "
                + ", ".join(
                    missing_train_from_autoencoder[:20]
                )
            )

        if extra_train_in_autoencoder:
            diagnostic_messages.append(
                "Training AE videos not selected: "
                + ", ".join(
                    extra_train_in_autoencoder[:20]
                )
            )

        if missing_validation_from_autoencoder:
            diagnostic_messages.append(
                "Validation videos missing from AE: "
                + ", ".join(
                    missing_validation_from_autoencoder[:20]
                )
            )

        if extra_validation_in_autoencoder:
            diagnostic_messages.append(
                "Validation AE videos not selected: "
                + ", ".join(
                    extra_validation_in_autoencoder[:20]
                )
            )

        raise ValueError(
            "The selected CLIP and autoencoder video subsets do "
            "not match exactly.\n"
            + "\n".join(diagnostic_messages)
        )

    # --------------------------------------------------------
    # Build autoencoder embedding lookup
    # --------------------------------------------------------

    # Index validated autoencoder vectors by split and video ID.
    autoencoder_embedding_lookup = {}

    for row in autoencoder_video_representation_df.to_dict(
        "records"
    ):

        lookup_key = (
            str(row["split"]),
            str(row[video_id_column]),
        )

        embedding = np.asarray(
            [
                row[column]
                for column in autoencoder_embedding_columns
            ],
            dtype=np.float32,
        )

        if embedding.shape != (
            AUTOENCODER_VIDEO_EMBEDDING_DIM,
        ):
            raise ValueError(
                "Invalid autoencoder embedding shape for "
                f"{lookup_key}: {embedding.shape}"
            )

        if not np.isfinite(embedding).all():
            raise ValueError(
                "Non-finite autoencoder embedding values found "
                f"for {lookup_key}."
            )

        autoencoder_embedding_lookup[lookup_key] = embedding

    # --------------------------------------------------------
    # Display Step 9 summary
    # --------------------------------------------------------

    print(
        "\nAutoencoder video representations loaded "
        "and validated successfully."
    )
    print(
        "Source autoencoder experiment : "
        f"{SOURCE_AUTOENCODER_EXPERIMENT_NAME}"
    )
    print(
        "Autoencoder source file       : "
        f"{autoencoder_video_representations_csv}"
    )
    print(
        "Autoencoder embedding dim     : "
        f"{len(autoencoder_embedding_columns)}"
    )
    print(
        "Training videos aligned       : "
        f"{len(selected_train_clip_video_ids):,}"
    )
    print(
        "Validation videos aligned     : "
        f"{len(selected_validation_clip_video_ids):,}"
    )
    print(
        "Embedding lookup records      : "
        f"{len(autoencoder_embedding_lookup):,}"
    )



### 🔷 Step 10 — Build Hybrid CLIP + Autoencoder Video Representations

* Retrieve the corresponding CLIP and autoencoder video representations for each grouped QA sample.
* Optionally L2-normalize each representation independently.
* Concatenate the 512-dimensional CLIP and 256-dimensional autoencoder embeddings to produce a 768-dimensional hybrid representation.
* Replace the grouped video representations and update the associated metadata.


In [ ]:
# ============================================================
# Step 10: Build Hybrid CLIP + Autoencoder Video Representations
# ============================================================

print("Building hybrid CLIP + autoencoder video representations...")

# ------------------------------------------------------------
# Preserve existing non-hybrid execution paths
# ------------------------------------------------------------

# Hybrid construction applies only to the hybrid experiment path.
if EXPERIMENT_TYPE != "hybrid_video":

    print(
        "Hybrid video representation construction skipped. "
        f"Experiment type: {EXPERIMENT_TYPE}"
    )

else:

    # --------------------------------------------------------
    # Verify required Step 10 inputs
    # --------------------------------------------------------

    # Confirm that Step 9 completed and that grouped QA samples
    # still contain their original CLIP video representations.
    required_step10_objects = [
        "train_qa_samples",
        "validation_qa_samples",
        "autoencoder_embedding_lookup",
        "video_embedding_dimension",
        "video_id_column",
        "evaluation_split",
        "CLIP_VIDEO_EMBEDDING_DIM",
        "AUTOENCODER_VIDEO_EMBEDDING_DIM",
        "HYBRID_VIDEO_EMBEDDING_DIM",
        "HYBRID_VIDEO_REPRESENTATION_SOURCE",
        "HYBRID_NORMALIZE_CLIP_VIDEO",
        "HYBRID_NORMALIZE_AUTOENCODER_VIDEO",
        "HYBRID_VIDEO_COMBINATION_METHOD",
    ]

    missing_step10_objects = [
        name
        for name in required_step10_objects
        if name not in globals()
    ]

    if missing_step10_objects:
        raise NameError(
            "Missing required Step 10 objects: "
            + ", ".join(missing_step10_objects)
        )

    # The current hybrid design combines embeddings by concatenation.
    if (
        HYBRID_VIDEO_COMBINATION_METHOD
        != "concatenate"
    ):
        raise ValueError(
            "Step 10 currently supports only hybrid video "
            "combination method 'concatenate'. "
            f"Configured method: "
            f"{HYBRID_VIDEO_COMBINATION_METHOD}"
        )

    # Grouped samples must still contain the original CLIP vectors.
    if (
        video_embedding_dimension
        != CLIP_VIDEO_EMBEDDING_DIM
    ):
        raise ValueError(
            "Expected the grouped QA samples to contain "
            f"{CLIP_VIDEO_EMBEDDING_DIM}-dimensional CLIP video "
            f"embeddings before hybrid construction, found "
            f"{video_embedding_dimension}."
        )

    # --------------------------------------------------------
    # Define strict L2 normalization helper
    # --------------------------------------------------------

    # Normalize one vector while rejecting invalid or zero-length inputs.
    def l2_normalize_embedding(
        embedding,
        embedding_name,
    ):

        embedding = np.asarray(
            embedding,
            dtype=np.float32,
        )

        if not np.isfinite(embedding).all():
            raise ValueError(
                f"Non-finite values found in {embedding_name}."
            )

        embedding_norm = float(
            np.linalg.norm(embedding)
        )

        if embedding_norm <= 0.0:
            raise ValueError(
                f"Cannot L2-normalize zero-length "
                f"{embedding_name}."
            )

        return (
            embedding / embedding_norm
        ).astype(np.float32)

    # --------------------------------------------------------
    # Transform grouped QA samples
    # --------------------------------------------------------

    # Replace each CLIP vector with its concatenated hybrid vector.
    def build_hybrid_qa_samples(
        qa_samples,
        expected_split_name,
    ):

        transformed_video_ids = set()

        for sample in qa_samples:

            # Grouped samples must retain required metadata and vectors.
            if "metadata" not in sample:
                raise ValueError(
                    "Grouped QA sample is missing metadata."
                )

            if "video_embedding" not in sample:
                raise ValueError(
                    "Grouped QA sample is missing video_embedding."
                )

            sample_split = str(
                sample["metadata"]["split"]
            )

            sample_video_id = str(
                sample["metadata"][video_id_column]
            )

            if sample_split != expected_split_name:
                raise ValueError(
                    f"Expected grouped sample split "
                    f"'{expected_split_name}', found "
                    f"'{sample_split}'."
                )

            # Preserve the original CLIP vector before concatenation.
            clip_embedding = np.asarray(
                sample["video_embedding"],
                dtype=np.float32,
            )

            if clip_embedding.shape != (
                CLIP_VIDEO_EMBEDDING_DIM,
            ):
                raise ValueError(
                    "Invalid CLIP video embedding shape for "
                    f"{sample_video_id}: {clip_embedding.shape}"
                )

            autoencoder_lookup_key = (
                sample_split,
                sample_video_id,
            )

            if (
                autoencoder_lookup_key
                not in autoencoder_embedding_lookup
            ):
                raise KeyError(
                    "Missing autoencoder video embedding for "
                    f"{autoencoder_lookup_key}."
                )

            autoencoder_embedding = (
                autoencoder_embedding_lookup[
                    autoencoder_lookup_key
                ]
            )

            # Normalize components independently when configured.
            if HYBRID_NORMALIZE_CLIP_VIDEO:
                clip_component = l2_normalize_embedding(
                    clip_embedding,
                    (
                        "CLIP video embedding "
                        f"{sample_video_id}"
                    ),
                )
            else:
                clip_component = clip_embedding.astype(
                    np.float32
                )

            if HYBRID_NORMALIZE_AUTOENCODER_VIDEO:
                autoencoder_component = l2_normalize_embedding(
                    autoencoder_embedding,
                    (
                        "autoencoder video embedding "
                        f"{sample_video_id}"
                    ),
                )
            else:
                autoencoder_component = (
                    autoencoder_embedding.astype(
                        np.float32
                    )
                )

            # Concatenate semantic and reconstruction-based components.
            hybrid_embedding = np.concatenate(
                [
                    clip_component,
                    autoencoder_component,
                ],
                axis=0,
            ).astype(np.float32)

            if hybrid_embedding.shape != (
                HYBRID_VIDEO_EMBEDDING_DIM,
            ):
                raise ValueError(
                    "Invalid hybrid video embedding shape for "
                    f"{sample_video_id}: "
                    f"{hybrid_embedding.shape}"
                )

            if not np.isfinite(hybrid_embedding).all():
                raise ValueError(
                    "Non-finite hybrid video embedding values "
                    f"found for {sample_video_id}."
                )

            # Replace representation identifiers with the hybrid source.
            hybrid_representation_id = (
                f"{sample_video_id}_"
                f"{HYBRID_VIDEO_REPRESENTATION_SOURCE}"
            )

            sample["video_embedding"] = hybrid_embedding

            sample["metadata"][
                "video_representation_id"
            ] = hybrid_representation_id

            sample["metadata"][
                "video_representation_source"
            ] = HYBRID_VIDEO_REPRESENTATION_SOURCE

            for candidate_metadata in sample[
                "candidate_metadata"
            ]:
                candidate_metadata[
                    "video_representation_id"
                ] = hybrid_representation_id

                candidate_metadata[
                    "video_representation_source"
                ] = HYBRID_VIDEO_REPRESENTATION_SOURCE

            transformed_video_ids.add(
                sample_video_id
            )

        return transformed_video_ids

    # Transform the grouped training and validation samples in place.
    transformed_train_video_ids = (
        build_hybrid_qa_samples(
            qa_samples=train_qa_samples,
            expected_split_name="train",
        )
    )

    transformed_validation_video_ids = (
        build_hybrid_qa_samples(
            qa_samples=validation_qa_samples,
            expected_split_name=evaluation_split,
        )
    )

    # --------------------------------------------------------
    # Display Step 10 summary
    # --------------------------------------------------------

    print(
        "\nHybrid video representations constructed successfully."
    )
    print(
        "Training QA samples updated   : "
        f"{len(train_qa_samples):,}"
    )
    print(
        "Validation QA samples updated : "
        f"{len(validation_qa_samples):,}"
    )
    print(
        "Training unique videos        : "
        f"{len(transformed_train_video_ids):,}"
    )
    print(
        "Validation unique videos      : "
        f"{len(transformed_validation_video_ids):,}"
    )



### 🔷 Step 11 — Validate Hybrid Video Representations

* Verify that all grouped training and validation samples contain valid hybrid video representations.
* Update the shared embedding dimensionality for downstream prediction methods.
* Display summary information describing the completed hybrid representation dataset.


In [ ]:
# ============================================================
# Step 11: Validate Hybrid Video Representations
# ============================================================

print("Validating hybrid video representations...")

# ------------------------------------------------------------
# Preserve existing non-hybrid execution paths
# ------------------------------------------------------------

# Hybrid validation applies only to the hybrid experiment path.
if EXPERIMENT_TYPE != "hybrid_video":

    print(
        "Hybrid video representation validation skipped. "
        f"Experiment type: {EXPERIMENT_TYPE}"
    )

else:

    # --------------------------------------------------------
    # Verify required Step 11 inputs
    # --------------------------------------------------------

    # Confirm that Steps 9 and 10 completed successfully.
    required_step11_objects = [
        "train_qa_samples",
        "validation_qa_samples",
        "selected_train_clip_video_ids",
        "selected_validation_clip_video_ids",
        "transformed_train_video_ids",
        "transformed_validation_video_ids",
        "SOURCE_AUTOENCODER_EXPERIMENT_NAME",
        "autoencoder_video_representations_csv",
        "CLIP_VIDEO_EMBEDDING_DIM",
        "AUTOENCODER_VIDEO_EMBEDDING_DIM",
        "HYBRID_VIDEO_EMBEDDING_DIM",
        "HYBRID_VIDEO_REPRESENTATION_SOURCE",
        "HYBRID_NORMALIZE_CLIP_VIDEO",
        "HYBRID_NORMALIZE_AUTOENCODER_VIDEO",
        "HYBRID_VIDEO_COMBINATION_METHOD",
    ]

    missing_step11_objects = [
        name
        for name in required_step11_objects
        if name not in globals()
    ]

    if missing_step11_objects:
        raise NameError(
            "Missing required Step 11 objects: "
            + ", ".join(missing_step11_objects)
        )

    # --------------------------------------------------------
    # Validate transformed sample coverage
    # --------------------------------------------------------

    # Every selected video must appear in the transformed samples.
    if (
        transformed_train_video_ids
        != selected_train_clip_video_ids
    ):
        raise ValueError(
            "Hybrid training sample video coverage does not "
            "match the selected training video subset."
        )

    if (
        transformed_validation_video_ids
        != selected_validation_clip_video_ids
    ):
        raise ValueError(
            "Hybrid validation sample video coverage does not "
            "match the selected validation video subset."
        )

    # --------------------------------------------------------
    # Update shared downstream embedding dimension
    # --------------------------------------------------------

    # Preserve component dimensions for reporting and downstream models.
    clip_video_embedding_dimension = (
        CLIP_VIDEO_EMBEDDING_DIM
    )

    autoencoder_video_embedding_dimension = (
        AUTOENCODER_VIDEO_EMBEDDING_DIM
    )

    video_embedding_dimension = (
        HYBRID_VIDEO_EMBEDDING_DIM
    )

    # --------------------------------------------------------
    # Final strict sample-shape validation
    # --------------------------------------------------------

    # Confirm that every transformed sample has the hybrid dimension.
    invalid_train_hybrid_shapes = [
        sample["qa_record_id"]
        for sample in train_qa_samples
        if np.asarray(
            sample["video_embedding"]
        ).shape != (
            HYBRID_VIDEO_EMBEDDING_DIM,
        )
    ]

    invalid_validation_hybrid_shapes = [
        sample["qa_record_id"]
        for sample in validation_qa_samples
        if np.asarray(
            sample["video_embedding"]
        ).shape != (
            HYBRID_VIDEO_EMBEDDING_DIM,
        )
    ]

    if invalid_train_hybrid_shapes:
        raise ValueError(
            "Invalid hybrid training sample embedding shapes "
            "found for QA records: "
            + ", ".join(
                invalid_train_hybrid_shapes[:20]
            )
        )

    if invalid_validation_hybrid_shapes:
        raise ValueError(
            "Invalid hybrid validation sample embedding shapes "
            "found for QA records: "
            + ", ".join(
                invalid_validation_hybrid_shapes[:20]
            )
        )

    # --------------------------------------------------------
    # Validate hybrid representation metadata
    # --------------------------------------------------------

    # All grouped samples and candidate records must identify the
    # newly constructed hybrid video representation source.
    invalid_train_hybrid_sources = [
        sample["qa_record_id"]
        for sample in train_qa_samples
        if (
            sample["metadata"][
                "video_representation_source"
            ]
            != HYBRID_VIDEO_REPRESENTATION_SOURCE
            or any(
                candidate_metadata[
                    "video_representation_source"
                ]
                != HYBRID_VIDEO_REPRESENTATION_SOURCE
                for candidate_metadata in sample[
                    "candidate_metadata"
                ]
            )
        )
    ]

    invalid_validation_hybrid_sources = [
        sample["qa_record_id"]
        for sample in validation_qa_samples
        if (
            sample["metadata"][
                "video_representation_source"
            ]
            != HYBRID_VIDEO_REPRESENTATION_SOURCE
            or any(
                candidate_metadata[
                    "video_representation_source"
                ]
                != HYBRID_VIDEO_REPRESENTATION_SOURCE
                for candidate_metadata in sample[
                    "candidate_metadata"
                ]
            )
        )
    ]

    if invalid_train_hybrid_sources:
        raise ValueError(
            "Invalid hybrid training representation sources "
            "found for QA records: "
            + ", ".join(
                invalid_train_hybrid_sources[:20]
            )
        )

    if invalid_validation_hybrid_sources:
        raise ValueError(
            "Invalid hybrid validation representation sources "
            "found for QA records: "
            + ", ".join(
                invalid_validation_hybrid_sources[:20]
            )
        )

    # --------------------------------------------------------
    # Display summary
    # --------------------------------------------------------

    # Report source artifacts, coverage, dimensions, and normalization.
    print(
        "\nHybrid video representations validated successfully."
    )
    print(
        "Source autoencoder experiment : "
        f"{SOURCE_AUTOENCODER_EXPERIMENT_NAME}"
    )
    print(
        "Autoencoder source file       : "
        f"{autoencoder_video_representations_csv}"
    )
    print(
        "Training QA samples validated : "
        f"{len(train_qa_samples):,}"
    )
    print(
        "Validation QA samples validated: "
        f"{len(validation_qa_samples):,}"
    )
    print(
        "Training unique videos        : "
        f"{len(transformed_train_video_ids):,}"
    )
    print(
        "Validation unique videos      : "
        f"{len(transformed_validation_video_ids):,}"
    )
    print(
        "CLIP component dimension      : "
        f"{clip_video_embedding_dimension}"
    )
    print(
        "Autoencoder component dim     : "
        f"{autoencoder_video_embedding_dimension}"
    )
    print(
        "Hybrid video dimension        : "
        f"{video_embedding_dimension}"
    )
    print(
        "CLIP L2 normalization         : "
        f"{HYBRID_NORMALIZE_CLIP_VIDEO}"
    )
    print(
        "Autoencoder L2 normalization  : "
        f"{HYBRID_NORMALIZE_AUTOENCODER_VIDEO}"
    )
    print(
        "Combination method            : "
        f"{HYBRID_VIDEO_COMBINATION_METHOD}"
    )



### 🔷 Step 12 — Run Cosine Similarity Scoring

This prediction method measures the semantic similarity between a video representation and each of the five candidate question-answer representations. Because cosine similarity compares vectors within a common feature space, the video and text embeddings must have the same dimensionality.

For each question:

- $\mathbf{v}$ denotes the video representation.
- $\mathbf{t}_i$ denotes the representation of candidate answer $i$.

Both representations are L2-normalized before comparison:

$$
\hat{\mathbf{v}}=\frac{\mathbf{v}}{\|\mathbf{v}\|_2},
\qquad
\hat{\mathbf{t}}_i=\frac{\mathbf{t}_i}{\|\mathbf{t}_i\|_2}.
$$

The cosine similarity for candidate answer $i$ is

$$
s_i=
\frac{\mathbf{v}^{T}\mathbf{t}_i}
{\|\mathbf{v}\|_2\|\mathbf{t}_i\|_2}.
$$

where $\|\cdot\|_2$ denotes the Euclidean (L2) norm. Normalization ensures that the similarity score depends only on the orientation of the vectors within the shared semantic space, rather than their magnitude.

The five similarity scores form the score vector

$$
\mathbf{s}=[s_1,s_2,s_3,s_4,s_5].
$$

This step:

- Executes only when `REPRESENTATION_VIDEOQA_METHOD = "cosine_similarity"`.
- Computes one cosine-similarity score for each candidate answer.
- Produces the standardized score matrix consumed by the shared prediction-generation step.
- Performs no model training or parameter updates.

In [ ]:
# ============================================================
# Step 12: Run Cosine-Similarity Scoring
# ============================================================

import numpy as np
import pandas as pd

print("Evaluating cosine-similarity scoring step...")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

# Confirm that earlier preparation steps produced all required objects.
required_objects = [
    "REPRESENTATION_VIDEOQA_METHOD",
    "validation_qa_samples",
    "text_embedding_dimension",
    "video_embedding_dimension",
    "NUM_MULTIPLE_CHOICE_ANSWERS",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects for cosine-similarity scoring: "
        + ", ".join(missing_objects)
    )

# ------------------------------------------------------------
# Run cosine-similarity scoring when selected
# ------------------------------------------------------------

if REPRESENTATION_VIDEOQA_METHOD == "cosine_similarity":

    print("Running cosine-similarity scoring...")

    # --------------------------------------------------------
    # Validate compatible embedding dimensions
    # --------------------------------------------------------

    # Cosine similarity requires both representations to share a vector space.
    if video_embedding_dimension != text_embedding_dimension:
        raise ValueError(
            "cosine_similarity requires matching video and text "
            "embedding dimensions. "
            f"Video dim={video_embedding_dimension}, "
            f"text dim={text_embedding_dimension}."
        )

    if not validation_qa_samples:
        raise RuntimeError(
            "No validation QA samples are available for "
            "cosine-similarity scoring."
        )

    # --------------------------------------------------------
    # Define normalization helper
    # --------------------------------------------------------

    def normalize_embedding_matrix(embedding_matrix):
        """
        Normalize each embedding vector to unit length.
        """

        embedding_matrix = np.asarray(
            embedding_matrix,
            dtype=np.float32,
        )

        # Enforce the row-per-embedding structure used by matrix scoring.
        if embedding_matrix.ndim != 2:
            raise ValueError(
                "Expected a two-dimensional embedding matrix, "
                f"found shape {embedding_matrix.shape}."
            )

        if not np.isfinite(embedding_matrix).all():
            raise ValueError(
                "Non-finite embedding values encountered during "
                "cosine-similarity scoring."
            )

        norms = np.linalg.norm(
            embedding_matrix,
            axis=1,
            keepdims=True,
        )

        # Zero-length vectors cannot be converted to unit vectors.
        if np.any(norms == 0):
            raise ValueError(
                "Zero-norm embedding encountered during "
                "cosine-similarity scoring."
            )

        return embedding_matrix / norms

    # --------------------------------------------------------
    # Score the five answer choices for each validation record
    # --------------------------------------------------------

    cosine_score_rows = []

    for sample in validation_qa_samples:

        # Represent the video as one row and the choices as one row each.
        video_embedding = np.asarray(
            sample["video_embedding"],
            dtype=np.float32,
        ).reshape(1, -1)

        question_answer_embeddings = np.asarray(
            sample["question_answer_embeddings"],
            dtype=np.float32,
        )

        expected_video_shape = (
            1,
            video_embedding_dimension,
        )

        expected_text_shape = (
            NUM_MULTIPLE_CHOICE_ANSWERS,
            text_embedding_dimension,
        )

        if video_embedding.shape != expected_video_shape:
            raise ValueError(
                f"Invalid video embedding shape for "
                f"{sample['qa_record_id']}: "
                f"{video_embedding.shape}; "
                f"expected {expected_video_shape}."
            )

        if question_answer_embeddings.shape != expected_text_shape:
            raise ValueError(
                f"Invalid question-answer embedding shape for "
                f"{sample['qa_record_id']}: "
                f"{question_answer_embeddings.shape}; "
                f"expected {expected_text_shape}."
            )

        normalized_video_embedding = normalize_embedding_matrix(
            video_embedding
        )

        normalized_question_answer_embeddings = (
            normalize_embedding_matrix(
                question_answer_embeddings
            )
        )

        # The matrix product yields one cosine score per answer choice.
        cosine_scores = (
            normalized_question_answer_embeddings
            @ normalized_video_embedding.T
        ).reshape(-1)

        if cosine_scores.shape != (
            NUM_MULTIPLE_CHOICE_ANSWERS,
        ):
            raise ValueError(
                f"Invalid cosine score shape for "
                f"{sample['qa_record_id']}: "
                f"{cosine_scores.shape}."
            )

        if not np.isfinite(cosine_scores).all():
            raise ValueError(
                f"Non-finite cosine scores generated for "
                f"{sample['qa_record_id']}."
            )

        cosine_score_rows.append(cosine_scores)

    # --------------------------------------------------------
    # Populate shared method outputs
    # --------------------------------------------------------

    # Stack rows into the common score format used by later steps.
    all_scores = np.vstack(cosine_score_rows)

    # Provide placeholder training metadata for this non-learned method.
    fusion_training_history_df = pd.DataFrame(
        [
            {
                "epoch": 0,
                "loss": np.nan,
            }
        ]
    )

    training_epochs = 0
    final_training_loss = np.nan
    fusion_embedding_dimension = 0

    # --------------------------------------------------------
    # Strict output validation
    # --------------------------------------------------------

    expected_score_shape = (
        len(validation_qa_samples),
        NUM_MULTIPLE_CHOICE_ANSWERS,
    )

    if all_scores.shape != expected_score_shape:
        raise ValueError(
            f"Expected cosine score matrix shape "
            f"{expected_score_shape}, found {all_scores.shape}."
        )

    if not np.isfinite(all_scores).all():
        raise ValueError(
            "The cosine-similarity score matrix contains "
            "non-finite values."
        )

    # Allow minimal floating-point tolerance around the cosine bounds.
    if np.any(all_scores < -1.000001) or np.any(
        all_scores > 1.000001
    ):
        raise ValueError(
            "Cosine-similarity scores were generated outside "
            "the expected range [-1, 1]."
        )

    print("Cosine-similarity scoring completed successfully.")
    print(f"Validation QA records : {len(validation_qa_samples):,}")
    print(f"Choices per QA record : {NUM_MULTIPLE_CHOICE_ANSWERS}")
    print(f"Score matrix shape    : {all_scores.shape}")
    print(f"Minimum score         : {all_scores.min():.6f}")
    print(f"Maximum score         : {all_scores.max():.6f}")
    print("Training epochs       : not applicable")
    print("Training loss         : not applicable")

# ------------------------------------------------------------
# Skip when another prediction method is selected
# ------------------------------------------------------------

else:
    print(
        "Cosine-similarity scoring skipped. "
        f"Selected method: {REPRESENTATION_VIDEOQA_METHOD}"
    )



### 🔷 Step 13 — Run Fusion MLP Classifier

This prediction method learns to identify the correct answer by combining video and question-answer representations within a shared fusion space. Unlike cosine similarity, the Fusion MLP classifier learns its parameters from the training data using supervised learning.

For each candidate answer:

- $\mathbf{v}$ denotes the video representation.
- $\mathbf{t}_i$ denotes the representation of candidate answer $i$.

The video and text representations are first projected into a common fusion space:

$$
\mathbf{v}' = f_v(\mathbf{v}),
\qquad
\mathbf{t}'_i = f_t(\mathbf{t}_i),
$$

where $f_v(\cdot)$ and $f_t(\cdot)$ are learned projection layers.

The projected representations are concatenated to form a fused representation:

$$
\mathbf{x}_i=
\left[
\mathbf{v}' \,;\,
\mathbf{t}'_i
\right].
$$

The Fusion MLP assigns a score to each candidate answer:

$$
s_i=\mathrm{MLP}(\mathbf{x}_i).
$$

The five candidate scores form the score vector

$$
\mathbf{s}=[s_1,s_2,s_3,s_4,s_5].
$$

During training, the classifier learns its parameters by minimizing the cross-entropy loss between the predicted score vector and the ground-truth answer.

This step:

- Executes only when `REPRESENTATION_VIDEOQA_METHOD = "fusion_mlp_classifier"`.
- Uses the grouped training and validation samples created in Step 8.
- Projects the video and question-answer representations into a shared fusion space.
- Concatenates the projected representations for each candidate answer.
- Trains the Fusion MLP classifier using cross-entropy loss.
- Produces the standardized validation score matrix consumed by the shared prediction-generation step.

In [ ]:
# ============================================================
# Step 13: Run Fusion MLP Classifier
# ============================================================

from src.videoqa_fusion_training import train_and_score_fusion_classifier

print("Evaluating Fusion MLP classifier step...")

# Confirm that configuration values and prepared QA samples are available.
required_objects = [
    "REPRESENTATION_VIDEOQA_METHOD",
    "train_qa_samples",
    "validation_qa_samples",
    "text_embedding_dimension",
    "video_embedding_dimension",
    "NUM_MULTIPLE_CHOICE_ANSWERS",
    "FUSION_EMBEDDING_DIM",
    "FUSION_HIDDEN_DIM_1",
    "FUSION_HIDDEN_DIM_2",
    "FUSION_OUTPUT_DIM",
    "FUSION_DROPOUT",
    "FUSION_BATCH_SIZE",
    "FUSION_LEARNING_RATE",
    "FUSION_WEIGHT_DECAY",
    "FUSION_EPOCHS",
    "FUSION_RANDOM_SEED",
    "USE_LAYER_NORMALIZATION",
]

missing_objects = [name for name in required_objects if name not in globals()]
if missing_objects:
    raise NameError(
        "Missing required objects for Fusion MLP classification: "
        + ", ".join(missing_objects)
    )

# Build and train this model only when selected by the experiment configuration.
if REPRESENTATION_VIDEOQA_METHOD == "fusion_mlp_classifier":

    # Project video and text embeddings into a shared fusion dimension.
    class FusionMLPClassifier(nn.Module):

        def __init__(
            self,
            video_input_dim,
            text_input_dim,
            fusion_embedding_dim,
            hidden_dim_1,
            hidden_dim_2,
            output_dim,
            dropout,
            use_layer_norm=True,
        ):
            super().__init__()

            self.video_projection = nn.Linear(
                video_input_dim,
                fusion_embedding_dim,
            )

            self.text_projection = nn.Linear(
                text_input_dim,
                fusion_embedding_dim,
            )

            # Concatenation combines one projected video vector with each choice.
            fusion_input_dim = fusion_embedding_dim * 2

            layers = [
                nn.Linear(
                    fusion_input_dim,
                    hidden_dim_1,
                ),
                nn.ReLU(),
            ]

            # Layer normalization is optional to support controlled experiments.
            if use_layer_norm:
                layers.append(
                    nn.LayerNorm(hidden_dim_1)
                )

            layers.extend(
                [
                    nn.Dropout(dropout),
                    nn.Linear(
                        hidden_dim_1,
                        hidden_dim_2,
                    ),
                    nn.ReLU(),
                ]
            )

            if use_layer_norm:
                layers.append(
                    nn.LayerNorm(hidden_dim_2)
                )

            # Produce one scalar compatibility score for each answer choice.
            layers.extend(
                [
                    nn.Dropout(dropout),
                    nn.Linear(
                        hidden_dim_2,
                        output_dim,
                    ),
                ]
            )

            self.scorer = nn.Sequential(*layers)

        def forward(
            self,
            video,
            question_answers,
        ):

            # Question-answer embeddings retain a separate choice dimension.
            batch_size, num_choices, _ = (
                question_answers.shape
            )

            video_projected = self.video_projection(
                video
            )

            question_answers_projected = (
                self.text_projection(
                    question_answers
                )
            )

            # Repeat each video representation across its candidate answers.
            video_expanded = (
                video_projected
                .unsqueeze(1)
                .expand(
                    -1,
                    num_choices,
                    -1,
                )
            )

            # Concatenate paired video and candidate-text representations.
            fused = torch.cat(
                [
                    video_expanded,
                    question_answers_projected,
                ],
                dim=-1,
            )

            scores = (
                self.scorer(fused)
                .squeeze(-1)
            )

            return scores

    # Seed CPU and GPU generators for reproducible initialization and training.
    np.random.seed(FUSION_RANDOM_SEED)
    torch.manual_seed(FUSION_RANDOM_SEED)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(FUSION_RANDOM_SEED)

    # Instantiate the classifier using the shared experiment configuration.
    fusion_model = FusionMLPClassifier(
        video_input_dim=video_embedding_dimension,
        text_input_dim=text_embedding_dimension,
        fusion_embedding_dim=FUSION_EMBEDDING_DIM,
        hidden_dim_1=FUSION_HIDDEN_DIM_1,
        hidden_dim_2=FUSION_HIDDEN_DIM_2,
        output_dim=FUSION_OUTPUT_DIM,
        dropout=FUSION_DROPOUT,
        use_layer_norm=USE_LAYER_NORMALIZATION,
    )

    # Use the common training utility to preserve consistent evaluation behavior.
    fusion_result = train_and_score_fusion_classifier(
        model=fusion_model,
        train_samples=train_qa_samples,
        validation_samples=validation_qa_samples,
        video_embedding_dimension=video_embedding_dimension,
        text_embedding_dimension=text_embedding_dimension,
        num_choices=NUM_MULTIPLE_CHOICE_ANSWERS,
        batch_size=FUSION_BATCH_SIZE,
        learning_rate=FUSION_LEARNING_RATE,
        weight_decay=FUSION_WEIGHT_DECAY,
        epochs=FUSION_EPOCHS,
        random_seed=FUSION_RANDOM_SEED,
        method_label="Fusion MLP",
    )

    # Populate the shared outputs consumed by later prediction and reporting steps.
    fusion_model = fusion_result.model
    device = fusion_result.device
    all_scores = fusion_result.all_scores
    fusion_training_history_df = fusion_result.training_history_df
    training_epochs = fusion_result.training_epochs
    final_training_loss = fusion_result.final_training_loss
    fusion_embedding_dimension = FUSION_EMBEDDING_DIM

    print("Fusion MLP classifier completed successfully.")
    print(f"Device                    : {device}")
    print(f"Training QA records       : {len(train_qa_samples):,}")
    print(f"Validation QA records     : {len(validation_qa_samples):,}")
    print(f"Choices per QA record     : {NUM_MULTIPLE_CHOICE_ANSWERS}")
    print(f"Text embedding dimension  : {text_embedding_dimension}")
    print(f"Video embedding dimension : {video_embedding_dimension}")
    print(f"Fusion embedding dimension: {fusion_embedding_dimension}")
    print(f"Training epochs           : {training_epochs}")
    print(f"Final training loss       : {final_training_loss:.4f}")
    print(f"Score matrix shape        : {all_scores.shape}")

# Preserve method-specific execution within the shared notebook workflow.
else:
    print(
        "Fusion MLP classifier skipped. "
        f"Selected method: {REPRESENTATION_VIDEOQA_METHOD}"
    )



### 🔷 Step 14 — Run Interaction Fusion Classifier

This prediction method learns to identify the correct answer by combining video and question-answer representations with explicit interaction features. Unlike the Fusion MLP classifier, this model augments the projected representations with measures of their similarity and difference before classification.

For each candidate answer:

- $\mathbf{v}$ denotes the video representation.
- $\mathbf{t}_i$ denotes the representation of candidate answer $i$.

The video and text representations are first projected into a common fusion space:

$$
\mathbf{v}' = f_v(\mathbf{v}),
\qquad
\mathbf{t}'_i = f_t(\mathbf{t}_i),
$$

where $f_v(\cdot)$ and $f_t(\cdot)$ are learned projection layers.

The classifier constructs an interaction feature vector by concatenating the projected representations, their elementwise absolute difference, and their elementwise product:

$$
\mathbf{x}_i=
\left[
\mathbf{v}' \,;\,
\mathbf{t}'_i \,;\,
|\mathbf{v}'-\mathbf{t}'_i| \,;\,
\mathbf{v}'\odot\mathbf{t}'_i
\right].
$$

where $\odot$ denotes elementwise multiplication.

The Interaction Fusion classifier assigns a score to each candidate answer:

$$
s_i=\mathrm{MLP}(\mathbf{x}_i).
$$

The five candidate scores form the score vector

$$
\mathbf{s}=[s_1,s_2,s_3,s_4,s_5].
$$

During training, the classifier learns its parameters by minimizing the cross-entropy loss between the predicted score vector and the ground-truth answer.

This step:

- Executes only when `REPRESENTATION_VIDEOQA_METHOD = "interaction_fusion_classifier"`.
- Uses the grouped training and validation samples created in Step 8.
- Projects the video and question-answer representations into a shared fusion space.
- Constructs interaction features using the projected representations, their absolute differences, and their elementwise products.
- Trains the Interaction Fusion classifier using cross-entropy loss.
- Produces the standardized validation score matrix consumed by the shared prediction-generation step.

In [ ]:
# ============================================================
# Step 14: Run Interaction Fusion Classifier
# ============================================================

from src.videoqa_fusion_training import train_and_score_fusion_classifier

print("Evaluating Interaction Fusion classifier step...")

# Confirm that prepared samples and model configuration are available.
required_objects = [
    "REPRESENTATION_VIDEOQA_METHOD",
    "train_qa_samples",
    "validation_qa_samples",
    "text_embedding_dimension",
    "video_embedding_dimension",
    "NUM_MULTIPLE_CHOICE_ANSWERS",
    "FUSION_EMBEDDING_DIM",
    "FUSION_HIDDEN_DIM_1",
    "FUSION_HIDDEN_DIM_2",
    "FUSION_OUTPUT_DIM",
    "FUSION_DROPOUT",
    "FUSION_BATCH_SIZE",
    "FUSION_LEARNING_RATE",
    "FUSION_WEIGHT_DECAY",
    "FUSION_EPOCHS",
    "FUSION_RANDOM_SEED",
    "USE_LAYER_NORMALIZATION",
]

missing_objects = [name for name in required_objects if name not in globals()]
if missing_objects:
    raise NameError(
        "Missing required objects for Interaction Fusion classification: "
        + ", ".join(missing_objects)
    )

# Build and train this classifier only when selected.
if REPRESENTATION_VIDEOQA_METHOD == "interaction_fusion_classifier":

    # Combine projected embeddings with explicit comparison features.
    class InteractionFusionClassifier(nn.Module):

        def __init__(
            self,
            video_input_dim,
            text_input_dim,
            fusion_embedding_dim,
            hidden_dim_1,
            hidden_dim_2,
            output_dim,
            dropout,
            use_layer_norm=True,
        ):
            super().__init__()

            self.video_projection = nn.Linear(
                video_input_dim,
                fusion_embedding_dim,
            )

            self.text_projection = nn.Linear(
                text_input_dim,
                fusion_embedding_dim,
            )

            # Four feature groups are concatenated for each answer choice.
            interaction_input_dim = (
                fusion_embedding_dim * 4
            )

            layers = [
                nn.Linear(
                    interaction_input_dim,
                    hidden_dim_1,
                ),
                nn.ReLU(),
            ]

            if use_layer_norm:
                layers.append(
                    nn.LayerNorm(
                        hidden_dim_1
                    )
                )

            layers.extend(
                [
                    nn.Dropout(
                        dropout
                    ),
                    nn.Linear(
                        hidden_dim_1,
                        hidden_dim_2,
                    ),
                    nn.ReLU(),
                ]
            )

            if use_layer_norm:
                layers.append(
                    nn.LayerNorm(
                        hidden_dim_2
                    )
                )

            # Produce one compatibility score for each candidate answer.
            layers.extend(
                [
                    nn.Dropout(
                        dropout
                    ),
                    nn.Linear(
                        hidden_dim_2,
                        output_dim,
                    ),
                ]
            )

            self.scorer = nn.Sequential(
                *layers
            )

        def forward(
            self,
            video,
            question_answers,
        ):

            _, num_choices, _ = (
                question_answers.shape
            )

            video_projected = (
                self.video_projection(
                    video
                )
            )

            question_answers_projected = (
                self.text_projection(
                    question_answers
                )
            )

            # Align each video vector with every answer choice.
            video_expanded = (
                video_projected
                .unsqueeze(1)
                .expand(
                    -1,
                    num_choices,
                    -1,
                )
            )

            # Absolute difference captures distance between paired features.
            absolute_difference = torch.abs(
                video_expanded
                - question_answers_projected
            )

            # Elementwise products capture direct feature agreement.
            elementwise_product = (
                video_expanded
                * question_answers_projected
            )

            # Preserve original features alongside their interactions.
            interaction_features = torch.cat(
                [
                    video_expanded,
                    question_answers_projected,
                    absolute_difference,
                    elementwise_product,
                ],
                dim=-1,
            )

            scores = (
                self.scorer(
                    interaction_features
                )
                .squeeze(-1)
            )

            return scores

    # Seed CPU and GPU generators for reproducible training.
    np.random.seed(FUSION_RANDOM_SEED)
    torch.manual_seed(FUSION_RANDOM_SEED)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(FUSION_RANDOM_SEED)

    interaction_model = InteractionFusionClassifier(
        video_input_dim=video_embedding_dimension,
        text_input_dim=text_embedding_dimension,
        fusion_embedding_dim=FUSION_EMBEDDING_DIM,
        hidden_dim_1=FUSION_HIDDEN_DIM_1,
        hidden_dim_2=FUSION_HIDDEN_DIM_2,
        output_dim=FUSION_OUTPUT_DIM,
        dropout=FUSION_DROPOUT,
        use_layer_norm=USE_LAYER_NORMALIZATION,
    )

    # Use the shared utility for consistent training and validation scoring.
    fusion_result = train_and_score_fusion_classifier(
        model=interaction_model,
        train_samples=train_qa_samples,
        validation_samples=validation_qa_samples,
        video_embedding_dimension=video_embedding_dimension,
        text_embedding_dimension=text_embedding_dimension,
        num_choices=NUM_MULTIPLE_CHOICE_ANSWERS,
        batch_size=FUSION_BATCH_SIZE,
        learning_rate=FUSION_LEARNING_RATE,
        weight_decay=FUSION_WEIGHT_DECAY,
        epochs=FUSION_EPOCHS,
        random_seed=FUSION_RANDOM_SEED,
        method_label="Interaction Fusion",
    )

    # Populate the shared outputs consumed by later notebook steps.
    interaction_model = fusion_result.model
    device = fusion_result.device
    all_scores = fusion_result.all_scores
    fusion_training_history_df = fusion_result.training_history_df
    training_epochs = fusion_result.training_epochs
    final_training_loss = fusion_result.final_training_loss
    fusion_embedding_dimension = FUSION_EMBEDDING_DIM

    print("Interaction Fusion classifier completed successfully.")
    print(f"Device                    : {device}")
    print(f"Training QA records       : {len(train_qa_samples):,}")
    print(f"Validation QA records     : {len(validation_qa_samples):,}")
    print(f"Choices per QA record     : {NUM_MULTIPLE_CHOICE_ANSWERS}")
    print(f"Text embedding dimension  : {text_embedding_dimension}")
    print(f"Video embedding dimension : {video_embedding_dimension}")
    print(f"Fusion embedding dimension: {fusion_embedding_dimension}")
    print(f"Training epochs           : {training_epochs}")
    print(f"Final training loss       : {final_training_loss:.4f}")
    print(f"Score matrix shape        : {all_scores.shape}")

else:
    print(
        "Interaction Fusion classifier skipped. "
        f"Selected method: {REPRESENTATION_VIDEOQA_METHOD}"
    )



### 🔷 Step 15 — Run Gated Fusion Classifier

This prediction method learns to identify the correct answer by adaptively weighting the contributions of the video and question-answer representations. Rather than treating both modalities equally, the model learns a gating function that determines how much information to retain from each representation.

For each candidate answer:

- $\mathbf{v}$ denotes the video representation.
- $\mathbf{t}_i$ denotes the representation of candidate answer $i$.

The video and text representations are first projected into a common fusion space:

$$
\mathbf{v}' = f_v(\mathbf{v}),
\qquad
\mathbf{t}'_i = f_t(\mathbf{t}_i),
$$

where $f_v(\cdot)$ and $f_t(\cdot)$ are learned projection layers.

A gating function computes a feature-wise weighting vector:

$$
\mathbf{g}_i
=
\sigma
\left(
W
\left[
\mathbf{v}' ; \mathbf{t}'_i
\right]
+
\mathbf{b}
\right),
$$

where $\sigma(\cdot)$ is the sigmoid activation function, producing gate values between 0 and 1.

The gated multimodal representation is then

$$
\mathbf{x}_i
=
\mathbf{g}_i \odot \mathbf{v}'
+
(1-\mathbf{g}_i)\odot\mathbf{t}'_i,
$$

where $\odot$ denotes elementwise multiplication. A gate value of 1 retains the corresponding video feature, a gate value of 0 retains the corresponding text feature, and intermediate values blend the two modalities.

The Gated Fusion classifier assigns a score to each candidate answer:

$$
s_i=\mathrm{MLP}(\mathbf{x}_i).
$$

The five candidate scores form the score vector

$$
\mathbf{s}=[s_1,s_2,s_3,s_4,s_5].
$$

During training, the classifier learns its parameters by minimizing the cross-entropy loss between the predicted score vector and the ground-truth answer.

This step:

- Executes only when `REPRESENTATION_VIDEOQA_METHOD = "gated_fusion_classifier"`.
- Uses the grouped training and validation samples created in Step 8.
- Projects the video and question-answer representations into a shared fusion space.
- Learns a gating function that adaptively combines the projected video and text representations.
- Trains the Gated Fusion classifier using cross-entropy loss.
- Records the training loss for each epoch.
- Produces the standardized validation score matrix consumed by the shared prediction-generation step.

In [ ]:
# ============================================================
# Step 15: Run Gated Fusion Classifier
# ============================================================

from src.videoqa_fusion_training import train_and_score_fusion_classifier

print("Evaluating Gated Fusion classifier step...")

# Confirm that prepared samples and model configuration are available.
required_objects = [
    "REPRESENTATION_VIDEOQA_METHOD",
    "train_qa_samples",
    "validation_qa_samples",
    "text_embedding_dimension",
    "video_embedding_dimension",
    "NUM_MULTIPLE_CHOICE_ANSWERS",
    "FUSION_EMBEDDING_DIM",
    "FUSION_HIDDEN_DIM_1",
    "FUSION_HIDDEN_DIM_2",
    "FUSION_OUTPUT_DIM",
    "FUSION_DROPOUT",
    "FUSION_BATCH_SIZE",
    "FUSION_LEARNING_RATE",
    "FUSION_WEIGHT_DECAY",
    "FUSION_EPOCHS",
    "FUSION_RANDOM_SEED",
    "USE_LAYER_NORMALIZATION",
]

missing_objects = [name for name in required_objects if name not in globals()]
if missing_objects:
    raise NameError(
        "Missing required objects for Gated Fusion classification: "
        + ", ".join(missing_objects)
    )

# Build and train this classifier only when selected.
if REPRESENTATION_VIDEOQA_METHOD == "gated_fusion_classifier":

    # Learn how much video and text information to retain per feature.
    class GatedFusionClassifier(nn.Module):

        def __init__(
            self,
            video_input_dim,
            text_input_dim,
            fusion_embedding_dim,
            hidden_dim_1,
            hidden_dim_2,
            output_dim,
            dropout,
            use_layer_norm=True,
        ):
            super().__init__()

            # Reject invalid dimensions before constructing model layers.
            if video_input_dim <= 0:
                raise ValueError(
                    "video_input_dim must be greater than zero."
                )

            if text_input_dim <= 0:
                raise ValueError(
                    "text_input_dim must be greater than zero."
                )

            if fusion_embedding_dim <= 0:
                raise ValueError(
                    "fusion_embedding_dim must be greater than zero."
                )

            # Each answer choice requires one scalar compatibility score.
            if output_dim != 1:
                raise ValueError(
                    "GatedFusionClassifier requires output_dim=1."
                )

            self.fusion_embedding_dim = fusion_embedding_dim

            # Map both modalities into the same learned feature space.
            self.video_projection = nn.Linear(
                video_input_dim,
                fusion_embedding_dim,
            )

            self.text_projection = nn.Linear(
                text_input_dim,
                fusion_embedding_dim,
            )

            # Apply optional normalization without changing the forward path.
            if use_layer_norm:
                self.video_projection_normalization = nn.LayerNorm(
                    fusion_embedding_dim
                )

                self.text_projection_normalization = nn.LayerNorm(
                    fusion_embedding_dim
                )
            else:
                self.video_projection_normalization = nn.Identity()
                self.text_projection_normalization = nn.Identity()

            self.projection_activation = nn.ReLU()

            # Generate feature-wise weights between zero and one.
            self.gate_network = nn.Sequential(
                nn.Linear(
                    fusion_embedding_dim * 2,
                    fusion_embedding_dim,
                ),
                nn.Sigmoid(),
            )

            scoring_layers = [
                nn.Linear(
                    fusion_embedding_dim,
                    hidden_dim_1,
                ),
                nn.ReLU(),
            ]

            if use_layer_norm:
                scoring_layers.append(
                    nn.LayerNorm(hidden_dim_1)
                )

            scoring_layers.extend(
                [
                    nn.Dropout(dropout),
                    nn.Linear(
                        hidden_dim_1,
                        hidden_dim_2,
                    ),
                    nn.ReLU(),
                ]
            )

            if use_layer_norm:
                scoring_layers.append(
                    nn.LayerNorm(hidden_dim_2)
                )

            # Produce one score for each fused video-answer pair.
            scoring_layers.extend(
                [
                    nn.Dropout(dropout),
                    nn.Linear(
                        hidden_dim_2,
                        output_dim,
                    ),
                ]
            )

            self.scorer = nn.Sequential(*scoring_layers)

        def forward(self, video, question_answers):

            # Validate the batch structures expected by the fusion model.
            if video.ndim != 2:
                raise ValueError(
                    "Expected video tensor shape "
                    "[batch_size, video_embedding_dimension], "
                    f"found {tuple(video.shape)}."
                )

            if question_answers.ndim != 3:
                raise ValueError(
                    "Expected question_answers tensor shape "
                    "[batch_size, num_choices, text_embedding_dimension], "
                    f"found {tuple(question_answers.shape)}."
                )

            batch_size, num_choices, _ = question_answers.shape

            if video.shape[0] != batch_size:
                raise ValueError(
                    "Video and question-answer batch sizes do not match. "
                    f"Video batch={video.shape[0]}, "
                    f"question-answer batch={batch_size}."
                )

            if num_choices != NUM_MULTIPLE_CHOICE_ANSWERS:
                raise ValueError(
                    f"Expected {NUM_MULTIPLE_CHOICE_ANSWERS} candidate "
                    f"answers per sample, found {num_choices}."
                )

            # Project and normalize the video representation.
            video_projected = self.video_projection(video)

            video_projected = (
                self.video_projection_normalization(
                    video_projected
                )
            )

            video_projected = self.projection_activation(
                video_projected
            )

            # Apply the same transformation to all candidate texts.
            question_answers_projected = self.text_projection(
                question_answers
            )

            question_answers_projected = (
                self.text_projection_normalization(
                    question_answers_projected
                )
            )

            question_answers_projected = self.projection_activation(
                question_answers_projected
            )

            # Pair each video representation with every answer choice.
            video_expanded = video_projected.unsqueeze(1).expand(
                -1,
                num_choices,
                -1,
            )

            # Use both modalities to determine the fusion weights.
            gate_input = torch.cat(
                [
                    video_expanded,
                    question_answers_projected,
                ],
                dim=-1,
            )

            fusion_gate = self.gate_network(gate_input)

            # Blend text and video features using complementary gate weights.
            gated_fusion = (
                fusion_gate * question_answers_projected
                + (1.0 - fusion_gate) * video_expanded
            )

            scores = self.scorer(gated_fusion).squeeze(-1)

            expected_score_shape = (
                batch_size,
                NUM_MULTIPLE_CHOICE_ANSWERS,
            )

            # Ensure one score was generated for every candidate answer.
            if scores.shape != expected_score_shape:
                raise RuntimeError(
                    "Gated Fusion produced an unexpected score shape. "
                    f"Expected {expected_score_shape}, "
                    f"found {tuple(scores.shape)}."
                )

            return scores

    # Seed CPU and GPU generators for reproducible training.
    np.random.seed(FUSION_RANDOM_SEED)
    torch.manual_seed(FUSION_RANDOM_SEED)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(FUSION_RANDOM_SEED)

    fusion_model = GatedFusionClassifier(
        video_input_dim=video_embedding_dimension,
        text_input_dim=text_embedding_dimension,
        fusion_embedding_dim=FUSION_EMBEDDING_DIM,
        hidden_dim_1=FUSION_HIDDEN_DIM_1,
        hidden_dim_2=FUSION_HIDDEN_DIM_2,
        output_dim=FUSION_OUTPUT_DIM,
        dropout=FUSION_DROPOUT,
        use_layer_norm=USE_LAYER_NORMALIZATION,
    )

    # Use the shared utility for consistent training and validation scoring.
    fusion_result = train_and_score_fusion_classifier(
        model=fusion_model,
        train_samples=train_qa_samples,
        validation_samples=validation_qa_samples,
        video_embedding_dimension=video_embedding_dimension,
        text_embedding_dimension=text_embedding_dimension,
        num_choices=NUM_MULTIPLE_CHOICE_ANSWERS,
        batch_size=FUSION_BATCH_SIZE,
        learning_rate=FUSION_LEARNING_RATE,
        weight_decay=FUSION_WEIGHT_DECAY,
        epochs=FUSION_EPOCHS,
        random_seed=FUSION_RANDOM_SEED,
        method_label="Gated Fusion",
    )

    # Populate the shared outputs consumed by later notebook steps.
    fusion_model = fusion_result.model
    device = fusion_result.device
    all_scores = fusion_result.all_scores
    fusion_training_history_df = fusion_result.training_history_df
    training_epochs = fusion_result.training_epochs
    final_training_loss = fusion_result.final_training_loss
    fusion_embedding_dimension = FUSION_EMBEDDING_DIM

    print("Gated Fusion classifier completed successfully.")
    print(f"Device                    : {device}")
    print(f"Training QA records       : {len(train_qa_samples):,}")
    print(f"Validation QA records     : {len(validation_qa_samples):,}")
    print(f"Choices per QA record     : {NUM_MULTIPLE_CHOICE_ANSWERS}")
    print(f"Text embedding dimension  : {text_embedding_dimension}")
    print(f"Video embedding dimension : {video_embedding_dimension}")
    print(f"Fusion embedding dimension: {fusion_embedding_dimension}")
    print(f"Training epochs           : {training_epochs}")
    print(f"Final training loss       : {final_training_loss:.4f}")
    print(f"Score matrix shape        : {all_scores.shape}")

else:
    print(
        "Gated Fusion classifier skipped. "
        f"Selected method: {REPRESENTATION_VIDEOQA_METHOD}"
    )



### 🔷 Step 16 — Run Bilinear Fusion Classifier

This prediction method learns to identify the correct answer by modeling pairwise interactions between the projected video and question-answer representations. Unlike the previous fusion methods, bilinear fusion captures multiplicative relationships between the two modalities, enabling the classifier to learn richer multimodal feature representations.

For each candidate answer:

- $\mathbf{v}$ denotes the video representation.
- $\mathbf{t}_i$ denotes the representation of candidate answer $i$.

The video and text representations are first projected into a common fusion space:

$$
\mathbf{v}' = f_v(\mathbf{v}),
\qquad
\mathbf{t}'_i = f_t(\mathbf{t}_i),
$$

where $f_v(\cdot)$ and $f_t(\cdot)$ are learned projection layers.

A bilinear transformation computes a multimodal feature representation:

$$
\mathbf{x}_i
=
\mathbf{v}'^{T}
W
\mathbf{t}'_i,
$$

where $W$ is a learned bilinear weight matrix (or tensor) that models pairwise interactions between the projected video and text representations.

The Bilinear Fusion classifier assigns a score to each candidate answer:

$$
s_i=\mathrm{MLP}(\mathbf{x}_i).
$$

The five candidate scores form the score vector

$$
\mathbf{s}=[s_1,s_2,s_3,s_4,s_5].
$$

During training, the classifier learns its parameters by minimizing the cross-entropy loss between the predicted score vector and the ground-truth answer.

This step:

- Executes only when `REPRESENTATION_VIDEOQA_METHOD = "bilinear_fusion_classifier"`.
- Uses the grouped training and validation samples created in Step 8.
- Projects the video and question-answer representations into a shared fusion space.
- Learns bilinear interactions between the projected video and text representations.
- Trains the Bilinear Fusion classifier using cross-entropy loss.
- Records the training loss for each epoch.
- Produces the standardized validation score matrix consumed by the shared prediction-generation step.

In [ ]:
# ============================================================
# Step 16: Run Bilinear Fusion Classifier
# ============================================================

from src.videoqa_fusion_training import train_and_score_fusion_classifier

print("Evaluating Bilinear Fusion classifier step...")

# Confirm that prepared samples and model configuration are available.
required_objects = [
    "REPRESENTATION_VIDEOQA_METHOD",
    "train_qa_samples",
    "validation_qa_samples",
    "text_embedding_dimension",
    "video_embedding_dimension",
    "NUM_MULTIPLE_CHOICE_ANSWERS",
    "FUSION_EMBEDDING_DIM",
    "FUSION_HIDDEN_DIM_1",
    "FUSION_HIDDEN_DIM_2",
    "FUSION_OUTPUT_DIM",
    "FUSION_DROPOUT",
    "FUSION_BATCH_SIZE",
    "FUSION_LEARNING_RATE",
    "FUSION_WEIGHT_DECAY",
    "FUSION_EPOCHS",
    "FUSION_RANDOM_SEED",
    "USE_LAYER_NORMALIZATION",
]

missing_objects = [name for name in required_objects if name not in globals()]
if missing_objects:
    raise NameError(
        "Missing required objects for Bilinear Fusion classification: "
        + ", ".join(missing_objects)
    )

# Build and train this classifier only when selected.
if REPRESENTATION_VIDEOQA_METHOD == "bilinear_fusion_classifier":

    # Learn a shared representation where video-text interactions are modeled
    # through elementwise products in the projected embedding space.
    class BilinearFusionClassifier(nn.Module):

        def __init__(
            self,
            video_input_dim,
            text_input_dim,
            fusion_embedding_dim,
            hidden_dim_1,
            hidden_dim_2,
            output_dim,
            dropout,
            use_layer_norm=True,
        ):
            super().__init__()

            # Validate model dimensions before constructing network layers.
            if video_input_dim <= 0:
                raise ValueError(
                    "video_input_dim must be greater than zero."
                )

            if text_input_dim <= 0:
                raise ValueError(
                    "text_input_dim must be greater than zero."
                )

            if fusion_embedding_dim <= 0:
                raise ValueError(
                    "fusion_embedding_dim must be greater than zero."
                )

            if hidden_dim_1 <= 0 or hidden_dim_2 <= 0:
                raise ValueError(
                    "Bilinear Fusion hidden dimensions must be greater "
                    "than zero."
                )

            # Each answer choice produces one compatibility score.
            if output_dim != 1:
                raise ValueError(
                    "BilinearFusionClassifier requires output_dim=1."
                )

            self.video_input_dim = video_input_dim
            self.text_input_dim = text_input_dim
            self.fusion_embedding_dim = fusion_embedding_dim

            # Project both modalities into a common fusion space.
            self.video_projection = nn.Linear(
                video_input_dim,
                fusion_embedding_dim,
            )

            self.text_projection = nn.Linear(
                text_input_dim,
                fusion_embedding_dim,
            )

            # Optional normalization is applied consistently throughout
            # the network while preserving the same forward path.
            if use_layer_norm:
                self.video_projection_normalization = nn.LayerNorm(
                    fusion_embedding_dim
                )

                self.text_projection_normalization = nn.LayerNorm(
                    fusion_embedding_dim
                )

                self.bilinear_normalization = nn.LayerNorm(
                    fusion_embedding_dim
                )
            else:
                self.video_projection_normalization = nn.Identity()
                self.text_projection_normalization = nn.Identity()
                self.bilinear_normalization = nn.Identity()

            self.projection_activation = nn.Tanh()

            scoring_layers = [
                nn.Linear(
                    fusion_embedding_dim,
                    hidden_dim_1,
                ),
                nn.ReLU(),
            ]

            if use_layer_norm:
                scoring_layers.append(
                    nn.LayerNorm(hidden_dim_1)
                )

            scoring_layers.extend(
                [
                    nn.Dropout(dropout),
                    nn.Linear(
                        hidden_dim_1,
                        hidden_dim_2,
                    ),
                    nn.ReLU(),
                ]
            )

            if use_layer_norm:
                scoring_layers.append(
                    nn.LayerNorm(hidden_dim_2)
                )

            # Convert each fused representation into a scalar score.
            scoring_layers.extend(
                [
                    nn.Dropout(dropout),
                    nn.Linear(
                        hidden_dim_2,
                        output_dim,
                    ),
                ]
            )

            self.scorer = nn.Sequential(*scoring_layers)

        def forward(self, video, question_answers):

            # Validate tensor shapes before performing fusion.
            if video.ndim != 2:
                raise ValueError(
                    "Expected video tensor shape "
                    "[batch_size, video_embedding_dimension], "
                    f"found {tuple(video.shape)}."
                )

            if question_answers.ndim != 3:
                raise ValueError(
                    "Expected question_answers tensor shape "
                    "[batch_size, num_choices, "
                    "text_embedding_dimension], "
                    f"found {tuple(question_answers.shape)}."
                )

            batch_size, num_choices, text_dimension = (
                question_answers.shape
            )

            if video.shape[0] != batch_size:
                raise ValueError(
                    "Video and question-answer batch sizes do not "
                    "match. "
                    f"Video batch={video.shape[0]}, "
                    f"question-answer batch={batch_size}."
                )

            if video.shape[1] != self.video_input_dim:
                raise ValueError(
                    "Video input dimension mismatch. Expected "
                    f"{self.video_input_dim}, found {video.shape[1]}."
                )

            if text_dimension != self.text_input_dim:
                raise ValueError(
                    "Question-answer input dimension mismatch. Expected "
                    f"{self.text_input_dim}, found {text_dimension}."
                )

            if num_choices != NUM_MULTIPLE_CHOICE_ANSWERS:
                raise ValueError(
                    f"Expected {NUM_MULTIPLE_CHOICE_ANSWERS} candidate "
                    f"answers per sample, found {num_choices}."
                )

            # Project the video embedding into the shared fusion space.
            video_projected = self.video_projection(video)

            video_projected = (
                self.video_projection_normalization(
                    video_projected
                )
            )

            video_projected = self.projection_activation(
                video_projected
            )

            # Apply the same projection to every candidate answer.
            question_answers_projected = self.text_projection(
                question_answers
            )

            question_answers_projected = (
                self.text_projection_normalization(
                    question_answers_projected
                )
            )

            question_answers_projected = self.projection_activation(
                question_answers_projected
            )

            # Pair each video representation with all answer choices.
            video_expanded = video_projected.unsqueeze(1).expand(
                -1,
                num_choices,
                -1,
            )

            # Factorized bilinear interaction:
            # projected video and text features are multiplied
            # elementwise in the shared fusion space.
            bilinear_fusion = (
                video_expanded
                * question_answers_projected
            )

            # Normalize the interaction features before scoring.
            bilinear_fusion = self.bilinear_normalization(
                bilinear_fusion
            )

            scores = self.scorer(
                bilinear_fusion
            ).squeeze(-1)

            expected_score_shape = (
                batch_size,
                NUM_MULTIPLE_CHOICE_ANSWERS,
            )

            # Verify that one score was produced for every answer choice.
            if tuple(scores.shape) != expected_score_shape:
                raise RuntimeError(
                    "Bilinear Fusion produced an unexpected score "
                    "shape. "
                    f"Expected {expected_score_shape}, "
                    f"found {tuple(scores.shape)}."
                )

            return scores

    # Seed CPU and GPU generators for reproducible training.
    np.random.seed(FUSION_RANDOM_SEED)
    torch.manual_seed(FUSION_RANDOM_SEED)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(FUSION_RANDOM_SEED)

    # Instantiate the classifier using the shared experiment configuration.
    fusion_model = BilinearFusionClassifier(
        video_input_dim=video_embedding_dimension,
        text_input_dim=text_embedding_dimension,
        fusion_embedding_dim=FUSION_EMBEDDING_DIM,
        hidden_dim_1=FUSION_HIDDEN_DIM_1,
        hidden_dim_2=FUSION_HIDDEN_DIM_2,
        output_dim=FUSION_OUTPUT_DIM,
        dropout=FUSION_DROPOUT,
        use_layer_norm=USE_LAYER_NORMALIZATION,
    )

    # Use the shared training utility to ensure consistent training,
    # validation, and reporting across all learned fusion methods.
    fusion_result = train_and_score_fusion_classifier(
        model=fusion_model,
        train_samples=train_qa_samples,
        validation_samples=validation_qa_samples,
        video_embedding_dimension=video_embedding_dimension,
        text_embedding_dimension=text_embedding_dimension,
        num_choices=NUM_MULTIPLE_CHOICE_ANSWERS,
        batch_size=FUSION_BATCH_SIZE,
        learning_rate=FUSION_LEARNING_RATE,
        weight_decay=FUSION_WEIGHT_DECAY,
        epochs=FUSION_EPOCHS,
        random_seed=FUSION_RANDOM_SEED,
        method_label="Bilinear Fusion",
    )

    # Populate the shared outputs consumed by later notebook steps.
    fusion_model = fusion_result.model
    device = fusion_result.device
    all_scores = fusion_result.all_scores
    fusion_training_history_df = fusion_result.training_history_df
    training_epochs = fusion_result.training_epochs
    final_training_loss = fusion_result.final_training_loss
    fusion_embedding_dimension = FUSION_EMBEDDING_DIM

    print("Bilinear Fusion classifier completed successfully.")
    print(f"Device                    : {device}")
    print(f"Training QA records       : {len(train_qa_samples):,}")
    print(f"Validation QA records     : {len(validation_qa_samples):,}")
    print(f"Choices per QA record     : {NUM_MULTIPLE_CHOICE_ANSWERS}")
    print(f"Text embedding dimension  : {text_embedding_dimension}")
    print(f"Video embedding dimension : {video_embedding_dimension}")
    print(f"Fusion embedding dimension: {fusion_embedding_dimension}")
    print(f"Training epochs           : {training_epochs}")
    print(f"Final training loss       : {final_training_loss:.4f}")
    print(f"Score matrix shape        : {all_scores.shape}")

else:
    print(
        "Bilinear Fusion classifier skipped. "
        f"Selected method: {REPRESENTATION_VIDEOQA_METHOD}"
    )



### 🔷 Step 17 — Build and Validate Prediction Datasets

* Build the standardized prediction dataset from the validation scores generated by the selected prediction method.
* Verify that the prediction dataset contains the expected number of question-answer prediction records.
* Confirm that one prediction exists for every evaluation question and that all candidate-answer records were processed.
* Validate required prediction fields, duplicate identifiers, dataset consistency, and missing values.
* Verify that all ground-truth and predicted answer indices are valid multiple-choice labels.
* Confirm that prediction correctness values are valid Boolean indicators.
* Display a validation summary confirming the integrity of the representation-based prediction dataset.

In [ ]:
# ============================================================
# Step 17: Build and Validate Prediction Datasets
# ============================================================

import numpy as np
import pandas as pd

print("Building and validating representation-based prediction datasets...")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

# Confirm that the selected prediction method produced all outputs
# required to build the final prediction datasets.
required_step14_objects = [
    "all_scores",
    "validation_qa_samples",
    "validation_qa_input_df",
    "choice_columns",
    "NUM_MULTIPLE_CHOICE_ANSWERS",
    "REPRESENTATION_VIDEOQA_METHOD",
    "VIDEO_REPRESENTATION_SOURCE",
    "TEXT_REPRESENTATION_SOURCE",
    "text_embedding_dimension",
    "video_embedding_dimension",
    "fusion_embedding_dimension",
    "training_epochs",
    "final_training_loss",
    "video_id_column",
    "question_column",
    "evaluation_split",
]

missing_step14_objects = [
    name
    for name in required_step14_objects
    if name not in globals()
]

if missing_step14_objects:
    raise NameError(
        "Missing required Step 14 objects: "
        + ", ".join(missing_step14_objects)
    )

# ------------------------------------------------------------
# Validate selected-method outputs
# ------------------------------------------------------------

# Verify that the prediction model generated one valid score
# for every answer choice in every validation QA record.
if all_scores is None:
    raise RuntimeError(
        "No prediction scores were generated. "
        "Verify that the selected prediction-method step ran successfully."
    )

all_scores = np.asarray(
    all_scores,
    dtype=np.float32,
)

expected_score_shape = (
    len(validation_qa_samples),
    NUM_MULTIPLE_CHOICE_ANSWERS,
)

if all_scores.shape != expected_score_shape:
    raise ValueError(
        f"Expected score matrix shape {expected_score_shape}, "
        f"found {all_scores.shape}."
    )

if not np.isfinite(all_scores).all():
    raise ValueError(
        "The prediction score matrix contains non-finite values."
    )

if len(choice_columns) != NUM_MULTIPLE_CHOICE_ANSWERS:
    raise ValueError(
        f"Expected {NUM_MULTIPLE_CHOICE_ANSWERS} answer choices, "
        f"found {len(choice_columns)}."
    )

# ------------------------------------------------------------
# Build scored candidate and QA-level prediction records
# ------------------------------------------------------------

# Create two complementary datasets:
#   • Candidate-level scores for all answer choices.
#   • QA-level predictions containing only the selected answer.
scored_candidate_records = []
prediction_rows = []

for sample_index, sample in enumerate(validation_qa_samples):

    scores = all_scores[sample_index]

    # Select the highest-scoring answer as the model prediction.
    predicted_choice = int(
        np.argmax(scores)
    )

    ground_truth_choice = int(
        sample["label"]
    )

    first_row = sample["metadata"]

    if predicted_choice not in range(
        NUM_MULTIPLE_CHOICE_ANSWERS
    ):
        raise ValueError(
            f"Invalid predicted choice for "
            f"{sample['qa_record_id']}: "
            f"{predicted_choice}"
        )

    if ground_truth_choice not in range(
        NUM_MULTIPLE_CHOICE_ANSWERS
    ):
        raise ValueError(
            f"Invalid ground-truth choice for "
            f"{sample['qa_record_id']}: "
            f"{ground_truth_choice}"
        )

    candidate_metadata = sample[
        "candidate_metadata"
    ]

    if len(candidate_metadata) != (
        NUM_MULTIPLE_CHOICE_ANSWERS
    ):
        raise ValueError(
            f"QA record {sample['qa_record_id']} has "
            f"{len(candidate_metadata)} candidate metadata rows; "
            f"expected {NUM_MULTIPLE_CHOICE_ANSWERS}."
        )

    # --------------------------------------------------------
    # Candidate-level scored records
    # --------------------------------------------------------

    # Preserve the prediction score for every answer choice to support
    # later analysis beyond the final selected prediction.
    for candidate_row in candidate_metadata:

        choice_index = int(
            candidate_row["choice_index"]
        )

        if choice_index not in range(
            NUM_MULTIPLE_CHOICE_ANSWERS
        ):
            raise ValueError(
                f"Invalid choice index for "
                f"{sample['qa_record_id']}: "
                f"{choice_index}"
            )

        scored_record = dict(
            candidate_row
        )

        scored_record[
            "prediction_score"
        ] = float(scores[choice_index])

        scored_record[
            "prediction_method"
        ] = REPRESENTATION_VIDEOQA_METHOD

        scored_record[
            "text_embedding_dimension"
        ] = text_embedding_dimension

        scored_record[
            "video_embedding_dimension"
        ] = video_embedding_dimension

        scored_record[
            "fusion_embedding_dimension"
        ] = fusion_embedding_dimension

        scored_record[
            "training_epochs"
        ] = training_epochs

        scored_candidate_records.append(
            scored_record
        )

    # --------------------------------------------------------
    # QA-level prediction record
    # --------------------------------------------------------

    # Store one summarized prediction record per validation question.
    prediction_rows.append(
        {
            "qa_record_id": sample["qa_record_id"],
            "annotation_id": first_row["annotation_id"],
            "split": first_row["split"],
            "video": first_row[video_id_column],
            "question": first_row[question_column],
            "ground_truth_choice": ground_truth_choice,
            "predicted_choice": predicted_choice,
            "choice_correct": (
                predicted_choice
                == ground_truth_choice
            ),
            "ground_truth": first_row[
                f"a{ground_truth_choice}"
            ],
            "prediction": first_row[
                f"a{predicted_choice}"
            ],
            "prediction_score": float(
                scores[predicted_choice]
            ),
            "prediction_method":
                REPRESENTATION_VIDEOQA_METHOD,
            "video_representation_source":
                VIDEO_REPRESENTATION_SOURCE,
            "text_representation_source":
                TEXT_REPRESENTATION_SOURCE,
            "text_embedding_dimension":
                text_embedding_dimension,
            "video_embedding_dimension":
                video_embedding_dimension,
            "fusion_embedding_dimension":
                fusion_embedding_dimension,
            "training_epochs":
                training_epochs,
            "final_training_loss":
                final_training_loss,
        }
    )

scored_candidate_df = pd.DataFrame(
    scored_candidate_records
)

representation_prediction_df = pd.DataFrame(
    prediction_rows
)

# ------------------------------------------------------------
# Validate required prediction columns
# ------------------------------------------------------------

# Confirm that all downstream reporting fields were generated.
required_prediction_columns = [
    "qa_record_id",
    "annotation_id",
    "split",
    "video",
    "question",
    "ground_truth_choice",
    "predicted_choice",
    "choice_correct",
    "ground_truth",
    "prediction",
    "prediction_score",
    "prediction_method",
    "video_representation_source",
    "text_representation_source",
    "text_embedding_dimension",
    "video_embedding_dimension",
    "fusion_embedding_dimension",
    "training_epochs",
    "final_training_loss",
]

required_non_null_prediction_columns = [
    column
    for column in required_prediction_columns
    if column != "final_training_loss"
]

missing_prediction_columns = [
    column
    for column in required_prediction_columns
    if column not in representation_prediction_df.columns
]

if missing_prediction_columns:
    raise ValueError(
        "representation_prediction_df is missing required columns: "
        + ", ".join(missing_prediction_columns)
    )

# ------------------------------------------------------------
# Compute validation checks
# ------------------------------------------------------------

# Compute structural and data-quality checks before saving results.
expected_prediction_records = len(
    validation_qa_input_df
)

expected_candidate_rows = (
    expected_prediction_records
    * NUM_MULTIPLE_CHOICE_ANSWERS
)

validation_checks = {
    "prediction_records":
        len(representation_prediction_df),

    "expected_prediction_records":
        expected_prediction_records,

    "candidate_rows":
        len(scored_candidate_df),

    "expected_candidate_rows":
        expected_candidate_rows,

    "unique_qa_records":
        representation_prediction_df[
            "qa_record_id"
        ].nunique(),

    "unique_videos":
        representation_prediction_df[
            "video"
        ].nunique(),

    "missing_values":
        int(
            representation_prediction_df[
                required_non_null_prediction_columns
            ]
            .isna()
            .sum()
            .sum()
        ),

    "duplicate_qa_record_ids":
        int(
            representation_prediction_df[
                "qa_record_id"
            ]
            .duplicated()
            .sum()
        ),

    "invalid_ground_truth_choices":
        int(
            (
                ~representation_prediction_df[
                    "ground_truth_choice"
                ].isin(
                    range(
                        NUM_MULTIPLE_CHOICE_ANSWERS
                    )
                )
            ).sum()
        ),

    "invalid_predicted_choices":
        int(
            (
                ~representation_prediction_df[
                    "predicted_choice"
                ].isin(
                    range(
                        NUM_MULTIPLE_CHOICE_ANSWERS
                    )
                )
            ).sum()
        ),

    "non_boolean_choice_correct":
        int(
            (
                ~representation_prediction_df[
                    "choice_correct"
                ].map(
                    lambda value: isinstance(
                        value,
                        (bool, np.bool_),
                    )
                )
            ).sum()
        ),

    "invalid_prediction_method":
        int(
            (
                representation_prediction_df[
                    "prediction_method"
                ]
                != REPRESENTATION_VIDEOQA_METHOD
            ).sum()
        ),

    "invalid_video_representation_source":
        int(
            (
                representation_prediction_df[
                    "video_representation_source"
                ]
                != VIDEO_REPRESENTATION_SOURCE
            ).sum()
        ),

    "invalid_text_representation_source":
        int(
            (
                representation_prediction_df[
                    "text_representation_source"
                ]
                != TEXT_REPRESENTATION_SOURCE
            ).sum()
        ),

    "invalid_prediction_split":
        int(
            (
                representation_prediction_df[
                    "split"
                ].astype(str)
                != str(evaluation_split)
            ).sum()
        ),

    "non_finite_prediction_scores":
        int(
            (
                ~np.isfinite(
                    representation_prediction_df[
                        "prediction_score"
                    ].astype(float)
                )
            ).sum()
        ),

    "non_finite_candidate_scores":
        int(
            (
                ~np.isfinite(
                    scored_candidate_df[
                        "prediction_score"
                    ].astype(float)
                )
            ).sum()
        ),
}

validation_summary_df = pd.DataFrame(
    validation_checks.items(),
    columns=[
        "Validation Check",
        "Value",
    ],
)

# ------------------------------------------------------------
# Fail on validation errors
# ------------------------------------------------------------

# Stop immediately if any required structural or integrity check fails.
if (
    validation_checks["prediction_records"]
    != validation_checks[
        "expected_prediction_records"
    ]
):
    raise ValueError(
        "Prediction record count does not match "
        "the expected validation QA input count."
    )

if (
    validation_checks["candidate_rows"]
    != validation_checks[
        "expected_candidate_rows"
    ]
):
    raise ValueError(
        "Candidate row count does not match "
        "the expected validation QA-choice count."
    )

if (
    validation_checks["unique_qa_records"]
    != validation_checks[
        "prediction_records"
    ]
):
    raise ValueError(
        "Prediction dataset does not contain "
        "exactly one row per QA record."
    )

zero_required_checks = [
    "missing_values",
    "duplicate_qa_record_ids",
    "invalid_ground_truth_choices",
    "invalid_predicted_choices",
    "non_boolean_choice_correct",
    "invalid_prediction_method",
    "invalid_video_representation_source",
    "invalid_text_representation_source",
    "invalid_prediction_split",
    "non_finite_prediction_scores",
    "non_finite_candidate_scores",
]

failed_checks = {
    check_name: validation_checks[
        check_name
    ]
    for check_name in zero_required_checks
    if validation_checks[check_name] > 0
}

if failed_checks:
    raise ValueError(
        "Prediction dataset validation failed: "
        + ", ".join(
            f"{name}={value}"
            for name, value
            in failed_checks.items()
        )
    )

# ------------------------------------------------------------
# Compute prediction accuracy
# ------------------------------------------------------------

# Compute overall multiple-choice accuracy for the evaluation split.
representation_choice_accuracy = float(
    representation_prediction_df[
        "choice_correct"
    ].mean()
)

# ------------------------------------------------------------
# Display summary
# ------------------------------------------------------------

# Summarize dataset integrity and provide a preview of predictions.
print(
    "Representation-based prediction datasets "
    "built and validated successfully."
)

print(
    f"Prediction method       : "
    f"{REPRESENTATION_VIDEOQA_METHOD}"
)

print(
    f"Prediction records      : "
    f"{len(representation_prediction_df):,}"
)

print(
    f"Candidate rows scored   : "
    f"{len(scored_candidate_df):,}"
)

print(
    f"Correct predictions     : "
    f"{representation_prediction_df['choice_correct'].sum():,}"
)

print(
    f"Validation accuracy     : "
    f"{representation_choice_accuracy:.3f}"
)

display(validation_summary_df)

print("\nValidation Prediction Preview:")

display(
    representation_prediction_df[
        [
            "qa_record_id",
            "video",
            "question",
            "ground_truth_choice",
            "predicted_choice",
            "choice_correct",
            "ground_truth",
            "prediction",
            "prediction_score",
        ]
    ].head(10)
)



### 🔷 Step 18 — Save Representation-Based VideoQA Results

* Verify that the generated outputs are consistent with the selected prediction method.
* Confirm that learned prediction methods contain valid training epochs, loss values, and training-history records.
* Confirm that the cosine similarity method contains the expected non-training placeholders.
* Validate prediction-method, representation-source, and experiment metadata.
* Create the configured local output directory.
* Build the experiment summary dataset.
* Save the prediction, validation, and summary datasets as CSV files locally.
* Reload the saved local datasets and verify their record counts, metadata, and file integrity.
* If Google Drive writes are enabled, create the configured Google Drive experiment directory, copy the prediction, validation, and summary artifacts, and verify the promoted files.
* Otherwise, retain the verified artifacts in local storage.
* Display the local output locations and, when applicable, the Google Drive output directory.



In [ ]:
# ============================================================
# Step 18: Save Representation-Based VideoQA Results
# ============================================================

import shutil
from pathlib import Path

import numpy as np
import pandas as pd

print("Saving representation-based VideoQA results...")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

# Confirm that all datasets, metadata, and output locations
# required for artifact generation are available.
required_objects = [
    "representation_prediction_df",
    "validation_summary_df",
    "representation_choice_accuracy",
    "text_embedding_dimension",
    "video_embedding_dimension",
    "fusion_embedding_dimension",
    "training_epochs",
    "final_training_loss",
    "fusion_training_history_df",
    "REPRESENTATION_VIDEOQA_METHOD",
    "EXPERIMENT_NAME",
    "EXPERIMENT_TYPE",
    "VIDEO_REPRESENTATION_SOURCE",
    "TEXT_REPRESENTATION_SOURCE",
    "REPRESENTATION_VIDEOQA_LOCAL_DIR",
    "REPRESENTATION_VIDEOQA_DRIVE_DIR",
    "REPRESENTATION_VIDEOQA_PREDICTIONS_CSV",
    "REPRESENTATION_VIDEOQA_VALIDATION_CSV",
    "REPRESENTATION_VIDEOQA_SUMMARY_CSV",
    "REPRESENTATION_VIDEOQA_PREDICTIONS_DRIVE_CSV",
    "REPRESENTATION_VIDEOQA_VALIDATION_DRIVE_CSV",
    "REPRESENTATION_VIDEOQA_SUMMARY_DRIVE_CSV",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects for saving results: "
        + ", ".join(missing_objects)
    )

# ------------------------------------------------------------
# Define supported prediction-method categories
# ------------------------------------------------------------

# Separate methods that require model training from those that
# perform direct similarity scoring.
non_training_prediction_methods = {
    "cosine_similarity",
}

learned_prediction_methods = {
    "fusion_mlp_classifier",
    "interaction_fusion_classifier",
    "gated_fusion_classifier",
    "bilinear_fusion_classifier",
}

supported_prediction_methods = (
    non_training_prediction_methods
    | learned_prediction_methods
)

if REPRESENTATION_VIDEOQA_METHOD not in supported_prediction_methods:
    raise ValueError(
        "Unsupported prediction method: "
        f"{REPRESENTATION_VIDEOQA_METHOD}"
    )

# ------------------------------------------------------------
# Validate result datasets
# ------------------------------------------------------------

# Verify that all generated datasets are complete before saving.
if representation_prediction_df.empty:
    raise ValueError(
        "representation_prediction_df is empty."
    )

if validation_summary_df.empty:
    raise ValueError(
        "validation_summary_df is empty."
    )

if fusion_training_history_df is None:
    raise ValueError(
        "fusion_training_history_df is None."
    )

if not isinstance(
    fusion_training_history_df,
    pd.DataFrame,
):
    raise TypeError(
        "fusion_training_history_df must be a pandas DataFrame."
    )

if fusion_training_history_df.empty:
    raise ValueError(
        "fusion_training_history_df is empty."
    )

required_training_history_columns = [
    "epoch",
    "loss",
]

missing_training_history_columns = [
    column
    for column in required_training_history_columns
    if column not in fusion_training_history_df.columns
]

if missing_training_history_columns:
    raise ValueError(
        "fusion_training_history_df is missing required columns: "
        + ", ".join(missing_training_history_columns)
    )

if not np.isfinite(
    float(representation_choice_accuracy)
):
    raise ValueError(
        "representation_choice_accuracy must be finite."
    )

if not (
    0.0
    <= float(representation_choice_accuracy)
    <= 1.0
):
    raise ValueError(
        "representation_choice_accuracy must be between 0 and 1."
    )

# ------------------------------------------------------------
# Validate method-specific training metadata
# ------------------------------------------------------------

# Enforce metadata appropriate for the selected prediction method.
if (
    REPRESENTATION_VIDEOQA_METHOD
    in learned_prediction_methods
):

    if int(training_epochs) <= 0:
        raise ValueError(
            f"{REPRESENTATION_VIDEOQA_METHOD} requires "
            "training_epochs greater than zero."
        )

    if pd.isna(final_training_loss):
        raise ValueError(
            f"{REPRESENTATION_VIDEOQA_METHOD} requires "
            "a valid final_training_loss."
        )

    if not np.isfinite(
        float(final_training_loss)
    ):
        raise ValueError(
            f"{REPRESENTATION_VIDEOQA_METHOD} produced "
            "a non-finite final_training_loss."
        )

    if len(fusion_training_history_df) != int(
        training_epochs
    ):
        raise ValueError(
            "Training-history row count does not match "
            f"training_epochs. History rows="
            f"{len(fusion_training_history_df)}, "
            f"training_epochs={training_epochs}."
        )

    learned_losses = (
        fusion_training_history_df["loss"]
        .astype(float)
        .to_numpy()
    )

    if not np.isfinite(learned_losses).all():
        raise ValueError(
            "Learned fusion training history contains "
            "non-finite loss values."
        )

elif (
    REPRESENTATION_VIDEOQA_METHOD
    in non_training_prediction_methods
):

    # Cosine similarity records placeholder training metadata to
    # preserve a common artifact format across all methods.
    if int(training_epochs) != 0:
        raise ValueError(
            f"{REPRESENTATION_VIDEOQA_METHOD} should have "
            "training_epochs equal to zero."
        )

    if not pd.isna(final_training_loss):
        raise ValueError(
            f"{REPRESENTATION_VIDEOQA_METHOD} should not "
            "have a final training loss."
        )

    if len(fusion_training_history_df) != 1:
        raise ValueError(
            f"{REPRESENTATION_VIDEOQA_METHOD} should have "
            "one placeholder training-history row."
        )

    placeholder_epoch = int(
        fusion_training_history_df[
            "epoch"
        ].iloc[0]
    )

    placeholder_loss = (
        fusion_training_history_df[
            "loss"
        ].iloc[0]
    )

    if placeholder_epoch != 0:
        raise ValueError(
            f"{REPRESENTATION_VIDEOQA_METHOD} should use "
            "placeholder epoch 0."
        )

    if not pd.isna(placeholder_loss):
        raise ValueError(
            f"{REPRESENTATION_VIDEOQA_METHOD} should use "
            "a missing placeholder loss."
        )

# ------------------------------------------------------------
# Validate prediction metadata consistency
# ------------------------------------------------------------

# Confirm that saved prediction metadata matches the active experiment.
prediction_methods = set(
    representation_prediction_df[
        "prediction_method"
    ]
    .astype(str)
    .unique()
)

if prediction_methods != {
    REPRESENTATION_VIDEOQA_METHOD
}:
    raise ValueError(
        "Prediction dataset contains unexpected methods: "
        f"{sorted(prediction_methods)}"
    )

video_sources = set(
    representation_prediction_df[
        "video_representation_source"
    ]
    .astype(str)
    .unique()
)

if video_sources != {
    VIDEO_REPRESENTATION_SOURCE
}:
    raise ValueError(
        "Prediction dataset contains unexpected video "
        f"representation sources: {sorted(video_sources)}"
    )

text_sources = set(
    representation_prediction_df[
        "text_representation_source"
    ]
    .astype(str)
    .unique()
)

if text_sources != {
    TEXT_REPRESENTATION_SOURCE
}:
    raise ValueError(
        "Prediction dataset contains unexpected text "
        f"representation sources: {sorted(text_sources)}"
    )

# ------------------------------------------------------------
# Build summary dataset
# ------------------------------------------------------------

# Create a one-row experiment summary for downstream comparison
# across multiple VideoQA experiments.
representation_summary_df = pd.DataFrame(
    [
        {
            "pipeline": "representation_videoqa",
            "experiment_name": EXPERIMENT_NAME,
            "experiment_type": EXPERIMENT_TYPE,
            "prediction_method":
                REPRESENTATION_VIDEOQA_METHOD,
            "video_representation_source":
                VIDEO_REPRESENTATION_SOURCE,
            "text_representation_source":
                TEXT_REPRESENTATION_SOURCE,
            "evaluation_samples":
                len(representation_prediction_df),
            "unique_questions":
                representation_prediction_df[
                    "qa_record_id"
                ].nunique(),
            "unique_videos":
                representation_prediction_df[
                    "video"
                ].nunique(),
            "valid_choice_predictions":
                len(representation_prediction_df),
            "correct_choice_predictions":
                int(
                    representation_prediction_df[
                        "choice_correct"
                    ].sum()
                ),
            "choice_accuracy":
                float(
                    representation_choice_accuracy
                ),
            "text_embedding_dimension":
                int(text_embedding_dimension),
            "video_embedding_dimension":
                int(video_embedding_dimension),
            "fusion_embedding_dimension":
                int(fusion_embedding_dimension),
            "training_epochs":
                int(training_epochs),
            "final_training_loss":
                (
                    None
                    if pd.isna(final_training_loss)
                    else float(final_training_loss)
                ),
        }
    ]
)

# ------------------------------------------------------------
# Validate summary dataset
# ------------------------------------------------------------

# Verify that the summary contains the expected schema.
required_summary_columns = [
    "pipeline",
    "experiment_name",
    "experiment_type",
    "prediction_method",
    "video_representation_source",
    "text_representation_source",
    "evaluation_samples",
    "unique_questions",
    "unique_videos",
    "valid_choice_predictions",
    "correct_choice_predictions",
    "choice_accuracy",
    "text_embedding_dimension",
    "video_embedding_dimension",
    "fusion_embedding_dimension",
    "training_epochs",
    "final_training_loss",
]

missing_summary_columns = [
    column
    for column in required_summary_columns
    if column not in representation_summary_df.columns
]

if missing_summary_columns:
    raise ValueError(
        "representation_summary_df is missing required columns: "
        + ", ".join(missing_summary_columns)
    )

if len(representation_summary_df) != 1:
    raise ValueError(
        "representation_summary_df must contain exactly "
        "one summary row."
    )

# ------------------------------------------------------------
# Normalize output paths
# ------------------------------------------------------------

# Convert configured output locations into Path objects for
# platform-independent file operations.
local_output_directory = Path(
    REPRESENTATION_VIDEOQA_LOCAL_DIR
)

drive_output_directory = Path(
    REPRESENTATION_VIDEOQA_DRIVE_DIR
)

local_prediction_path = Path(
    REPRESENTATION_VIDEOQA_PREDICTIONS_CSV
)

local_validation_path = Path(
    REPRESENTATION_VIDEOQA_VALIDATION_CSV
)

local_summary_path = Path(
    REPRESENTATION_VIDEOQA_SUMMARY_CSV
)

drive_prediction_path = Path(
    REPRESENTATION_VIDEOQA_PREDICTIONS_DRIVE_CSV
)

drive_validation_path = Path(
    REPRESENTATION_VIDEOQA_VALIDATION_DRIVE_CSV
)

drive_summary_path = Path(
    REPRESENTATION_VIDEOQA_SUMMARY_DRIVE_CSV
)

# ------------------------------------------------------------
# Create local output directory
# ------------------------------------------------------------

local_output_directory.mkdir(
    parents=True,
    exist_ok=True,
)

if not local_output_directory.exists():
    raise FileNotFoundError(
        "Output directory was not created: "
        f"{local_output_directory}"
    )

# ------------------------------------------------------------
# Save local artifacts
# ------------------------------------------------------------

# Save prediction, validation, and summary artifacts locally.
representation_prediction_df.to_csv(
    local_prediction_path,
    index=False,
)

validation_summary_df.to_csv(
    local_validation_path,
    index=False,
)

representation_summary_df.to_csv(
    local_summary_path,
    index=False,
)

local_artifact_paths = [
    local_prediction_path,
    local_validation_path,
    local_summary_path,
]

# Verify that each artifact was written successfully.
for local_path in local_artifact_paths:

    if not local_path.exists():
        raise FileNotFoundError(
            f"Failed to create local artifact: {local_path}"
        )

    if local_path.stat().st_size <= 0:
        raise ValueError(
            f"Local artifact is empty: {local_path}"
        )

# ------------------------------------------------------------
# Verify saved local files
# ------------------------------------------------------------

# Reload each file to verify that the saved contents match the
# in-memory datasets.
saved_predictions_check_df = pd.read_csv(
    local_prediction_path
)

if len(saved_predictions_check_df) != len(
    representation_prediction_df
):
    raise ValueError(
        "Saved prediction file row count does not match "
        "the in-memory prediction dataset."
    )

saved_validation_check_df = pd.read_csv(
    local_validation_path
)

if len(saved_validation_check_df) != len(
    validation_summary_df
):
    raise ValueError(
        "Saved validation file row count does not match "
        "the in-memory validation summary."
    )

saved_summary_check_df = pd.read_csv(
    local_summary_path
)

if len(saved_summary_check_df) != len(
    representation_summary_df
):
    raise ValueError(
        "Saved summary file row count does not match "
        "the in-memory summary dataset."
    )

missing_saved_summary_columns = [
    column
    for column in required_summary_columns
    if column not in saved_summary_check_df.columns
]

if missing_saved_summary_columns:
    raise ValueError(
        "Saved summary file is missing required columns: "
        + ", ".join(missing_saved_summary_columns)
    )

saved_method = str(
    saved_summary_check_df[
        "prediction_method"
    ].iloc[0]
)

if saved_method != REPRESENTATION_VIDEOQA_METHOD:
    raise ValueError(
        "Saved summary prediction method does not match "
        "the selected method."
    )

# ------------------------------------------------------------
# Promote artifacts to Google Drive
# ------------------------------------------------------------

if ENABLE_GOOGLE_DRIVE_WRITES:

    # Copy verified local artifacts into the persistent Google Drive
    # experiment directory.
    drive_output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    if not drive_output_directory.exists():
        raise FileNotFoundError(
            "Google Drive output directory was not created: "
            f"{drive_output_directory}"
        )

    artifact_pairs = [
        (
            local_prediction_path,
            drive_prediction_path,
        ),
        (
            local_validation_path,
            drive_validation_path,
        ),
        (
            local_summary_path,
            drive_summary_path,
        ),
    ]

    for local_path, drive_path in artifact_pairs:

        drive_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        shutil.copy2(
            local_path,
            drive_path,
        )

        # Verify that the promoted artifact matches the local copy.
        if not drive_path.exists():
            raise FileNotFoundError(
                f"Failed to promote artifact: {drive_path}"
            )

        if drive_path.stat().st_size != local_path.stat().st_size:
            raise ValueError(
                "Promoted artifact size does not match the "
                f"local artifact: {drive_path}"
            )

    print("Artifacts promoted to Google Drive.")

else:

    print("Google Drive writes are disabled.")
    print("Artifacts remain in local storage.")

# ------------------------------------------------------------
# Display save summary
# ------------------------------------------------------------

print(
    "Representation-based VideoQA results saved successfully."
)

print(
    f"Local output directory : "
    f"{local_output_directory}"
)

if ENABLE_GOOGLE_DRIVE_WRITES:
    print(
        f"Drive output directory : "
        f"{drive_output_directory}"
    )

print(
    f"Prediction method      : "
    f"{REPRESENTATION_VIDEOQA_METHOD}"
)

print(
    f"Prediction file        : "
    f"{local_prediction_path}"
)

print(
    f"Validation file        : "
    f"{local_validation_path}"
)

print(
    f"Summary file           : "
    f"{local_summary_path}"
)

print(
    f"Prediction records     : "
    f"{len(representation_prediction_df):,}"
)

print(
    f"Validation checks      : "
    f"{len(validation_summary_df):,}"
)

print(
    f"Summary records        : "
    f"{len(representation_summary_df):,}"
)



### 🔷 Step 19 — Generate Prediction Summary Report

* Compute summary statistics describing the completed representation-based VideoQA experiment.
* Record the dataset mode, evaluation split, answer mode, selected prediction method, and representation sources.
* Report the text, video, and fusion representation dimensions.
* Record the training epochs and final training loss for learned prediction methods.
* Record the expected non-training placeholder values for the cosine similarity method.
* Summarize question-answer record counts, candidate-answer records, unique videos, correct predictions, incorrect predictions, and validation accuracy.
* Save and verify the completed experiment summary locally.
* If Google Drive writes are enabled, copy the summary report to the configured experiment directory and verify successful promotion.
* Otherwise, retain the verified summary report in local storage.


In [ ]:
# ============================================================
# Step 19: Generate Prediction Summary Report
# ============================================================

import pandas as pd
import shutil
from pathlib import Path

print("Generating representation-based VideoQA summary report...")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

# Confirm that prediction datasets, experiment metadata, and
# configured artifact paths are available.
required_step11_objects = [
    "representation_prediction_df",
    "scored_candidate_df",
    "validation_summary_df",
    "RUN_FULL_EVALUATION_SPLIT",
    "evaluation_split",
    "answer_mode",
    "choice_columns",
    "NUM_MULTIPLE_CHOICE_ANSWERS",
    "REPRESENTATION_VIDEOQA_METHOD",
    "VIDEO_REPRESENTATION_SOURCE",
    "TEXT_REPRESENTATION_SOURCE",
    "REPRESENTATION_VIDEOQA_LOCAL_DIR",
    "REPRESENTATION_VIDEOQA_DRIVE_DIR",
    "REPRESENTATION_VIDEOQA_PREDICTIONS_CSV",
    "REPRESENTATION_VIDEOQA_VALIDATION_CSV",
    "REPRESENTATION_VIDEOQA_SUMMARY_CSV",
    "REPRESENTATION_VIDEOQA_SUMMARY_DRIVE_CSV",
]

missing_step11_objects = [
    name for name in required_step11_objects
    if name not in globals()
]

if missing_step11_objects:
    raise NameError(
        "Missing required Step 11 objects: "
        + ", ".join(missing_step11_objects)
    )

# ------------------------------------------------------------
# Compute summary metrics
# ------------------------------------------------------------

# Compute overall prediction counts and multiple-choice accuracy.
correct_predictions = int(
    representation_prediction_df["choice_correct"].sum()
)

prediction_records = len(representation_prediction_df)

choice_accuracy = (
    correct_predictions / prediction_records
    if prediction_records > 0
    else 0.0
)

# Label whether the experiment used the complete evaluation split
# or a development subset.
dataset_mode_label = (
    "full_split"
    if RUN_FULL_EVALUATION_SPLIT
    else "development"
)

# Recover embedding and training metadata from the prediction output.
text_embedding_dimension = int(
    representation_prediction_df["text_embedding_dimension"].iloc[0]
)

video_embedding_dimension = int(
    representation_prediction_df["video_embedding_dimension"].iloc[0]
)

fusion_embedding_dimension = int(
    representation_prediction_df["fusion_embedding_dimension"].iloc[0]
)

training_epochs = (
    int(representation_prediction_df["training_epochs"].iloc[0])
    if "training_epochs" in representation_prediction_df.columns
    else None
)

final_training_loss = (
    representation_prediction_df["final_training_loss"].iloc[0]
    if "final_training_loss" in representation_prediction_df.columns
    else None
)

# Store experiment configuration, performance, dimensions, and
# artifact locations in a compact metric-value report.
summary_rows = [
    {"metric": "dataset_mode", "value": dataset_mode_label},
    {"metric": "evaluation_split", "value": evaluation_split},
    {"metric": "answer_mode", "value": answer_mode},
    {"metric": "prediction_method", "value": REPRESENTATION_VIDEOQA_METHOD},
    {"metric": "video_representation_source", "value": VIDEO_REPRESENTATION_SOURCE},
    {"metric": "text_representation_source", "value": TEXT_REPRESENTATION_SOURCE},
    {"metric": "qa_records", "value": prediction_records},
    {"metric": "candidate_rows", "value": len(scored_candidate_df)},
    {"metric": "choice_count", "value": NUM_MULTIPLE_CHOICE_ANSWERS},
    {"metric": "correct_predictions", "value": correct_predictions},
    {"metric": "choice_accuracy", "value": choice_accuracy},
    {"metric": "text_embedding_dimension", "value": text_embedding_dimension},
    {"metric": "video_embedding_dimension", "value": video_embedding_dimension},
    {"metric": "fusion_embedding_dimension", "value": fusion_embedding_dimension},
    {"metric": "training_epochs", "value": training_epochs},
    {"metric": "final_training_loss", "value": final_training_loss},
    {
        "metric": "unique_videos",
        "value": representation_prediction_df["video"].nunique(),
    },
    {
        "metric": "prediction_file",
        "value": str(REPRESENTATION_VIDEOQA_PREDICTIONS_CSV),
    },
    {
        "metric": "validation_file",
        "value": str(REPRESENTATION_VIDEOQA_VALIDATION_CSV),
    },
    {
        "metric": "summary_file",
        "value": str(REPRESENTATION_VIDEOQA_SUMMARY_CSV),
    },
]

representation_videoqa_summary_df = pd.DataFrame(summary_rows)

# ------------------------------------------------------------
# Save summary report locally
# ------------------------------------------------------------

# Create the local output directory and save the summary report.
REPRESENTATION_VIDEOQA_LOCAL_DIR.mkdir(parents=True, exist_ok=True)

representation_videoqa_summary_df.to_csv(
    REPRESENTATION_VIDEOQA_SUMMARY_CSV,
    index=False,
)

if not Path(REPRESENTATION_VIDEOQA_SUMMARY_CSV).exists():
    raise FileNotFoundError(
        f"Failed to create summary file: {REPRESENTATION_VIDEOQA_SUMMARY_CSV}"
    )

# ------------------------------------------------------------
# Verify saved local summary can be reloaded
# ------------------------------------------------------------

# Reload the CSV to verify that all summary rows were persisted.
saved_summary_check_df = pd.read_csv(REPRESENTATION_VIDEOQA_SUMMARY_CSV)

if len(saved_summary_check_df) != len(representation_videoqa_summary_df):
    raise ValueError(
        "Saved summary file row count does not match "
        "the in-memory summary report."
    )

# ------------------------------------------------------------
# Promote summary report to Google Drive
# ------------------------------------------------------------

if ENABLE_GOOGLE_DRIVE_WRITES:

    # Copy the verified local report into persistent Google Drive storage.
    REPRESENTATION_VIDEOQA_DRIVE_DIR.mkdir(parents=True, exist_ok=True)

    shutil.copy2(
        REPRESENTATION_VIDEOQA_SUMMARY_CSV,
        REPRESENTATION_VIDEOQA_SUMMARY_DRIVE_CSV,
    )

    if not Path(REPRESENTATION_VIDEOQA_SUMMARY_DRIVE_CSV).exists():
        raise FileNotFoundError(
            f"Failed to promote summary file: "
            f"{REPRESENTATION_VIDEOQA_SUMMARY_DRIVE_CSV}"
        )

# ------------------------------------------------------------
# Display summary report
# ------------------------------------------------------------

# Report key experiment results and display the complete metric table.
print("Representation-based VideoQA summary report saved.")
print(f"Dataset mode       : {dataset_mode_label}")
print(f"Evaluation split   : {evaluation_split}")
print(f"Prediction method  : {REPRESENTATION_VIDEOQA_METHOD}")
print(f"Choice accuracy    : {choice_accuracy:.3f}")
print(f"Training epochs    : {training_epochs}")
print(f"Local summary file : {REPRESENTATION_VIDEOQA_SUMMARY_CSV}")

if ENABLE_GOOGLE_DRIVE_WRITES:
    print(f"Drive summary file : {REPRESENTATION_VIDEOQA_SUMMARY_DRIVE_CSV}")
else:
    print("Google Drive writes are disabled.")

display(representation_videoqa_summary_df)



### 🔷 Step 20 — Display Sample Predictions

* Randomly select representative prediction records from the completed representation-based VideoQA experiment.
* Display the evaluation question, candidate answers, ground-truth answer, predicted answer, and prediction scores.
* Identify the prediction method and representation sources used to generate each prediction.
* Support qualitative assessment of representation-based multiple-choice prediction performance.
* Provide representative examples for experiment interpretation, verification, and comparison across prediction methods.

In [ ]:
# ============================================================
# Step 20: Display Sample Predictions
# ============================================================

import pandas as pd

print("Displaying sample representation-based VideoQA predictions...")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

# Confirm that prediction data and experiment metadata are available.
required_step12_objects = [
    "representation_prediction_df",
    "random_seed",
    "dataset_mode_label",
    "evaluation_split",
    "REPRESENTATION_VIDEOQA_METHOD",
    "VIDEO_REPRESENTATION_SOURCE",
    "TEXT_REPRESENTATION_SOURCE",
]

missing_step12_objects = [
    name for name in required_step12_objects
    if name not in globals()
]

if missing_step12_objects:
    raise NameError(
        "Missing required Step 12 objects: "
        + ", ".join(missing_step12_objects)
    )

if representation_prediction_df.empty:
    raise RuntimeError(
        "representation_prediction_df is empty."
    )

# ------------------------------------------------------------
# Select sample predictions
# ------------------------------------------------------------

# Select up to ten reproducible examples for qualitative review.
sample_count = min(
    10,
    len(representation_prediction_df),
)

sample_prediction_df = (
    representation_prediction_df
    .sample(
        n=sample_count,
        random_state=random_seed,
    )
    .sort_values(
        [
            "choice_correct",
            "video",
            "question",
        ],
        ascending=[
            True,
            True,
            True,
        ],
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Validate display columns
# ------------------------------------------------------------

# Verify that the sample contains all fields needed to interpret
# predictions and representation configuration.
display_columns = [
    "qa_record_id",
    "video",
    "question",
    "ground_truth_choice",
    "predicted_choice",
    "choice_correct",
    "ground_truth",
    "prediction",
    "prediction_score",
    "prediction_method",
    "video_representation_source",
    "text_representation_source",
    "text_embedding_dimension",
    "video_embedding_dimension",
    "fusion_embedding_dimension",
]

missing_display_columns = [
    col for col in display_columns
    if col not in sample_prediction_df.columns
]

if missing_display_columns:
    raise ValueError(
        "Sample prediction display is missing required columns: "
        + ", ".join(missing_display_columns)
    )

# ------------------------------------------------------------
# Display sample predictions
# ------------------------------------------------------------

# Report the active experiment configuration and sample-level accuracy.
print(f"Displaying {sample_count} sample prediction records...")
print(f"Dataset mode                 : {dataset_mode_label}")
print(f"Evaluation split             : {evaluation_split}")
print(f"Prediction method            : {REPRESENTATION_VIDEOQA_METHOD}")
print(f"Video representation source  : {VIDEO_REPRESENTATION_SOURCE}")
print(f"Text representation source   : {TEXT_REPRESENTATION_SOURCE}")
print(f"Text embedding dimension     : {sample_prediction_df['text_embedding_dimension'].iloc[0]}")
print(f"Video embedding dimension    : {sample_prediction_df['video_embedding_dimension'].iloc[0]}")
print(f"Fusion embedding dimension   : {sample_prediction_df['fusion_embedding_dimension'].iloc[0]}")
print(
    f"Sample accuracy              : "
    f"{sample_prediction_df['choice_correct'].mean():.3f}"
)

display(
    sample_prediction_df[
        display_columns
    ]
)



### 🔷 Step 21 — Notebook Summary

* Summarize the completed representation-based VideoQA experiment.
* Report the dataset mode, training and validation splits, answer mode, selected prediction method, and representation sources.
* Display the training and validation question-answer record counts, unique-video counts, and validation candidate-answer record count.
* Report the text, video, and fusion representation dimensions.
* Display prediction-method-specific training information or cosine-similarity statistics, as applicable.
* Report the validation prediction count, correct predictions, incorrect predictions, and validation accuracy.
* Display the locations of the generated prediction, validation, and summary artifacts.
* Confirm that all required experiment artifacts have been successfully generated and are ready for downstream evaluation in Notebook 08.





In [ ]:
# ============================================================
# Step 21: Notebook Summary
# ============================================================

print("Notebook 07 complete.")
print("=" * 60)

# Summarize the active dataset, representation, and prediction settings.
print("\nRepresentation-Based VideoQA — Configuration")
print("-" * 60)
print(f"Dataset mode                 : {dataset_mode_label}")
print(f"Training split               : train")
print(f"Validation split             : {evaluation_split}")
print(f"Answer mode                  : {answer_mode}")
print(f"Prediction method            : {REPRESENTATION_VIDEOQA_METHOD}")
print(f"Video representation source  : {VIDEO_REPRESENTATION_SOURCE}")
print(f"Text representation source   : {TEXT_REPRESENTATION_SOURCE}")

# Report the QA, candidate, and video counts used by the experiment.
print("\nInput Dataset Summary")
print("-" * 60)
print(f"Training QA records          : {len(train_qa_input_df):,}")
print(f"Validation QA records        : {len(validation_qa_input_df):,}")
print(f"Validation candidate rows    : {len(scored_candidate_df):,}")
print(f"Training unique videos       : {train_qa_input_df[video_id_column].nunique():,}")
print(f"Validation unique videos     : {representation_prediction_df['video'].nunique():,}")
print(f"Answer choices per question  : {NUM_MULTIPLE_CHOICE_ANSWERS}")

# Summarize embedding dimensions and loaded representation records.
print("\nRepresentation Summary")
print("-" * 60)
print(f"Text embedding dimensions    : {text_embedding_dimension:,}")
print(f"Video embedding dimensions   : {video_embedding_dimension:,}")
print(f"Fusion embedding dimensions  : {fusion_embedding_dimension:,}")
print(f"Training text records        : {len(train_clip_text_df):,}")
print(f"Validation text records      : {len(validation_clip_text_df):,}")
print(f"Training video records       : {len(train_video_representation_df):,}")
print(f"Validation video records     : {len(validation_video_representation_df):,}")

# ------------------------------------------------------------
# Method-specific summary
# ------------------------------------------------------------

# Report metadata appropriate for the selected scoring method.
if REPRESENTATION_VIDEOQA_METHOD == "cosine_similarity":

    print("\nCosine Similarity Summary")
    print("-" * 60)
    print("Training epochs              : 0")
    print("Final training loss          : not applicable")
    print(
        "Scoring method               : "
        "cosine(video_embedding, answer_choice_embedding)"
    )
    print("Inference split              : validation only")

elif REPRESENTATION_VIDEOQA_METHOD in {
    "fusion_mlp_classifier",
    "interaction_fusion_classifier",
    "gated_fusion_classifier",
    "bilinear_fusion_classifier",
}:

    method_label = REPRESENTATION_VIDEOQA_METHOD_LABELS[
        REPRESENTATION_VIDEOQA_METHOD
    ]

    print(f"\n{method_label} Summary")
    print("-" * 60)
    print(f"Training epochs              : {training_epochs}")
    print(f"Final training loss          : {final_training_loss:.4f}")
    print("Inference split              : validation only")

else:
    raise ValueError(
        f"Unsupported prediction method: "
        f"{REPRESENTATION_VIDEOQA_METHOD}"
    )

# Report final validation performance.
print("\nPrediction Results")
print("-" * 60)
print(f"Validation prediction records : {len(representation_prediction_df):,}")
print(f"Correct validation predictions: {int(representation_prediction_df['choice_correct'].sum()):,}")
print(f"Validation accuracy           : {representation_choice_accuracy:.3f}")

# List the primary artifacts produced by the notebook.
print("\nOutput Artifacts")
print("-" * 60)
print(f"Prediction file              : {REPRESENTATION_VIDEOQA_PREDICTIONS_CSV}")
print(f"Validation file              : {REPRESENTATION_VIDEOQA_VALIDATION_CSV}")
print(f"Summary file                 : {REPRESENTATION_VIDEOQA_SUMMARY_CSV}")
print(f"Output directory             : {REPRESENTATION_VIDEOQA_LOCAL_DIR}")

print("\nNotebook 07 generated:")

# Describe whether the run used direct scoring or a trained classifier.
if REPRESENTATION_VIDEOQA_METHOD == "cosine_similarity":

    print(
        "- CLIP cosine-similarity baseline "
        "with no classifier training"
    )

else:

    method_label = REPRESENTATION_VIDEOQA_METHOD_LABELS[
        REPRESENTATION_VIDEOQA_METHOD
    ]

    print(
        f"- {method_label} trained on the training split"
    )

print("- Validation-only representation-based VideoQA prediction dataset")
print("- Prediction validation report")
print("- Representation-based VideoQA summary report")
print("- Sample validation prediction records")

print("\nNotebook 07 outputs are ready for downstream evaluation.")

